# LF_01 · Breathe Barcelona · Dashboard interactivo

## Integración, visualización y explotación de resultados

Este notebook desarrolla **Breathe Barcelona**, la aplicación interactiva construida como capa final de integración, exploración y comunicación de los resultados obtenidos en el TFM.

El dashboard se plantea como una capa de explotación independiente de los pipelines analíticos. Para ello, utiliza **datasets procesados, resultados estadísticos, predicciones y artefactos de modelización previamente generados**, evitando reproducir durante la ejecución de la aplicación los procesos de tratamiento y entrenamiento realizados en las fases anteriores.

El desarrollo comprende la preparación y validación de los datos destinados al dashboard y la construcción de la aplicación en **Streamlit**, organizada en cinco vistas complementarias:

- **Overview**, como síntesis general de la calidad del aire en Barcelona.
- **Temporal**, para la exploración de patrones temporales, factores asociados y predicciones de series temporales.
- **Spatial**, orientada al análisis territorial del NO₂, la dependencia espacial y la modelización geoespacial.
- **Model Insights**, destinada a la evaluación, generalización e interpretabilidad de los modelos.
- **Predictor**, como capa interactiva para explorar predicciones espaciales de NO₂ y escenarios temporales de calidad del aire.

La aplicación se completa con una **Home / Landing**, una identidad visual común y los procedimientos necesarios para su ejecución y acceso externo.

> **Objetivo:** transformar los resultados analíticos del TFM en un producto interactivo que integre análisis temporal y espacial, modelización predictiva, interpretabilidad y exploración de escenarios.

## 1 · Configuración del entorno

En esta primera etapa se prepara el entorno de ejecución necesario para **Breathe Barcelona**, instalando las dependencias utilizadas en la construcción y visualización del dashboard interactivo.

Las principales librerías empleadas son:

- **Streamlit** · desarrollo de la aplicación web interactiva.
- **Plotly** · generación de gráficos dinámicos e interactivos.
- **Folium** · representación cartográfica y visualización geoespacial.
- **Streamlit-Folium** · integración de los mapas de Folium en la aplicación.
- **Streamlit Option Menu** · construcción del sistema de navegación del dashboard.

> **Objetivo:** disponer de un entorno reproducible con todas las dependencias necesarias para ejecutar la aplicación.

In [1]:
# ============================================================
# BREATHE BARCELONA
# 01 · Instalación de dependencias del dashboard
# ============================================================

!pip -q install streamlit plotly streamlit-option-menu folium streamlit-folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.9/536.9 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 53.2 MB/s eta 0:00:00


## 2 . Configuración global e identidad visual

En esta sección se define la configuración base del dashboard y la identidad gráfica de **Breathe Barcelona**.

La interfaz utilizará una paleta oscura basada en azul petróleo y turquesa, reservando los colores semánticos para la clasificación de la calidad del aire.

### Resultado de la configuración

La configuración inicial de **Breathe Barcelona** se ha completado correctamente.

- **Directorio de trabajo:** `/content/drive/MyDrive/TFM/12_Dashboard`
- **Resolución de referencia:** 1920 × 1080 px
- **Escala de Windows:** 125 %
- **Zoom de navegador de referencia:** 80 %
- **Diseño:** panorámico y optimizado para minimizar el desplazamiento vertical.
- **Identidad visual:** fondo azul petróleo, superficies oscuras y turquesa como color principal.
- **Colores semánticos:** verde, ámbar y rojo reservados para representar los niveles de calidad del aire.

Esta configuración constituye la base común sobre la que se desarrollarán las cinco vistas de la aplicación: **Overview, Temporal, Spatial, Model Insights y Predictor**.

In [2]:
# ============================================================
# BREATHE BARCELONA
# 02 · Imports, configuración global y paleta visual
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

import folium
from folium.plugins import Fullscreen

# ------------------------------------------------------------
# Rutas base
# ------------------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/TFM")
DASHBOARD_DIR = BASE_DIR / "12_Dashboard"

# IMPORTANTE:
# No crear directorios aquí.
# Google Drive se monta posteriormente en la sección 03.

# ------------------------------------------------------------
# Identidad visual Breathe Barcelona
# ------------------------------------------------------------

COLORS = {
    # Fondos
    "bg": "#071A24",            # fondo principal azul petróleo
    "surface": "#0B2833",       # tarjetas oscuras
    "surface_2": "#0B3442",     # tarjetas secundarias
    "border": "#174654",        # bordes sutiles

    # Identidad Breathe Barcelona
    "teal": "#18C5C8",          # turquesa principal
    "teal_light": "#72E6E0",    # turquesa claro
    "teal_dark": "#087F8C",     # turquesa oscuro

    # Texto
    "text": "#E6F2F3",          # texto principal
    "text_muted": "#8FAAB3",    # texto secundario
    "white": "#FFFFFF",

    # Colores semánticos de calidad del aire
    "good": "#49C96D",          # Buena
    "medium": "#F2B84B",        # Media
    "bad": "#EF5A5A",           # Mala
}

# ------------------------------------------------------------
# Identidad de la aplicación
# ------------------------------------------------------------

APP_TITLE = "BREATHE BARCELONA"
APP_SUBTITLE = "Urban Air Quality Intelligence"

# ------------------------------------------------------------
# Configuración de pantalla de referencia
# ------------------------------------------------------------

TARGET_SCREEN = "1920 × 1080"
WINDOWS_SCALE = "125 %"
REFERENCE_ZOOM = "80 %"

# Alturas orientativas.
# Se ajustarán definitivamente al probar la aplicación real.
HEADER_HEIGHT = 52
FILTER_HEIGHT = 58
MAIN_HEIGHT = 610

# ------------------------------------------------------------
# Configuración gráfica común de Plotly
# ------------------------------------------------------------

PLOTLY_TEMPLATE = go.layout.Template(
    layout=dict(
        paper_bgcolor=COLORS["bg"],
        plot_bgcolor=COLORS["surface"],

        font=dict(
            color=COLORS["text"],
            family="Arial"
        ),

        margin=dict(
            l=20,
            r=20,
            t=35,
            b=20
        ),

        xaxis=dict(
            gridcolor="rgba(255,255,255,0.08)",
            zerolinecolor="rgba(255,255,255,0.10)"
        ),

        yaxis=dict(
            gridcolor="rgba(255,255,255,0.08)",
            zerolinecolor="rgba(255,255,255,0.10)"
        )
    )
)

# ------------------------------------------------------------
# Comprobación
# ------------------------------------------------------------

print("=" * 60)
print("🌬️  BREATHE BARCELONA")
print("=" * 60)

print("Configuración cargada correctamente.")
print(f"Carpeta dashboard : {DASHBOARD_DIR}")
print(f"Resolución objetivo: {TARGET_SCREEN}")
print(f"Escala Windows     : {WINDOWS_SCALE}")
print(f"Zoom navegador     : {REFERENCE_ZOOM}")

print("=" * 60)

🌬️  BREATHE BARCELONA
Configuración cargada correctamente.
Carpeta dashboard : /content/drive/MyDrive/TFM/12_Dashboard
Resolución objetivo: 1920 × 1080
Escala Windows     : 125 %
Zoom navegador     : 80 %


## 3 . Conexión con Google Drive

Los datasets procesados, resultados de modelización y outputs analíticos utilizados por **Breathe Barcelona** se encuentran almacenados en la estructura de trabajo del TFM en Google Drive.

Antes de realizar el inventario de archivos se establece la conexión con el directorio principal del proyecto:

`/content/drive/MyDrive/TFM`

De este modo, el dashboard podrá consumir directamente los resultados generados por los pipelines temporal y geoespacial, evitando duplicar procesos de cálculo.

In [7]:
from google.colab import drive

try:
    drive.flush_and_unmount()
    print("✓ Google Drive desmontado correctamente.")
except Exception as e:
    print("Resultado del desmontaje:")
    print(e)

Drive not mounted, so nothing to flush and unmount.
✓ Google Drive desmontado correctamente.


In [4]:
from pathlib import Path

ruta = Path("/content/drive")

print("Existe /content/drive:", ruta.exists())

if ruta.exists():
    print("\nContenido LOCAL actualmente en /content/drive:\n")

    for elemento in ruta.rglob("*"):
        print(elemento)

Existe /content/drive: False


In [9]:
# ============================================================
 # Preservar contenido local antes de montar Google Drive
# ============================================================

from pathlib import Path
import shutil

origen = Path("/content/drive")
destino = Path("/content/drive_residuo_local")

if origen.exists():

    if destino.exists():
        print(f"⚠️ Ya existe: {destino}")
        print("No se realiza ningún movimiento automático.")

    else:
        shutil.move(str(origen), str(destino))

        print("✅ Contenido local preservado correctamente.")
        print(f"   Origen anterior: /content/drive")
        print(f"   Nueva ubicación: {destino}")

else:
    print("ℹ️ /content/drive ya no existe.")

ℹ️ /content/drive ya no existe.


In [10]:
# ============================================================
#
# Montaje seguro de Google Drive
# ============================================================

from google.colab import drive
from pathlib import Path
import os
import shutil
from datetime import datetime

MOUNT_POINT = Path("/content/drive")

# ------------------------------------------------------------
# 1. Limpiar únicamente restos LOCALES
# ------------------------------------------------------------

if MOUNT_POINT.exists() and not os.path.ismount(MOUNT_POINT):

    contenido = list(MOUNT_POINT.iterdir())

    if contenido:
        marca = datetime.now().strftime("%Y%m%d_%H%M%S")
        residuo = Path(f"/content/drive_residuo_local_{marca}")

        shutil.move(str(MOUNT_POINT), str(residuo))

        print("⚠️ Se detectó contenido local previo en /content/drive")
        print("✅ Se ha apartado sin borrar:")
        print(f"   {residuo}")

# ------------------------------------------------------------
# 2. Montar Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

# ------------------------------------------------------------
# 3. Definir rutas SOLO después del montaje
# ------------------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/TFM")
DASHBOARD_DIR = BASE_DIR / "12_Dashboard"

# ------------------------------------------------------------
# 4. Comprobación
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🌬️ BREATHE BARCELONA — GOOGLE DRIVE")
print("=" * 70)

print("Drive montado:", os.path.ismount("/content/drive"))
print("TFM existe:", BASE_DIR.exists())

if BASE_DIR.exists():

    print("\n✅ Google Drive montado correctamente.")
    print(f"✅ Directorio TFM: {BASE_DIR}")

    print("\n📁 Carpetas detectadas:\n")

    for carpeta in sorted(
        p.name for p in BASE_DIR.iterdir()
        if p.is_dir()
    ):
        print(f"   • {carpeta}")

else:
    raise FileNotFoundError(
        f"❌ No se ha encontrado el directorio del TFM:\n{BASE_DIR}"
    )

print("\n" + "=" * 70)

Mounted at /content/drive

🌬️ BREATHE BARCELONA — GOOGLE DRIVE
Drive montado: True
TFM existe: True

✅ Google Drive montado correctamente.
✅ Directorio TFM: /content/drive/MyDrive/TFM

📁 Carpetas detectadas:

   • 01_Modelo_Digital_del_Terreno
   • 02_Trafico_Rodado_y_Aforos
   • 03_Transporte_Maritimo
   • 04_Otros_Transportes
   • 05_Transporte_Aereo
   • 06_Analisis_Geoespacial
   • 06_Contaminacion_Atmosferica
   • 06_Dataset_Maestro_V4
   • 07_Contaminacion_Acustica
   • 08_Meteorologia
   • 09_Indicadores_Socioeconomicos
   • 10_Machine_Learning_Geoespacial
   • 11_Machine_Learning_Temporal
   • 12_Dashboard
   • 13_Modelos_Series_Temporales
   • CAPAS_GEO_LIMPIAS
   • MODELO BASE LINE
   • NOTEBOOKS_GEO
   • RESULTADOS_AFOROS



### Resultado de la conexión

La conexión con Google Drive se ha realizado correctamente, estableciendo como directorio principal del proyecto:

`/content/drive/MyDrive/TFM`

Se ha verificado la estructura de carpetas del TFM y la disponibilidad de los directorios correspondientes a las distintas fuentes de datos, análisis, modelos y resultados.

Durante el proceso se detectó contenido local previamente creado en el punto de montaje `/content/drive`. Para evitar cualquier pérdida de información, dicho contenido fue preservado antes de realizar el montaje del Drive real.

A partir de este punto, **Breathe Barcelona** trabajará directamente sobre la estructura real del proyecto almacenada en Google Drive.

## 4 . Inventario de datos y resultados

Una vez establecida la conexión con Google Drive, se realiza un inventario de los principales archivos generados durante las fases de procesamiento, análisis y modelización del TFM.

El objetivo de esta etapa es identificar los **datasets finales, resultados analíticos, predicciones, métricas y artefactos de modelización** que serán utilizados por **Breathe Barcelona**.

El dashboard se plantea como una capa de explotación y productivización de los resultados previamente obtenidos. Por tanto, los modelos y análisis ya finalizados no se recalculan durante la ejecución de la aplicación, sino que se consumen a partir de sus outputs almacenados.

El inventario se centra especialmente en los resultados procedentes de:

- análisis geoespacial;
- dataset maestro;
- modelización temporal;
- modelización geoespacial;
- modelos clásicos de series temporales;
- outputs preparados específicamente para el dashboard.

Esta comprobación permitirá definir posteriormente qué archivos alimentarán cada una de las cinco vistas de la aplicación: **Overview, Temporal, Spatial, Model Insights y Predictor**.

###  4.1 · Inventario automático de outputs

Antes de construir las visualizaciones del dashboard se realiza una **exploración automática de los resultados generados en las distintas etapas del proyecto**.

Esta celda recorre los principales directorios de *Breathe Barcelona* y construye un inventario de los archivos potencialmente reutilizables por la aplicación, incluyendo:

- **datasets procesados** y versiones finales del dataset maestro;
- **resultados del análisis geoespacial**, como Moran, LISA y modelos espaciales;
- **outputs de Machine Learning**, predicciones y métricas;
- **resultados de series temporales** y análisis del efecto COVID;
- **artefactos de modelos** almacenados en formatos `.pkl`, `.pickle` o `.joblib`;
- **recursos cartográficos y gráficos** destinados a la visualización.

La identificación se realiza mediante **extensiones de archivo y palabras clave**, permitiendo localizar de forma reproducible los outputs disponibles sin depender de rutas introducidas manualmente.

> **Objetivo:** verificar qué resultados del pipeline analítico están disponibles y determinar cuáles pueden integrarse directamente en el dashboard.

In [11]:
# ============================================================
# BREATHE BARCELONA
# 04.1 · Inventario automático de outputs
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Rutas principales del proyecto
# ------------------------------------------------------------

RUTAS_TFM = {
    "Análisis geoespacial":
        BASE_DIR / "06_Analisis_Geoespacial",

    "Contaminación atmosférica":
        BASE_DIR / "06_Contaminacion_Atmosferica",

    "Dataset maestro":
        BASE_DIR / "06_Dataset_Maestro_V4",

    "Machine Learning geoespacial":
        BASE_DIR / "10_Machine_Learning_Geoespacial",

    "Machine Learning temporal":
        BASE_DIR / "11_Machine_Learning_Temporal",

    "Dashboard":
        BASE_DIR / "12_Dashboard",

    "Series temporales":
        BASE_DIR / "13_Modelos_Series_Temporales",
}

# ------------------------------------------------------------
# Extensiones relevantes para el dashboard
# ------------------------------------------------------------

EXTENSIONES = {
    ".csv",
    ".parquet",
    ".geojson",
    ".json",
    ".gpkg",
    ".shp",
    ".pkl",
    ".pickle",
    ".joblib",
    ".png",
}

# ------------------------------------------------------------
# Palabras clave para detectar outputs importantes
# ------------------------------------------------------------

PALABRAS_CLAVE = [
    "maestro",
    "final",
    "modelo",
    "metric",
    "predic",
    "rolling",
    "resid",
    "covid",
    "moran",
    "lisa",
    "sar",
    "shap",
    "xgb",
    "seccion",
    "espacial",
    "no2",
]

# ------------------------------------------------------------
# Construcción del inventario
# ------------------------------------------------------------

registros = []

for bloque, ruta in RUTAS_TFM.items():

    if not ruta.exists():
        print(f"⚠️ Ruta no encontrada: {ruta}")
        continue

    for archivo in ruta.rglob("*"):

        if not archivo.is_file():
            continue

        if archivo.suffix.lower() not in EXTENSIONES:
            continue

        nombre_lower = archivo.name.lower()

        relevante = any(
            palabra in nombre_lower
            for palabra in PALABRAS_CLAVE
        )

        registros.append({
            "Bloque": bloque,
            "Archivo": archivo.name,
            "Extensión": archivo.suffix.lower(),
            "Relevante": relevante,
            "Ruta": str(archivo)
        })

inventario = pd.DataFrame(registros)

# ------------------------------------------------------------
# Resumen general
# ------------------------------------------------------------

print("=" * 90)
print("🌬️ BREATHE BARCELONA — INVENTARIO DE OUTPUTS")
print("=" * 90)

print(f"\nArchivos detectados: {len(inventario):,}")

if not inventario.empty:

    print("\nArchivos por bloque:\n")

    resumen = (
        inventario
        .groupby("Bloque")
        .size()
        .sort_values(ascending=False)
    )

    print(resumen)

    # --------------------------------------------------------
    # Archivos potencialmente relevantes
    # --------------------------------------------------------

    print("\n" + "-" * 90)
    print("ARCHIVOS POTENCIALMENTE RELEVANTES PARA EL DASHBOARD")
    print("-" * 90)

    relevantes = (
        inventario[inventario["Relevante"]]
        .sort_values(["Bloque", "Archivo"])
    )

    for _, fila in relevantes.iterrows():

        print(
            f"\n[{fila['Bloque']}]"
            f"\n  Archivo: {fila['Archivo']}"
            f"\n  Ruta:    {fila['Ruta']}"
        )

    # --------------------------------------------------------
    # Comprobación de artefactos de modelos
    # --------------------------------------------------------

    extensiones_modelo = {
        ".pkl",
        ".pickle",
        ".joblib"
    }

    artefactos_modelo = inventario[
        inventario["Extensión"].isin(extensiones_modelo)
    ]

    print("\n" + "=" * 90)
    print("ARTEFACTOS DE MODELO DETECTADOS")
    print("=" * 90)

    if artefactos_modelo.empty:

        print(
            "\n⚠️ No se han detectado archivos "
            ".pkl / .pickle / .joblib."
        )

    else:

        for _, fila in artefactos_modelo.iterrows():

            print(f"\n✓ {fila['Archivo']}")
            print(f"  {fila['Ruta']}")

else:

    print("\n⚠️ No se han detectado archivos en las rutas analizadas.")

print("\n" + "=" * 90)

🌬️ BREATHE BARCELONA — INVENTARIO DE OUTPUTS

Archivos detectados: 440

Archivos por bloque:

Bloque
Machine Learning temporal       178
Contaminación atmosférica       113
Análisis geoespacial             68
Machine Learning geoespacial     56
Dashboard                        15
Series temporales                 6
Dataset maestro                   4
dtype: int64

------------------------------------------------------------------------------------------
ARCHIVOS POTENCIALMENTE RELEVANTES PARA EL DASHBOARD
------------------------------------------------------------------------------------------

[Análisis geoespacial]
  Archivo: 01_elevacion_media_secciones.png
  Ruta:    /content/drive/MyDrive/TFM/06_Analisis_Geoespacial/RESULTADOS/FIGURAS/MAPAS_EXPLORATORIOS/01_elevacion_media_secciones.png

[Análisis geoespacial]
  Archivo: 02_NO2_medio_2018_2024.png
  Ruta:    /content/drive/MyDrive/TFM/06_Analisis_Geoespacial/RESULTADOS/FIGURAS/MAPAS_EXPLORATORIOS/02_NO2_medio_2018_2024.png

[Anál

### Resultados

El inventario automático confirma la disponibilidad de los principales datasets y resultados necesarios para construir **Breathe Barcelona**.

Se han identificado outputs correspondientes al análisis temporal, análisis geoespacial, modelos de Machine Learning y modelos clásicos de series temporales, incluyendo predicciones, métricas de evaluación, resultados de interpretabilidad y diagnósticos.

También se ha localizado un artefacto de modelización geoespacial previamente serializado, lo que permitirá estudiar su incorporación directa a la aplicación sin necesidad de reentrenar el modelo durante la ejecución del dashboard.

A partir de este inventario se realizará una selección de los archivos finales que alimentarán cada una de las vistas de la aplicación.

### 4.2 · Verificación de datasets candidatos

Una vez identificados los principales outputs del proyecto, se realiza una **verificación estructurada de los datasets candidatos a alimentar el dashboard**.

Esta etapa comprueba automáticamente la disponibilidad y estructura de las principales fuentes de información:

- **base temporal integrada**, con contaminación, meteorología, tráfico, ruido, actividad aeroportuaria y portuaria;
- **dataset de modelización temporal** y predicciones del modelo final **XGBoost**;
- resultados de **validación espacial LOSO**;
- predicciones y métricas de los modelos clásicos de series temporales **SARIMAX**;
- **base geoespacial integrada** a nivel de sección censal;
- resultados del análisis de autocorrelación espacial **LISA**;
- datasets específicamente preparados para los módulos **geoespacial y COVID** del dashboard.

Para cada archivo se verifica su existencia, formato, dimensiones, variables disponibles y, cuando corresponde, su **sistema de referencia de coordenadas (CRS)**.

> **Objetivo:** confirmar que las fuentes seleccionadas están disponibles, son legibles y contienen la estructura necesaria antes de incorporarlas a la arquitectura de datos de *Breathe Barcelona*.

In [12]:
# ============================================================
# BREATHE BARCELONA
# 04.2 · Verificación de datasets candidatos
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Archivos candidatos principales
# ------------------------------------------------------------

CANDIDATOS = {
    # Base temporal completa
    "Maestro temporal":
        BASE_DIR / "11_Machine_Learning_Temporal/INTEGRACION/"
        "06_maestro_contaminacion_meteorologia_trafico_ruido_vuelos_puerto.csv",

    # Datos preparados para modelización temporal
    "Dataset temporal modelado":
        BASE_DIR / "11_Machine_Learning_Temporal/03_Datos_Modelado/"
        "dataset_NO2_feature_engineering.csv",

    # Predicciones del modelo temporal final
    "Predicciones XGBoost temporal":
        BASE_DIR / "11_Machine_Learning_Temporal/03_Datos_Modelado/"
        "modelo_final_predicciones_TEST.csv",

    # Validación LOSO
    "Validación LOSO":
        BASE_DIR / "11_Machine_Learning_Temporal/03_Datos_Modelado/"
        "validacion_espacial_LOSO_resultados.csv",

    # Series temporales clásicas
    "Predicciones rolling SARIMAX":
        BASE_DIR / "13_Modelos_Series_Temporales/"
        "LF04_predicciones_rolling_2024.csv",

    "Métricas rolling SARIMAX":
        BASE_DIR / "13_Modelos_Series_Temporales/"
        "LF04_metricas_modelos_rolling.csv",

    # Base espacial integrada
    "Base geoespacial":
        BASE_DIR / "06_Analisis_Geoespacial/RESULTADOS/BASE_INTEGRADA/"
        "base_espacial_integrada_secciones.gpkg",

    # LISA
    "LISA NO2":
        BASE_DIR / "06_Analisis_Geoespacial/RESULTADOS/MORAN/LISA/"
        "LISA_NO2_secciones.gpkg",

    # Datos ya preparados para dashboard
    "Dashboard geoespacial":
        BASE_DIR / "12_Dashboard/01_Datos/"
        "dashboard_geoespacial.gpkg",

    "Dashboard COVID":
        BASE_DIR / "12_Dashboard/01_Datos/"
        "dashboard_comparativa_covid.parquet",
}

# ------------------------------------------------------------
# Función de inspección
# ------------------------------------------------------------

def inspeccionar_archivo(nombre, ruta):

    print("\n" + "=" * 90)
    print(f"📄 {nombre}")
    print("=" * 90)

    print(f"Ruta: {ruta}")

    if not ruta.exists():
        print("❌ ARCHIVO NO ENCONTRADO")
        return

    print("✅ Archivo encontrado")

    try:

        extension = ruta.suffix.lower()

        if extension == ".csv":
            df = pd.read_csv(ruta)

        elif extension == ".parquet":
            df = pd.read_parquet(ruta)

        elif extension in {".gpkg", ".geojson", ".shp"}:
            import geopandas as gpd
            df = gpd.read_file(ruta)

        else:
            print(f"⚠️ Formato no inspeccionado: {extension}")
            return

        print(f"Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        if hasattr(df, "crs"):
            print(f"CRS: {df.crs}")

        print("\nColumnas:")

        for col in df.columns:
            print(f"   • {col}")

    except Exception as e:

        print("❌ Error durante la lectura:")
        print(e)


# ------------------------------------------------------------
# Ejecutar inspección
# ------------------------------------------------------------

for nombre, ruta in CANDIDATOS.items():
    inspeccionar_archivo(nombre, ruta)

print("\n" + "=" * 90)
print("🌬️ VERIFICACIÓN DE DATASETS FINALIZADA")
print("=" * 90)


📄 Maestro temporal
Ruta: /content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/INTEGRACION/06_maestro_contaminacion_meteorologia_trafico_ruido_vuelos_puerto.csv
✅ Archivo encontrado


/tmp/ipykernel_1582/967576609.py:86: DtypeWarning: Columns (1,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta)


Dimensiones: 43,642 filas × 73 columnas

Columnas:
   • fecha
   • codi_estacio
   • nom_cabina
   • contaminant
   • valor
   • horas_validas
   • codi_estacio_str
   • estacion_fisica
   • estacion_geo
   • lat
   • lon
   • tipo
   • X_ETRS89
   • Y_ETRS89
   • codigo_meteo
   • estacion_meteo_asignada
   • distancia_meteo_m
   • distancia_meteo_km
   • TM
   • TN
   • TX
   • HRM
   • HRN
   • HRX
   • PM
   • PN
   • PX
   • PPT
   • RS24h
   • DVM10
   • DVVX10
   • VVM10
   • VVX10
   • n_aforos_total_500m
   • trafico_media_500m
   • trafico_suma_500m
   • n_aforos_obs_500m
   • dist_aforo_min_500m
   • trafico_idw_500m
   • n_aforos_total_750m
   • trafico_media_750m
   • trafico_suma_750m
   • n_aforos_obs_750m
   • dist_aforo_min_750m
   • trafico_idw_750m
   • n_aforos_total_1000m
   • trafico_media_1000m
   • trafico_suma_1000m
   • n_aforos_obs_1000m
   • dist_aforo_min_1000m
   • trafico_idw_1000m
   • anio
   • n_aforos_disponibles_500m
   • n_aforos_disponibles_750m
  

### Resultados

La inspección de los datasets candidatos confirma la disponibilidad y compatibilidad de las principales fuentes necesarias para construir **Breathe Barcelona**.

La componente temporal dispone del dataset maestro integrado, del conjunto utilizado para la modelización de NO₂, de las predicciones del modelo XGBoost, de los resultados de validación espacial LOSO y de las predicciones rolling obtenidas mediante los modelos clásicos de series temporales.

La componente geoespacial dispone de una base integrada para las 1.068 secciones censales de Barcelona, de los resultados del análisis LISA y de una capa específicamente preparada para el dashboard que incorpora valores observados, predicciones XGBoost, residuos y variables territoriales.

Asimismo, se dispone de una tabla agregada para la comparación entre los periodos pre-COVID, COVID y post-COVID.

Esta estructura permite construir las distintas vistas de la aplicación a partir de resultados previamente calculados, manteniendo separada la fase analítica de la fase de visualización y productivización.

### 4.3 · Control de calidad de los inputs definitivos

Tras verificar la disponibilidad de las fuentes candidatas, se realiza un **control de calidad de los datasets seleccionados definitivamente para alimentar el dashboard**.

La validación se estructura sobre los seis componentes principales de *Breathe Barcelona*:

- **dataset temporal**, comprobando cobertura temporal, estaciones disponibles, periodos de análisis, clases de NO₂ y disponibilidad de contaminantes;
- **predicciones XGBoost**, contrastando la distribución de clases reales y predichas y los rangos de probabilidad obtenidos;
- **series temporales**, verificando la cobertura de las predicciones *rolling* y la disponibilidad de observaciones reales;
- **dataset geoespacial**, revisando número de secciones, CRS, conjuntos de modelización y rangos de las principales variables territoriales;
- **resultados LISA**, comprobando la significación estadística y distribución de los *clusters* espaciales de NO₂;
- **comparativa COVID**, verificando la presencia de los periodos pre-COVID, COVID y post-COVID y sus correspondientes niveles medios de contaminación.

Este control permite detectar **valores ausentes, coberturas incompletas, rangos anómalos o inconsistencias estructurales** antes de conectar los datos con las visualizaciones.

> **Objetivo:** garantizar que los inputs definitivos presentan la cobertura, consistencia y calidad necesarias para construir un dashboard fiable y reproducible.

In [13]:
# ============================================================
# BREATHE BARCELONA
# 04.3 · Control de calidad de los inputs definitivos
# ============================================================

import pandas as pd
import geopandas as gpd

print("=" * 90)
print("🌬️ BREATHE BARCELONA — CONTROL DE INPUTS")
print("=" * 90)

# ============================================================
# 1. DATASET TEMPORAL
# ============================================================

ruta_temporal = (
    BASE_DIR /
    "11_Machine_Learning_Temporal/03_Datos_Modelado/"
    "dataset_NO2_feature_engineering.csv"
)

df_temporal = pd.read_csv(
    ruta_temporal,
    parse_dates=["fecha"],
    low_memory=False
)

print("\n📅 1. DATASET TEMPORAL")
print("-" * 90)

print(f"Registros: {len(df_temporal):,}")
print(f"Fecha inicial: {df_temporal['fecha'].min().date()}")
print(f"Fecha final:   {df_temporal['fecha'].max().date()}")

print("\nEstaciones:")
for estacion in sorted(df_temporal["estacion_geo"].dropna().unique()):
    print(f"   • {estacion}")

print("\nPeriodos:")
print(df_temporal["periodo_covid"].value_counts(dropna=False))

print("\nClases NO₂:")
print(df_temporal["clase_NO2_3"].value_counts(dropna=False))

print("\nDisponibilidad de contaminantes:")
for variable in [
    "NO2_ug_m3",
    "O3_ug_m3",
    "PM10_ug_m3",
    "PM25_ug_m3"
]:
    n = df_temporal[variable].notna().sum()
    pct = 100 * n / len(df_temporal)
    print(f"   {variable:<12} {n:>6,} observaciones ({pct:5.1f} %)")

# ============================================================
# 2. PREDICCIONES XGBOOST TEMPORAL
# ============================================================

ruta_pred_xgb = (
    BASE_DIR /
    "11_Machine_Learning_Temporal/03_Datos_Modelado/"
    "modelo_final_predicciones_TEST.csv"
)

df_pred_xgb = pd.read_csv(ruta_pred_xgb)

print("\n🤖 2. PREDICCIONES XGBOOST TEMPORAL")
print("-" * 90)

print(f"Registros TEST: {len(df_pred_xgb):,}")

print("\nClases reales:")
print(df_pred_xgb["y_real"].value_counts(dropna=False).sort_index())

print("\nClases predichas:")
print(df_pred_xgb["y_pred"].value_counts(dropna=False).sort_index())

print("\nRango de probabilidades:")
for col in ["prob_Buena", "prob_Media", "prob_Mala"]:
    print(
        f"   {col:<12} "
        f"{df_pred_xgb[col].min():.4f} → "
        f"{df_pred_xgb[col].max():.4f}"
    )

# ============================================================
# 3. SERIES TEMPORALES
# ============================================================

ruta_rolling = (
    BASE_DIR /
    "13_Modelos_Series_Temporales/"
    "LF04_predicciones_rolling_2024.csv"
)

df_rolling = pd.read_csv(
    ruta_rolling,
    parse_dates=["fecha"]
)

print("\n📈 3. SERIES TEMPORALES — ROLLING 2024")
print("-" * 90)

print(f"Registros: {len(df_rolling):,}")
print(f"Fecha inicial: {df_rolling['fecha'].min().date()}")
print(f"Fecha final:   {df_rolling['fecha'].max().date()}")

print("\nContaminantes:")
print(df_rolling["contaminante"].value_counts(dropna=False))

print("\nDisponibilidad del dato observado:")
print(
    df_rolling["dato_observado_disponible"]
    .value_counts(dropna=False)
)

# ============================================================
# 4. DATASET GEOESPACIAL DEL DASHBOARD
# ============================================================

ruta_geo = (
    BASE_DIR /
    "12_Dashboard/01_Datos/"
    "dashboard_geoespacial.gpkg"
)

gdf_geo = gpd.read_file(ruta_geo)

print("\n🗺️ 4. DATASET GEOESPACIAL")
print("-" * 90)

print(f"Secciones: {len(gdf_geo):,}")
print(f"CRS: {gdf_geo.crs}")

print("\nClases NO₂:")
print(gdf_geo["NO2_CLASE"].value_counts(dropna=False))

print("\nConjunto ML:")
print(gdf_geo["CONJUNTO"].value_counts(dropna=False))

print("\nRangos espaciales principales:")

for variable in [
    "NO2_MEDIO_SECCION",
    "NO2_PRED_XGB",
    "RENTA_EUR_PERSONA",
    "DENSIDAD_HAB_HA",
    "ELEVACION_MEDIA_M",
    "DISTANCIA_AFORO_M",
]:
    serie = pd.to_numeric(gdf_geo[variable], errors="coerce")

    print(
        f"   {variable:<24} "
        f"{serie.min():>10.2f} → {serie.max():>10.2f} "
        f"| nulos: {serie.isna().sum():,}"
    )

# ============================================================
# 5. LISA NO2
# ============================================================

ruta_lisa = (
    BASE_DIR /
    "06_Analisis_Geoespacial/RESULTADOS/MORAN/LISA/"
    "LISA_NO2_secciones.gpkg"
)

gdf_lisa = gpd.read_file(ruta_lisa)

print("\n📍 5. LISA NO₂")
print("-" * 90)

print(f"Secciones: {len(gdf_lisa):,}")

print("\nSignificación LISA:")
print(
    gdf_lisa["LISA_SIGNIFICATIVO"]
    .value_counts(dropna=False)
)

print("\nClusters LISA:")
print(
    gdf_lisa["LISA_CLUSTER"]
    .value_counts(dropna=False)
)

# ============================================================
# 6. COMPARATIVA COVID
# ============================================================

ruta_covid = (
    BASE_DIR /
    "12_Dashboard/01_Datos/"
    "dashboard_comparativa_covid.parquet"
)

df_covid = pd.read_parquet(ruta_covid)

print("\n🦠 6. COMPARATIVA COVID")
print("-" * 90)

print(f"Registros: {len(df_covid)}")

print("\nPeriodos disponibles:")
print(df_covid["periodo_dash"].tolist())

print("\nNO₂ medio por periodo:")
print(
    df_covid[
        ["periodo_dash", "NO2_medio_ciudad_mean"]
    ].to_string(index=False)
)

# ============================================================
# FIN
# ============================================================

print("\n" + "=" * 90)
print("✅ CONTROL DE INPUTS FINALIZADO")
print("=" * 90)

🌬️ BREATHE BARCELONA — CONTROL DE INPUTS

📅 1. DATASET TEMPORAL
------------------------------------------------------------------------------------------
Registros: 14,022
Fecha inicial: 2019-01-01
Fecha final:   2024-12-31

Estaciones:
   • Ciutadella
   • Eixample
   • Gràcia
   • Observatori Fabra
   • Palau Reial
   • Poblenou
   • Sants
   • Vall d'Hebron

Periodos:
periodo_covid
Post-COVID    7560
COVID         4900
Pre-COVID     1562
Name: count, dtype: int64

Clases NO₂:
clase_NO2_3
Buena    8079
Media    5680
Mala      263
Name: count, dtype: int64

Disponibilidad de contaminantes:
   NO2_ug_m3    14,022 observaciones (100.0 %)
   O3_ug_m3     11,820 observaciones ( 84.3 %)
   PM10_ug_m3   11,057 observaciones ( 78.9 %)
   PM25_ug_m3    1,724 observaciones ( 12.3 %)

🤖 2. PREDICCIONES XGBOOST TEMPORAL
------------------------------------------------------------------------------------------
Registros TEST: 2,506

Clases reales:
y_real
0    1679
1     805
2      22
Name: count

### Resultados

La verificación final confirma que los datasets seleccionados presentan una estructura adecuada para su integración en **Breathe Barcelona**.

El dataset temporal contiene 14.022 observaciones correspondientes al periodo 2019–2024 y a ocho estaciones de calidad del aire. NO₂ presenta cobertura completa, mientras que O₃ y PM10 disponen de una cobertura elevada y PM2.5 presenta una disponibilidad temporal más limitada.

Los resultados del modelo temporal incluyen 2.506 observaciones del conjunto de test, junto con la clase observada, la clase predicha y las probabilidades estimadas para las categorías Buena, Media y Mala.

La evaluación mediante series temporales proporciona predicciones rolling para NO₂, PM10 y PM2.5 durante 2024. Se identifican 35 fechas sin observación disponible, que serán excluidas de las comparaciones entre valores observados y predichos.

La componente geoespacial contiene las 1.068 secciones censales analizadas, sin valores ausentes en las principales variables utilizadas por el dashboard. Asimismo, se dispone de la clasificación LISA y de los resultados de predicción espacial mediante XGBoost.

Finalmente, la comparación temporal distingue de forma explícita los periodos Pre-COVID, COVID y Post-COVID.

Con estas comprobaciones queda definida y validada la capa de datos que alimentará las distintas vistas del dashboard.

## 5 . Preparación de datos para Breathe Barcelona

Una vez identificados y validados los outputs de las distintas ramas analíticas, se construye una capa de datos específica para **Breathe Barcelona**.

El objetivo es desacoplar la aplicación de los datasets originales de procesamiento y modelización, generando archivos ligeros y optimizados que puedan ser cargados directamente por Streamlit.

Esta capa mantiene los resultados analíticos originales, pero selecciona únicamente las variables necesarias para la visualización, interacción y explotación de los modelos.

La preparación se estructura en cuatro bloques principales:

- **Temporal:** evolución de contaminantes, estaciones y periodos de análisis.
- **Series temporales:** predicciones rolling y métricas de los modelos Naive, SARIMA y SARIMAX.
- **Geoespacial:** concentraciones, predicciones, errores y variables territoriales por sección censal.
- **Model Insights:** métricas, probabilidades, validación LOSO e interpretabilidad de los modelos.

Los archivos resultantes se almacenarán en `12_Dashboard/01_Datos`, que actuará como fuente de datos principal de la aplicación.

### 5.1. Dataset temporal optimizado

Se ha generado un dataset temporal específico para **Breathe Barcelona** a partir del conjunto utilizado en la modelización temporal.

La nueva estructura contiene **14.022 observaciones y 25 variables**, correspondientes a ocho estaciones y al periodo comprendido entre 2019 y 2024.

Se han conservado exclusivamente las variables necesarias para la explotación interactiva: concentraciones de contaminantes, clasificación de NO₂, información temporal, meteorología, tráfico, ruido, actividad aeroportuaria y portuaria, así como algunas variables de memoria temporal.

El archivo resultante ocupa aproximadamente **0,57 MB**, reduciendo considerablemente la información que deberá cargar la aplicación sin modificar los datasets originales del proyecto.

Archivo generado:

`12_Dashboard/01_Datos/dashboard_temporal.parquet`

In [14]:
# ============================================================
# BREATHE BARCELONA
# 05.1 · Dataset temporal optimizado para el dashboard
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Rutas
# ------------------------------------------------------------

DASHBOARD_DIR = BASE_DIR / "12_Dashboard"
DATA_DIR = DASHBOARD_DIR / "01_Datos"

# La carpeta ya existe según la verificación previa.
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"No se ha encontrado la carpeta esperada: {DATA_DIR}"
    )

ruta_origen = (
    BASE_DIR /
    "11_Machine_Learning_Temporal/03_Datos_Modelado/"
    "dataset_NO2_feature_engineering.csv"
)

ruta_salida = DATA_DIR / "dashboard_temporal.parquet"

# ------------------------------------------------------------
# Variables necesarias para la aplicación
# ------------------------------------------------------------

columnas_dashboard = [
    # Identificación
    "fecha",
    "estacion_geo",
    "lat",
    "lon",

    # Contaminación
    "NO2_ug_m3",
    "O3_ug_m3",
    "PM10_ug_m3",
    "PM25_ug_m3",

    # Clasificación
    "clase_NO2_3",
    "target",

    # Periodo
    "anio",
    "mes",
    "dia_semana",
    "periodo_covid",

    # Meteorología
    "TM",
    "HRM",
    "PPT",
    "VVM10",

    # Movilidad / actividad urbana
    "trafico_idw_500m",
    "LAeq_idw_1500m",
    "Vuelos_total",
    "Puerto_movimientos",

    # Memoria temporal
    "NO2_lag_1",
    "NO2_lag_7",
    "NO2_rolling_7d",
]

# ------------------------------------------------------------
# Lectura selectiva
# ------------------------------------------------------------

df_dash_temporal = pd.read_csv(
    ruta_origen,
    usecols=columnas_dashboard,
    parse_dates=["fecha"],
    low_memory=False
)

# ------------------------------------------------------------
# Orden
# ------------------------------------------------------------

df_dash_temporal = (
    df_dash_temporal
    .sort_values(["fecha", "estacion_geo"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Guardado optimizado
# ------------------------------------------------------------

df_dash_temporal.to_parquet(
    ruta_salida,
    index=False
)

# ------------------------------------------------------------
# Verificación
# ------------------------------------------------------------

tamano_mb = ruta_salida.stat().st_size / (1024 ** 2)

print("=" * 90)
print("🌬️ BREATHE BARCELONA — DATASET TEMPORAL")
print("=" * 90)

print(f"\n✅ Archivo creado:")
print(f"   {ruta_salida}")

print(
    f"\nDimensiones: "
    f"{df_dash_temporal.shape[0]:,} filas × "
    f"{df_dash_temporal.shape[1]} columnas"
)

print(f"Tamaño: {tamano_mb:.2f} MB")

print(
    f"\nPeriodo: "
    f"{df_dash_temporal['fecha'].min().date()} "
    f"→ "
    f"{df_dash_temporal['fecha'].max().date()}"
)

print(
    f"Estaciones: "
    f"{df_dash_temporal['estacion_geo'].nunique()}"
)

print("\nColumnas finales:")

for columna in df_dash_temporal.columns:
    print(f"   • {columna}")

print("\n" + "=" * 90)
print("✅ DATASET TEMPORAL PREPARADO")
print("=" * 90)

🌬️ BREATHE BARCELONA — DATASET TEMPORAL

✅ Archivo creado:
   /content/drive/MyDrive/TFM/12_Dashboard/01_Datos/dashboard_temporal.parquet

Dimensiones: 14,022 filas × 25 columnas
Tamaño: 0.57 MB

Periodo: 2019-01-01 → 2024-12-31
Estaciones: 8

Columnas finales:
   • fecha
   • estacion_geo
   • NO2_ug_m3
   • O3_ug_m3
   • PM10_ug_m3
   • PM25_ug_m3
   • lat
   • lon
   • TM
   • HRM
   • PPT
   • VVM10
   • trafico_idw_500m
   • anio
   • LAeq_idw_1500m
   • Vuelos_total
   • Puerto_movimientos
   • mes
   • periodo_covid
   • clase_NO2_3
   • target
   • dia_semana
   • NO2_lag_1
   • NO2_lag_7
   • NO2_rolling_7d

✅ DATASET TEMPORAL PREPARADO


### Resultados

La preparación del bloque temporal genera un dataset específico para el dashboard con **14.022 registros y 25 variables**, cubriendo de forma continua el periodo **2019–2024** y las **8 estaciones de calidad del aire** incluidas en el estudio.

El dataset integra en una única estructura las dimensiones necesarias para el análisis interactivo:

- **contaminación atmosférica:** NO₂, O₃, PM₁₀ y PM₂.₅;
- **meteorología:** temperatura, humedad, precipitación y velocidad del viento;
- **movilidad y actividad urbana:** tráfico, ruido, vuelos y movimientos portuarios;
- **contexto temporal:** año, mes, día de la semana y periodo COVID;
- **memoria temporal del NO₂:** *lags* de 1 y 7 días y media móvil de 7 días;
- **clasificación:** categoría de calidad del aire y variable objetivo empleada en la modelización.

El almacenamiento en formato **Parquet** reduce el dataset final a aproximadamente **0,57 MB**, proporcionando una fuente compacta y adecuada para realizar filtrados y visualizaciones dinámicas con tiempos de carga reducidos.

> **Resultado:** `dashboard_temporal.parquet` queda preparado como fuente principal del módulo temporal de *Breathe Barcelona*.

### 5.2 · Preparación de outputs de series temporales

En esta etapa se preparan los resultados de los **modelos de series temporales** para su integración en *Breathe Barcelona*, transformando los outputs originales en estructuras optimizadas para la visualización interactiva.

Se procesan dos fuentes principales:

- **predicciones *rolling* de 2024**, con la evolución diaria de los valores observados y predichos para NO₂, PM₁₀ y PM₂.₅;
- **métricas de evaluación**, utilizadas para comparar el rendimiento de las distintas familias y configuraciones de modelos.

Las predicciones se conservan completas para mantener la **continuidad de las series temporales**, incorporando un indicador que permite distinguir las fechas con observación real disponible de aquellas para las que únicamente existe predicción. De este modo, las comparaciones observado–predicho pueden realizarse únicamente cuando ambos valores están disponibles.

Finalmente, los resultados se ordenan y almacenan en formato **Parquet**, reduciendo los tiempos de lectura durante la ejecución del dashboard.

> **Objetivo:** disponer de una fuente optimizada y metodológicamente consistente para visualizar la evolución temporal, las predicciones y el rendimiento de los modelos de series temporales.

In [15]:
# ============================================================
# BREATHE BARCELONA
# 05.2 · Preparación de outputs de series temporales
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Rutas de origen
# ------------------------------------------------------------

ruta_pred_rolling = (
    BASE_DIR /
    "13_Modelos_Series_Temporales/"
    "LF04_predicciones_rolling_2024.csv"
)

ruta_metricas_rolling = (
    BASE_DIR /
    "13_Modelos_Series_Temporales/"
    "LF04_metricas_modelos_rolling.csv"
)

# ------------------------------------------------------------
# Lectura
# ------------------------------------------------------------

df_series_pred = pd.read_csv(
    ruta_pred_rolling,
    parse_dates=["fecha"]
)

df_series_metricas = pd.read_csv(
    ruta_metricas_rolling
)

# ------------------------------------------------------------
# Protección de las comparaciones observado/predicho
# ------------------------------------------------------------
# Las predicciones se conservan completas para mantener la
# continuidad temporal, pero sabemos qué fechas disponen
# realmente de observación.

df_series_pred["dato_observado_disponible"] = (
    df_series_pred["dato_observado_disponible"]
    .astype(bool)
)

# ------------------------------------------------------------
# Orden
# ------------------------------------------------------------

df_series_pred = (
    df_series_pred
    .sort_values(["contaminante", "fecha"])
    .reset_index(drop=True)
)

df_series_metricas = (
    df_series_metricas
    .sort_values(["contaminante", "familia", "modelo"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Guardado en formato Parquet
# ------------------------------------------------------------

ruta_pred_dash = DATA_DIR / "dashboard_series_predicciones.parquet"
ruta_metricas_dash = DATA_DIR / "dashboard_series_metricas.parquet"

df_series_pred.to_parquet(
    ruta_pred_dash,
    index=False
)

df_series_metricas.to_parquet(
    ruta_metricas_dash,
    index=False
)

# ------------------------------------------------------------
# Verificación
# ------------------------------------------------------------

print("=" * 90)
print("📈 BREATHE BARCELONA — SERIES TEMPORALES")
print("=" * 90)

print("\n✅ Predicciones:")
print(f"   {len(df_series_pred):,} registros")
print(
    f"   {df_series_pred['fecha'].min().date()} "
    f"→ {df_series_pred['fecha'].max().date()}"
)
print(
    "   Contaminantes:",
    ", ".join(df_series_pred["contaminante"].unique())
)

n_observados = df_series_pred["dato_observado_disponible"].sum()
n_sin_observar = (~df_series_pred["dato_observado_disponible"]).sum()

print(f"   Con observación: {n_observados:,}")
print(f"   Sin observación: {n_sin_observar:,}")

print("\n✅ Métricas:")
print(f"   {len(df_series_metricas):,} registros")

print("\nArchivos creados:")
print(f"   • {ruta_pred_dash}")
print(f"   • {ruta_metricas_dash}")

print("\n" + "=" * 90)
print("✅ OUTPUTS DE SERIES TEMPORALES PREPARADOS")
print("=" * 90)

📈 BREATHE BARCELONA — SERIES TEMPORALES

✅ Predicciones:
   1,098 registros
   2024-01-01 → 2024-12-31
   Contaminantes: NO2, PM10, PM2.5
   Con observación: 1,063
   Sin observación: 35

✅ Métricas:
   9 registros

Archivos creados:
   • /content/drive/MyDrive/TFM/12_Dashboard/01_Datos/dashboard_series_predicciones.parquet
   • /content/drive/MyDrive/TFM/12_Dashboard/01_Datos/dashboard_series_metricas.parquet

✅ OUTPUTS DE SERIES TEMPORALES PREPARADOS


### Resultados

El módulo de series temporales queda preparado a partir de **1.098 predicciones diarias correspondientes a 2024**, incorporando los tres contaminantes modelizados: **NO₂, PM₁₀ y PM₂.₅**.

La estructura conserva la serie completa de predicciones para garantizar la **continuidad temporal de las visualizaciones**. De los registros disponibles:

- **1.063** cuentan con una observación real asociada;
- **35** corresponden a fechas sin dato observado disponible.

Estas fechas permanecen en el dataset para mantener la continuidad de las predicciones, pero quedan explícitamente identificadas mediante `dato_observado_disponible`, evitando su utilización en comparaciones o métricas que requieran valores reales.

Adicionalmente, se prepara una tabla independiente con **9 registros de métricas**, destinada a comparar el rendimiento de las diferentes familias y configuraciones de modelos de series temporales.

Los resultados se almacenan en dos archivos optimizados:

- `dashboard_series_predicciones.parquet` · evolución temporal y comparación observado–predicho;
- `dashboard_series_metricas.parquet` · evaluación comparativa del rendimiento de los modelos.

> **Resultado:** el módulo de series temporales queda preparado para representar de forma diferenciada **observaciones, predicciones y métricas de rendimiento**, preservando la integridad de las comparaciones.

### 5.3 · Preparación de *Model Insights*

Esta etapa prepara los resultados necesarios para incorporar al dashboard una visión integrada del **rendimiento, capacidad de generalización e interpretabilidad del modelo temporal de Machine Learning**.

Se procesan tres componentes complementarios:

- **predicciones sobre el conjunto TEST**, incluyendo las clases reales, las clases predichas y las probabilidades estimadas para cada categoría de calidad del aire;
- **validación espacial Leave-One-Station-Out (LOSO)**, utilizada para evaluar la capacidad del modelo de generalizar hacia estaciones no utilizadas durante el entrenamiento;
- **importancia de variables mediante SHAP**, que permite identificar qué características presentan una mayor contribución global a las predicciones del modelo.

Antes de generar los archivos del dashboard se comprueba la presencia de las variables esenciales para las predicciones y la validación espacial. Los resultados se almacenan posteriormente en formato **Parquet** para facilitar su consulta desde la aplicación.

> **Objetivo:** proporcionar al módulo *Model Insights* información sobre **qué predice el modelo, cómo generaliza espacialmente y qué variables sustentan sus predicciones**.

In [16]:
# ============================================================
# BREATHE BARCELONA
# 05.3 · Preparación de Model Insights
# ============================================================

import pandas as pd

MODEL_DIR = (
    BASE_DIR /
    "11_Machine_Learning_Temporal/03_Datos_Modelado"
)

# ------------------------------------------------------------
# Archivos de origen
# ------------------------------------------------------------

ruta_pred = MODEL_DIR / "modelo_final_predicciones_TEST.csv"
ruta_loso = MODEL_DIR / "validacion_espacial_LOSO_resultados.csv"
ruta_shap = MODEL_DIR / "modelo_final_shap_importance.csv"

# ------------------------------------------------------------
# Lectura
# ------------------------------------------------------------

df_model_pred = pd.read_csv(ruta_pred)
df_loso = pd.read_csv(ruta_loso)
df_shap = pd.read_csv(ruta_shap)

# ------------------------------------------------------------
# Validaciones mínimas
# ------------------------------------------------------------

columnas_pred = {
    "y_real", "y_pred",
    "prob_Buena", "prob_Media", "prob_Mala"
}

columnas_loso = {
    "estacion", "accuracy",
    "balanced_accuracy", "f1_macro", "f1_weighted"
}

if not columnas_pred.issubset(df_model_pred.columns):
    raise ValueError(
        "Faltan columnas esperadas en las predicciones temporales."
    )

if not columnas_loso.issubset(df_loso.columns):
    raise ValueError(
        "Faltan columnas esperadas en la validación LOSO."
    )

# ------------------------------------------------------------
# Guardado optimizado
# ------------------------------------------------------------

ruta_pred_dash = DATA_DIR / "dashboard_modelo_temporal_predicciones.parquet"
ruta_loso_dash = DATA_DIR / "dashboard_modelo_temporal_LOSO.parquet"
ruta_shap_dash = DATA_DIR / "dashboard_modelo_temporal_SHAP.parquet"

df_model_pred.to_parquet(ruta_pred_dash, index=False)
df_loso.to_parquet(ruta_loso_dash, index=False)
df_shap.to_parquet(ruta_shap_dash, index=False)

# ------------------------------------------------------------
# Resumen
# ------------------------------------------------------------

print("=" * 90)
print("🤖 BREATHE BARCELONA — MODEL INSIGHTS")
print("=" * 90)

print(f"\nPredicciones TEST : {len(df_model_pred):,} registros")
print(f"Estaciones LOSO   : {len(df_loso):,}")
print(f"Variables SHAP    : {len(df_shap):,}")

print("\nColumnas SHAP:")
for col in df_shap.columns:
    print(f"   • {col}")

print("\nResumen LOSO:")
print(
    df_loso[
        ["accuracy", "balanced_accuracy", "f1_macro", "f1_weighted"]
    ]
    .mean()
    .round(4)
    .to_string()
)

print("\nArchivos creados:")
print(f"   • {ruta_pred_dash.name}")
print(f"   • {ruta_loso_dash.name}")
print(f"   • {ruta_shap_dash.name}")

print("\n" + "=" * 90)
print("✅ MODEL INSIGHTS PREPARADO")
print("=" * 90)

🤖 BREATHE BARCELONA — MODEL INSIGHTS

Predicciones TEST : 2,506 registros
Estaciones LOSO   : 8
Variables SHAP    : 122

Columnas SHAP:
   • feature
   • mean_abs_shap

Resumen LOSO:
accuracy             0.8597
balanced_accuracy    0.6276
f1_macro             0.6497
f1_weighted          0.8539

Archivos creados:
   • dashboard_modelo_temporal_predicciones.parquet
   • dashboard_modelo_temporal_LOSO.parquet
   • dashboard_modelo_temporal_SHAP.parquet

✅ MODEL INSIGHTS PREPARADO


### Resultados

La preparación del módulo genera tres fuentes complementarias para analizar el comportamiento del modelo temporal de *Breathe Barcelona*:

- **2.506 predicciones** correspondientes al conjunto TEST;
- resultados de validación **LOSO para las 8 estaciones** incluidas en el estudio;
- importancia global mediante **SHAP para 122 variables explicativas**.

La validación espacial LOSO alcanza, en promedio, una **accuracy de 0,860** y un **F1 weighted de 0,854**. Al considerar de forma equilibrada el comportamiento entre clases, los valores descienden a una **balanced accuracy de 0,628** y un **F1 macro de 0,650**.

Esta diferencia entre las métricas globales y balanceadas resulta especialmente relevante en un problema con **distribución desigual de las clases**, por lo que el dashboard permitirá analizar el rendimiento más allá de la accuracy agregada y comparar la capacidad de generalización entre estaciones.

La incorporación de los valores **SHAP** añade una capa de interpretabilidad al modelo, permitiendo identificar las variables con mayor influencia global sobre sus predicciones.

> **Resultado:** el módulo *Model Insights* queda preparado para integrar **rendimiento predictivo, generalización espacial e interpretabilidad** dentro de una misma sección del dashboard.

### 5.4 · Dataset geoespacial definitivo y reconstrucción reproducible de LISA

Esta etapa construye la **fuente geoespacial definitiva de Breathe Barcelona**, integrando la información territorial utilizada por el dashboard con los resultados del análisis de autocorrelación espacial local del NO₂.

Para garantizar la **reproducibilidad del análisis**, los indicadores LISA se recalculan desde cero a partir de la concentración media de NO₂ de las **1.068 secciones censales**. Se utiliza una matriz de pesos espaciales basada en **contigüidad Queen**, estandarizada por filas, y el estadístico **Moran Local** con **999 permutaciones y semilla fija (`seed = 42`)**.

El procedimiento comprende:

- validación de la geometría y de la variable espacial de NO₂;
- construcción y comprobación de la matriz de vecindad Queen;
- recálculo reproducible del estadístico **Moran Local (LISA)**;
- identificación de asociaciones espaciales **High–High, Low–Low, Low–High y High–Low**, considerando significativos los resultados con *p* < 0,05;
- integración de los resultados LISA con el dataset geoespacial utilizado por la aplicación;
- verificación de que la unión conserva íntegramente las **1.068 secciones** y sus resultados espaciales;
- transformación final a **WGS84 (EPSG:4326)** para su utilización en cartografía web;
- creación de una copia de seguridad y validación del archivo definitivo tras su escritura.

> **Objetivo:** generar un dataset espacial único, validado y reproducible que permita representar en el dashboard tanto los patrones territoriales del NO₂ como sus **clusters locales de autocorrelación espacial**.

In [17]:
# ============================================================
# BREATHE BARCELONA
# 05.4 · Dataset geoespacial definitivo
#      · LISA recalculado de forma reproducible
# ============================================================

import geopandas as gpd
import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime
import shutil

from libpysal.weights import Queen
from esda.moran import Moran_Local


# ============================================================
# 1. RUTAS
# ============================================================

TFM_DIR = Path("/content/drive/MyDrive/TFM")

ruta_geo = (
    TFM_DIR /
    "12_Dashboard/01_Datos/dashboard_geoespacial.gpkg"
)

ruta_lisa = (
    TFM_DIR /
    "06_Analisis_Geoespacial/RESULTADOS/MORAN/LISA/"
    "LISA_NO2_secciones.gpkg"
)

ruta_salida = (
    TFM_DIR /
    "12_Dashboard/01_Datos/dashboard_spatial.gpkg"
)


# ============================================================
# 2. COMPROBACIÓN DE ARCHIVOS
# ============================================================

if not ruta_geo.exists():
    raise FileNotFoundError(
        f"❌ No existe:\n{ruta_geo}"
    )

if not ruta_lisa.exists():
    raise FileNotFoundError(
        f"❌ No existe:\n{ruta_lisa}"
    )


# ============================================================
# 3. LECTURA
# ============================================================

gdf_geo = gpd.read_file(ruta_geo)

# Este fichero se utiliza como base geométrica LISA.
# Sus columnas LISA antiguas se recalcularán desde cero.
gdf_lisa = gpd.read_file(ruta_lisa)


print("=" * 100)
print("🗺️ BREATHE BARCELONA — DATASET ESPACIAL DEFINITIVO")
print("=" * 100)

print(f"\nSecciones geoespaciales : {len(gdf_geo):,}")
print(f"Secciones base LISA     : {len(gdf_lisa):,}")
print(f"CRS LISA                : {gdf_lisa.crs}")


# ============================================================
# 4. VALIDACIÓN DE VARIABLES
# ============================================================

columnas_necesarias = [
    "CLAVE_SECCION",
    "NO2_MEDIO_SECCION",
    "geometry"
]

faltantes = [
    c for c in columnas_necesarias
    if c not in gdf_lisa.columns
]

if faltantes:
    raise KeyError(
        "❌ Faltan columnas necesarias: "
        + ", ".join(faltantes)
    )

if len(gdf_lisa) != 1068:
    raise ValueError(
        f"❌ Se esperaban 1.068 secciones y hay {len(gdf_lisa):,}."
    )


# ============================================================
# 5. VARIABLE NO2
# ============================================================

gdf_lisa["NO2_MEDIO_SECCION"] = pd.to_numeric(
    gdf_lisa["NO2_MEDIO_SECCION"],
    errors="coerce"
)

n_nulos = gdf_lisa["NO2_MEDIO_SECCION"].isna().sum()

if n_nulos > 0:
    raise ValueError(
        f"❌ Hay {n_nulos} valores nulos en NO2_MEDIO_SECCION."
    )

y_lisa = (
    gdf_lisa["NO2_MEDIO_SECCION"]
    .astype(float)
    .values
)

print("\nNO₂ utilizado:")
print(f"Media : {np.mean(y_lisa):.6f}")
print(f"Std   : {np.std(y_lisa, ddof=1):.6f}")

# Control de coherencia con el notebook geoespacial
if not np.isclose(
    np.mean(y_lisa),
    30.329684175554544,
    atol=1e-6
):
    raise ValueError(
        "❌ La media de NO₂ no coincide con la base definitiva."
    )


# ============================================================
# 6. MATRIZ ESPACIAL QUEEN
# ============================================================

print("\n" + "=" * 100)
print("MATRIZ ESPACIAL")
print("=" * 100)

w_lisa = Queen.from_dataframe(
    gdf_lisa,
    use_index=False
)

w_lisa.transform = "R"

print("Matriz espacial : Queen")
print(f"Secciones       : {w_lisa.n:,}")

n_vecinos = sum(
    len(v)
    for v in w_lisa.neighbors.values()
)

print(f"Vecinos totales : {n_vecinos:,}")
print(f"Islas           : {len(w_lisa.islands):,}")

if len(w_lisa.islands) > 0:
    raise ValueError(
        "❌ Existen islas espaciales."
    )

if n_vecinos != 6856:
    raise ValueError(
        f"❌ La matriz Queen tiene {n_vecinos:,} relaciones "
        "y se esperaban 6.856."
    )

print("✅ Pesos espaciales estandarizados por filas.")


# ============================================================
# 7. MORAN LOCAL — RECÁLCULO DEFINITIVO
# ============================================================

print("\n" + "=" * 100)
print("MORAN LOCAL — LISA")
print("=" * 100)

lisa_model = Moran_Local(
    y_lisa,
    w_lisa,
    permutations=999,
    seed=42
)

gdf_lisa["LISA_I"] = lisa_model.Is
gdf_lisa["LISA_P"] = lisa_model.p_sim
gdf_lisa["LISA_Q"] = lisa_model.q

print("✅ Moran Local recalculado.")
print("Permutaciones : 999")
print("Semilla       : 42")
print(
    f"Media Moran Local : "
    f"{np.mean(lisa_model.Is):.6f}"
)


# ============================================================
# 8. CLASIFICACIÓN LISA CORRECTA
# ============================================================

# Moran Local:
# 1 = High-High
# 2 = Low-High
# 3 = Low-Low
# 4 = High-Low

def clasificar_lisa(row):

    if row["LISA_P"] >= 0.05:
        return "No significativo"

    q = row["LISA_Q"]

    if q == 1:
        return "High-High"

    elif q == 2:
        return "Low-High"

    elif q == 3:
        return "Low-Low"

    elif q == 4:
        return "High-Low"

    return "No significativo"


gdf_lisa["LISA_CLUSTER"] = gdf_lisa.apply(
    clasificar_lisa,
    axis=1
)

gdf_lisa["LISA_SIGNIFICATIVO"] = (
    gdf_lisa["LISA_P"] < 0.05
)


# ============================================================
# 9. COLORES LISA
# ============================================================

color_lisa = {
    "High-High": "#EF6A6B",
    "Low-Low": "#4B97E4",
    "Low-High": "#43D2C1",
    "High-Low": "#E9BE59",
    "No significativo": "#243C46"
}

gdf_lisa["COLOR_LISA"] = (
    gdf_lisa["LISA_CLUSTER"]
    .map(color_lisa)
)


# ============================================================
# 10. VALIDACIÓN DEL RESULTADO LISA
# ============================================================

conteos_lisa = (
    gdf_lisa["LISA_CLUSTER"]
    .value_counts()
)

n_significativos = int(
    gdf_lisa["LISA_SIGNIFICATIVO"]
    .sum()
)

print("\n" + "=" * 100)
print("RESULTADOS LISA RECALCULADOS")
print("=" * 100)

print(conteos_lisa)

print(
    f"\nSecciones significativas p < 0.05 : "
    f"{n_significativos:,}"
)

esperado = {
    "High-High": 189,
    "Low-Low": 210,
    "Low-High": 3,
    "High-Low": 10,
    "No significativo": 656
}

for cluster, n_esperado in esperado.items():

    n_real = int(
        conteos_lisa.get(
            cluster,
            0
        )
    )

    if n_real != n_esperado:
        raise ValueError(
            f"❌ {cluster}: "
            f"{n_real} obtenido / "
            f"{n_esperado} esperado."
        )

if n_significativos != 412:
    raise ValueError(
        f"❌ Significativos: "
        f"{n_significativos} obtenidos / "
        "412 esperados."
    )

print("\n✅ LISA reproducido exactamente:")
print("   High-High          : 189")
print("   Low-Low            : 210")
print("   Low-High           : 3")
print("   High-Low           : 10")
print("   No significativo   : 656")
print("   Significativos     : 412")


# ============================================================
# 11. VARIABLES LISA PARA EL DASHBOARD
# ============================================================

columnas_lisa = [
    "CLAVE_SECCION",
    "LISA_I",
    "LISA_P",
    "LISA_SIGNIFICATIVO",
    "LISA_CLUSTER",
    "COLOR_LISA"
]

lisa_df = (
    gdf_lisa[
        columnas_lisa
    ]
    .copy()
)


# ============================================================
# 12. UNIÓN CON DATASET GEOESPACIAL DEL DASHBOARD
# ============================================================

if "CLAVE_SECCION" in gdf_geo.columns:

    gdf_spatial = gdf_geo.merge(
        lisa_df,
        on="CLAVE_SECCION",
        how="left"
    )

    metodo_union = "CLAVE_SECCION"

else:

    # --------------------------------------------------------
    # Si dashboard_geoespacial no dispone de CLAVE_SECCION,
    # se usa unión espacial mediante punto representativo.
    # --------------------------------------------------------

    geo_base = gdf_geo.copy()

    lisa_ref = gdf_lisa[
        columnas_lisa
        + ["geometry"]
    ].copy()

    if geo_base.crs != lisa_ref.crs:
        lisa_ref = lisa_ref.to_crs(
            geo_base.crs
        )

    puntos = geo_base.copy()

    puntos["geometry"] = (
        puntos.geometry
        .representative_point()
    )

    gdf_spatial = gpd.sjoin(
        puntos,
        lisa_ref,
        how="left",
        predicate="within"
    )

    # Recuperar polígonos originales
    gdf_spatial["geometry"] = (
        geo_base.geometry.values
    )

    if "index_right" in gdf_spatial.columns:
        gdf_spatial = gdf_spatial.drop(
            columns="index_right"
        )

    metodo_union = "spatial join"


# ============================================================
# 13. VALIDACIÓN DE LA UNIÓN
# ============================================================

print("\n" + "=" * 100)
print("VALIDACIÓN DEL DATASET DEL DASHBOARD")
print("=" * 100)

print(f"Método de unión : {metodo_union}")
print(f"Resultado       : {len(gdf_spatial):,} secciones")

n_sin_lisa = (
    gdf_spatial["LISA_CLUSTER"]
    .isna()
    .sum()
)

print(
    f"Sin correspondencia LISA : "
    f"{n_sin_lisa:,}"
)

if len(gdf_spatial) != 1068:
    raise ValueError(
        "❌ El dataset final no contiene 1.068 secciones."
    )

if n_sin_lisa > 0:
    raise ValueError(
        "❌ Existen secciones sin correspondencia LISA."
    )

conteos_finales = (
    gdf_spatial["LISA_CLUSTER"]
    .value_counts()
)

print("\nConteos después de la unión:")
print(conteos_finales)

for cluster, n_esperado in esperado.items():

    n_real = int(
        conteos_finales.get(
            cluster,
            0
        )
    )

    if n_real != n_esperado:
        raise ValueError(
            f"❌ Tras la unión, {cluster}: "
            f"{n_real} / {n_esperado}"
        )

print(
    "\n✅ La unión conserva exactamente "
    "los resultados LISA."
)


# ============================================================
# 14. TRANSFORMACIÓN A WGS84
# ============================================================

gdf_spatial = gdf_spatial.to_crs(
    epsg=4326
)


# ============================================================
# 15. BACKUP DEL ARCHIVO ACTUAL
# ============================================================

if ruta_salida.exists():

    marca = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    ruta_backup = (
        ruta_salida.parent /
        f"dashboard_spatial_backup_{marca}.gpkg"
    )

    shutil.copy2(
        ruta_salida,
        ruta_backup
    )

    print("\n📦 Backup creado:")
    print(f"   {ruta_backup}")


# ============================================================
# 16. GUARDADO
# ============================================================

gdf_spatial.to_file(
    ruta_salida,
    driver="GPKG"
)


# ============================================================
# 17. VERIFICACIÓN DEL ARCHIVO GUARDADO
# ============================================================

gdf_check = gpd.read_file(
    ruta_salida
)

conteos_guardados = (
    gdf_check["LISA_CLUSTER"]
    .value_counts()
)

print("\n" + "=" * 100)
print("VERIFICACIÓN DEL ARCHIVO GUARDADO")
print("=" * 100)

print(conteos_guardados)

for cluster, n_esperado in esperado.items():

    n_real = int(
        conteos_guardados.get(
            cluster,
            0
        )
    )

    if n_real != n_esperado:
        raise ValueError(
            f"❌ Archivo guardado incorrecto: "
            f"{cluster} = {n_real}"
        )

tamano_mb = (
    ruta_salida.stat().st_size /
    (1024 ** 2)
)

print(f"\nCRS final : {gdf_check.crs}")
print(f"Tamaño    : {tamano_mb:.2f} MB")

print("\nArchivo creado:")
print(f"   {ruta_salida}")

print("\n" + "=" * 100)
print("✅ DATASET ESPACIAL DEFINITIVO PREPARADO")
print("✅ MORAN LOCAL RECALCULADO CON SEED = 42")
print("✅ HH 189 · LL 210 · LH 3 · HL 10 · NS 656")
print("✅ 412 SECCIONES SIGNIFICATIVAS")
print("=" * 100)

🗺️ BREATHE BARCELONA — DATASET ESPACIAL DEFINITIVO

Secciones geoespaciales : 1,068
Secciones base LISA     : 1,068
CRS LISA                : EPSG:25831

NO₂ utilizado:
Media : 30.329684
Std   : 5.403082

MATRIZ ESPACIAL
Matriz espacial : Queen
Secciones       : 1,068
Vecinos totales : 6,856
Islas           : 0
✅ Pesos espaciales estandarizados por filas.

MORAN LOCAL — LISA
✅ Moran Local recalculado.
Permutaciones : 999
Semilla       : 42
Media Moran Local : 0.781417

RESULTADOS LISA RECALCULADOS
LISA_CLUSTER
No significativo    656
Low-Low             210
High-High           189
High-Low             10
Low-High              3
Name: count, dtype: int64

Secciones significativas p < 0.05 : 412

✅ LISA reproducido exactamente:
   High-High          : 189
   Low-Low            : 210
   Low-High           : 3
   High-Low           : 10
   No significativo   : 656
   Significativos     : 412

VALIDACIÓN DEL DATASET DEL DASHBOARD
Método de unión : spatial join
Resultado       : 1,068 seccio

### Resultados

El procesamiento genera correctamente el **dataset espacial definitivo de Breathe Barcelona**, conservando las **1.068 secciones censales** y reproduciendo de forma controlada el análisis de autocorrelación espacial local del NO₂.

La matriz de pesos espaciales basada en **contigüidad Queen** contiene **6.856 relaciones de vecindad y ninguna isla espacial**, garantizando que todas las secciones participan en la estructura de dependencia territorial.

El recálculo de **Moran Local**, realizado con **999 permutaciones y `seed = 42`**, obtiene un valor medio de **I = 0,781**, evidenciando una marcada estructura de autocorrelación espacial local en la distribución del NO₂.

De las 1.068 secciones analizadas, **412 presentan asociación espacial local estadísticamente significativa (*p* < 0,05)**:

- **189 High–High** · concentraciones elevadas rodeadas de valores igualmente elevados;
- **210 Low–Low** · concentraciones reducidas asociadas espacialmente con valores bajos;
- **10 High–Low** · valores elevados en entornos predominantemente bajos;
- **3 Low–High** · valores reducidos en entornos predominantemente elevados;
- **656 no significativas**.

La integración mediante **spatial join** conserva la totalidad de las secciones y reproduce exactamente la clasificación LISA, sin registros sin correspondencia.

Finalmente, el dataset se transforma a **WGS84 (EPSG:4326)** para su representación cartográfica en entorno web y se almacena como `dashboard_spatial.gpkg`, con un tamaño aproximado de **1,64 MB**. Antes de sobrescribir la versión anterior se genera automáticamente una copia de seguridad.

> **Resultado:** `dashboard_spatial.gpkg` queda validado como fuente geoespacial definitiva del dashboard, integrando en una única estructura la información territorial y los **patrones locales de autocorrelación espacial del NO₂**.

### 5.5 · Inspección del artefacto de modelización

Antes de integrar la capacidad predictiva en el dashboard se realiza una **inspección del artefacto correspondiente al modelo XGBoost geoespacial previamente entrenado**.

El modelo, almacenado en formato `joblib`, se carga directamente desde el directorio de artefactos de *Breathe Barcelona* para comprobar su integridad y recuperar la información necesaria para su posterior utilización.

La inspección permite verificar:

- el **tipo de objeto** almacenado;
- el **número de variables de entrada** esperado por el modelo;
- los **nombres y el orden de las features**, cuando esta información está disponible en el artefacto;
- los principales **hiperparámetros de XGBoost**, como número de estimadores, profundidad máxima, *learning rate*, submuestreo y semilla aleatoria.

Esta comprobación resulta especialmente importante antes de conectar el modelo con nuevos datos, ya que permite garantizar que la estructura utilizada durante la inferencia sea compatible con la empleada durante el entrenamiento.

> **Objetivo:** validar el artefacto de modelización y recuperar su esquema de entrada antes de incorporarlo al módulo predictivo del dashboard.

In [18]:
# ============================================================
# BREATHE BARCELONA
# 05.5 · Inspección del artefacto de modelización
# ============================================================

import joblib
from pathlib import Path

ruta_modelo = (
    BASE_DIR /
    "12_Dashboard/05_Modelos/"
    "XGBoost_geoespacial.joblib"
)

print("=" * 90)
print("🧠 BREATHE BARCELONA — ARTEFACTO DE MODELO")
print("=" * 90)

if not ruta_modelo.exists():
    raise FileNotFoundError(
        f"No se encuentra el artefacto:\n{ruta_modelo}"
    )

modelo_geo = joblib.load(ruta_modelo)

print("\n✅ Artefacto cargado correctamente")
print(f"Ruta: {ruta_modelo}")

print("\nTipo de objeto:")
print(f"   {type(modelo_geo)}")

# ------------------------------------------------------------
# Información disponible en el modelo
# ------------------------------------------------------------

if hasattr(modelo_geo, "n_features_in_"):
    print(
        f"\nNúmero de variables de entrada: "
        f"{modelo_geo.n_features_in_}"
    )

if hasattr(modelo_geo, "feature_names_in_"):
    print("\nVariables esperadas por el modelo:")

    for i, feature in enumerate(
        modelo_geo.feature_names_in_, start=1
    ):
        print(f"   {i:02d}. {feature}")

else:
    print(
        "\n⚠️ El artefacto no contiene "
        "`feature_names_in_`."
    )

if hasattr(modelo_geo, "get_params"):
    params = modelo_geo.get_params()

    print("\nParámetros identificativos:")
    for parametro in [
        "n_estimators",
        "max_depth",
        "learning_rate",
        "subsample",
        "colsample_bytree",
        "random_state"
    ]:
        if parametro in params:
            print(
                f"   {parametro:<20}: "
                f"{params[parametro]}"
            )

print("\n" + "=" * 90)
print("✅ INSPECCIÓN DEL MODELO FINALIZADA")
print("=" * 90)

🧠 BREATHE BARCELONA — ARTEFACTO DE MODELO

✅ Artefacto cargado correctamente
Ruta: /content/drive/MyDrive/TFM/12_Dashboard/05_Modelos/XGBoost_geoespacial.joblib

Tipo de objeto:
   <class 'xgboost.sklearn.XGBRegressor'>

Número de variables de entrada: 5

Variables esperadas por el modelo:
   01. RENTA_EUR_PERSONA
   02. DENSIDAD_HAB_HA
   03. ELEVACION_MEDIA_M
   04. DENSIDAD_RED_KM_KM2
   05. DISTANCIA_AFORO_M

Parámetros identificativos:
   n_estimators        : 900
   max_depth           : 4
   learning_rate       : 0.025
   subsample           : 0.85
   colsample_bytree    : 0.85
   random_state        : 42

✅ INSPECCIÓN DEL MODELO FINALIZADA


/usr/lib/python3.13/pickle.py:1754: UserWarning: [11:28:49] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


### Resultados

El artefacto correspondiente al modelo **XGBoost geoespacial** se carga correctamente desde el repositorio de modelos del dashboard, confirmando que el objeto serializado está disponible para su utilización durante la fase de inferencia.

La inspección permite recuperar la **estructura de entrada esperada por el modelo** y sus principales parámetros de configuración. Esta información establece el contrato que deberán respetar los nuevos datos antes de generar predicciones desde la aplicación.

La conservación del modelo como artefacto independiente evita la necesidad de volver a entrenarlo durante la ejecución de *Breathe Barcelona*, separando claramente las etapas de **entrenamiento e inferencia** y reduciendo el coste computacional del dashboard.

> **Resultado:** el modelo geoespacial queda disponible y verificado para su posterior integración en el módulo predictivo de *Breathe Barcelona*.

### 5.6 · Dominio de entrada del Predictor

Antes de habilitar el módulo de predicción interactiva se define el **dominio de entrada de las variables modificables por el usuario**, utilizando como referencia la distribución observada en el dataset geoespacial de Barcelona.

El predictor considera cinco variables territoriales:

- **renta por persona** (`RENTA_EUR_PERSONA`);
- **densidad de población** (`DENSIDAD_HAB_HA`);
- **elevación media** (`ELEVACION_MEDIA_M`);
- **densidad de la red viaria** (`DENSIDAD_RED_KM_KM2`);
- **distancia al punto de aforo más próximo** (`DISTANCIA_AFORO_M`).

Para cada variable se calculan el **mínimo, primer cuartil, mediana, tercer cuartil y máximo** observados. Estos estadísticos permiten caracterizar el espacio de valores sobre el que se ha construido la información geoespacial utilizada por el modelo.

El dominio resultante se almacena en formato **Parquet** para que Streamlit pueda utilizarlo directamente en la configuración de controles, rangos y valores de referencia del predictor.

> **Objetivo:** mantener las simulaciones interactivas dentro de un dominio coherente con los datos observados y proporcionar valores de referencia para la configuración de los controles del predictor.

In [19]:
# ============================================================
# BREATHE BARCELONA
# 05.6 · Dominio de entrada del Predictor
# ============================================================

variables_predictor = [
    "RENTA_EUR_PERSONA",
    "DENSIDAD_HAB_HA",
    "ELEVACION_MEDIA_M",
    "DENSIDAD_RED_KM_KM2",
    "DISTANCIA_AFORO_M"
]

# ------------------------------------------------------------
# Estadísticos del dominio observado
# ------------------------------------------------------------

resumen_predictor = (
    gdf_geo[variables_predictor]
    .describe(percentiles=[0.25, 0.50, 0.75])
    .T[
        ["min", "25%", "50%", "75%", "max"]
    ]
    .round(2)
)

resumen_predictor.columns = [
    "min",
    "q25",
    "mediana",
    "q75",
    "max"
]

# ------------------------------------------------------------
# Guardado para consumo directo desde Streamlit
# ------------------------------------------------------------

ruta_dominio = (
    DATA_DIR /
    "dashboard_predictor_dominio.parquet"
)

resumen_predictor.reset_index(
    names="variable"
).to_parquet(
    ruta_dominio,
    index=False
)

# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

print("=" * 90)
print("🎯 BREATHE BARCELONA — DOMINIO DEL PREDICTOR")
print("=" * 90)

print()
print(resumen_predictor.to_string())

print("\nArchivo creado:")
print(f"   {ruta_dominio}")

print("\n" + "=" * 90)
print("✅ DOMINIO DEL PREDICTOR PREPARADO")
print("=" * 90)

🎯 BREATHE BARCELONA — DOMINIO DEL PREDICTOR

                          min       q25   mediana       q75       max
RENTA_EUR_PERSONA    11097.00  21346.75  24926.00  28946.75  54089.00
DENSIDAD_HAB_HA          1.00    211.00    322.00    426.00    593.00
ELEVACION_MEDIA_M        2.54     18.37     36.02     69.84    341.03
DENSIDAD_RED_KM_KM2      3.30     15.55     20.91     26.18     78.28
DISTANCIA_AFORO_M       11.23    117.05    194.44    294.97   2852.44

Archivo creado:
   /content/drive/MyDrive/TFM/12_Dashboard/01_Datos/dashboard_predictor_dominio.parquet

✅ DOMINIO DEL PREDICTOR PREPARADO


### Resultado · Dominio del Predictor

El dominio empírico de las variables de entrada queda definido a partir de la distribución observada en las secciones censales analizadas.

Los valores muestran una **heterogeneidad territorial considerable** en las cinco dimensiones utilizadas por el predictor:

- la **renta por persona** oscila entre **11.097 € y 54.089 €**, con una mediana de **24.926 €**;
- la **densidad de población** presenta valores entre **1 y 593 hab/ha**, con una mediana de **322 hab/ha**;
- la **elevación media** varía entre **2,54 y 341,03 m**, reflejando el gradiente topográfico de la ciudad;
- la **densidad de red viaria** se sitúa entre **3,30 y 78,28 km/km²**, con una mediana de **20,91 km/km²**;
- la **distancia al aforo más próximo** presenta el rango más asimétrico, desde **11,23 hasta 2.852,44 m**, aunque el 75 % de las secciones se sitúa por debajo de aproximadamente **295 m**.

Los cuartiles permiten además establecer **valores centrales y rangos habituales** para configurar los controles interactivos, evitando que la experiencia de simulación dependa únicamente de los valores extremos observados.

> **Resultado:** `dashboard_predictor_dominio.parquet` queda preparado como referencia para definir los rangos y valores iniciales del **Predictor interactivo de Breathe Barcelona**.

## 6 · Construcción de *Breathe Barcelona*

Una vez finalizadas las etapas de **preparación, modelización y validación de los datos**, se inicia la construcción de **Breathe Barcelona**, la capa interactiva de explotación y comunicación de los resultados del proyecto.

La aplicación se desarrolla con **Streamlit** como un componente independiente del pipeline analítico. Su arquitectura se basa en el consumo de **datasets específicamente optimizados para visualización, resultados analíticos previamente validados y modelos ya entrenados**, evitando repetir durante la ejecución los procesos computacionalmente más costosos de preparación, análisis espacial o entrenamiento.

La aplicación integra cuatro dimensiones principales del estudio —**temporal, espacial, predictiva e interpretativa**— y las organiza en cinco vistas:

1. **Overview** — visión sintética del estado y evolución de la calidad del aire en Barcelona, proporcionando acceso a los principales indicadores del estudio.

2. **Temporal** — exploración de la evolución de los contaminantes entre **2019 y 2024**, comparación de los periodos **Pre-COVID, COVID y Post-COVID** y visualización de los resultados de los modelos de series temporales.

3. **Spatial** — análisis de la distribución territorial del NO₂ mediante las **1.068 secciones censales**, incorporando patrones de concentración, autocorrelación espacial local **LISA** y resultados de la modelización geoespacial.

4. **Model Insights** — evaluación del rendimiento de los modelos mediante predicciones sobre TEST, validación espacial **Leave-One-Station-Out (LOSO)** e interpretación global mediante **SHAP**.

5. **Predictor** — simulación interactiva de la concentración espacial de **NO₂** mediante el modelo **XGBoost geoespacial** previamente entrenado, restringiendo las entradas al dominio observado en los datos de Barcelona.

La arquitectura separa, por tanto, la **generación del conocimiento analítico** de su **explotación interactiva**: los cálculos y modelos se desarrollan y validan previamente, mientras que Streamlit actúa como una capa ligera de consulta, visualización y simulación.

El desarrollo de la aplicación se realiza de forma incremental. Primero se establece y valida la **arquitectura funcional y la navegación**; posteriormente se incorporan los componentes gráficos, cartográficos e interactivos de cada módulo.

La interfaz adopta un formato panorámico y una identidad visual consistente basada en **azul petróleo y turquesa**, reservando **verde, ámbar y rojo** para transmitir de forma semántica las diferentes categorías y niveles de calidad del aire.

> **Objetivo:** transformar los resultados técnicos del proyecto en una herramienta interactiva que permita **explorar, interpretar y comunicar los patrones temporales y espaciales de la contaminación atmosférica de Barcelona**, manteniendo la trazabilidad con el pipeline analítico original.

### 6.0 · Localización de las fuentes principales del dashboard

Antes de iniciar la construcción de la aplicación se verifica la **localización física de las dos fuentes de datos principales** que alimentarán las vistas temporal y espacial de *Breathe Barcelona*.

La búsqueda se realiza de forma recursiva dentro del directorio raíz del proyecto para localizar:

- `dashboard_temporal.parquet` · dataset optimizado para la exploración temporal;
- `dashboard_spatial.gpkg` · dataset geoespacial definitivo, incluyendo los resultados de autocorrelación espacial LISA.

Esta comprobación permite confirmar las rutas reales de los archivos antes de incorporarlas a la configuración de Streamlit.

> **Objetivo:** verificar que las fuentes temporal y espacial definitivas están disponibles y correctamente localizadas antes de iniciar la construcción funcional de la aplicación.

In [20]:
# ============================================================
# 06.0 · LOCALIZAR ARCHIVOS REALES DEL DASHBOARD
# ============================================================

from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/TFM"
)

print("=" * 90)
print("BUSCANDO ARCHIVOS DEL DASHBOARD")
print("=" * 90)


# ============================================================
# BUSCAR PARQUET TEMPORAL
# ============================================================

temporal_candidates = list(
    ROOT.rglob(
        "dashboard_temporal.parquet"
    )
)


# ============================================================
# BUSCAR GPKG ESPACIAL
# ============================================================

spatial_candidates = list(
    ROOT.rglob(
        "dashboard_spatial.gpkg"
    )
)


print("\nTEMPORAL:")

if temporal_candidates:

    for p in temporal_candidates:
        print("✅", p)

else:

    print(
        "❌ No se encontró "
        "dashboard_temporal.parquet"
    )


print("\nSPATIAL:")

if spatial_candidates:

    for p in spatial_candidates:
        print("✅", p)

else:

    print(
        "❌ No se encontró "
        "dashboard_spatial.gpkg"
    )


print("\n" + "=" * 90)
print("FIN DE BÚSQUEDA")
print("=" * 90)

BUSCANDO ARCHIVOS DEL DASHBOARD

TEMPORAL:
✅ /content/drive/MyDrive/TFM/12_Dashboard/01_Datos/dashboard_temporal.parquet

SPATIAL:
✅ /content/drive/MyDrive/TFM/12_Dashboard/01_Datos/dashboard_spatial.gpkg

FIN DE BÚSQUEDA


### Resultados

La búsqueda confirma la disponibilidad de las dos fuentes de datos fundamentales de *Breathe Barcelona*:

- **fuente temporal:** `dashboard_temporal.parquet`;
- **fuente espacial:** `dashboard_spatial.gpkg`.

Ambos archivos se encuentran correctamente almacenados en `12_Dashboard/01_Datos`, confirmando que la aplicación dispone de acceso a los datasets definitivos preparados en las etapas anteriores.

> **Resultado:** las fuentes temporal y espacial quedan correctamente localizadas y disponibles para su conexión con la arquitectura de Streamlit.

### 6.1 · Configuración y comprobación de las fuentes

Una vez localizados los datasets definitivos, se establece la **configuración base de Breathe Barcelona**, definiendo las rutas que utilizará la aplicación para acceder a sus principales fuentes de información.

En esta comprobación se:

- define el directorio principal del dashboard y la ubicación de `app.py`;
- establece la ruta del **dataset temporal optimizado**;
- establece la ruta del **dataset geoespacial definitivo**;
- verifica la existencia física de ambos archivos;
- realiza una **lectura de prueba** para confirmar su accesibilidad e integridad estructural.

La ejecución se detiene automáticamente si alguna de las fuentes principales no está disponible, evitando iniciar la construcción de la aplicación sobre una configuración incompleta.

> **Objetivo:** establecer y validar las rutas de datos que constituirán la configuración básica de la aplicación Streamlit.

In [21]:
# ============================================================
# 06.1 · BREATHE BARCELONA · CONFIGURACIÓN Y COMPROBACIÓN
# ============================================================

from pathlib import Path
import pandas as pd
import geopandas as gpd

BASE_DIR = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard"
)

DASHBOARD_DIR = BASE_DIR

APP_PATH = (
    DASHBOARD_DIR / "app.py"
)

TEMPORAL_PATH = (
    BASE_DIR
    / "01_Datos"
    / "dashboard_temporal.parquet"
)

SPATIAL_PATH = (
    BASE_DIR
    / "01_Datos"
    / "dashboard_spatial.gpkg"
)


print("=" * 90)
print("06.1 · COMPROBACIÓN")
print("=" * 90)

print(
    "Temporal:",
    TEMPORAL_PATH.exists()
)

print(
    "Spatial:",
    SPATIAL_PATH.exists()
)


if not TEMPORAL_PATH.exists():
    raise FileNotFoundError(
        TEMPORAL_PATH
    )

if not SPATIAL_PATH.exists():
    raise FileNotFoundError(
        SPATIAL_PATH
    )


df_test = pd.read_parquet(
    TEMPORAL_PATH
)

gdf_test = gpd.read_file(
    SPATIAL_PATH
)


print()
print(
    "Temporal:",
    df_test.shape
)

print(
    "Spatial:",
    gdf_test.shape
)

print()
print(
    "✅ 06.1 COMPLETADA"
)

print("=" * 90)

06.1 · COMPROBACIÓN
Temporal: True
Spatial: True

Temporal: (14022, 25)
Spatial: (1068, 26)

✅ 06.1 COMPLETADA


### Resultados

La comprobación confirma que las dos fuentes principales de *Breathe Barcelona* están disponibles y pueden cargarse correctamente desde la configuración establecida.

Los datasets recuperados presentan las dimensiones esperadas:

- **dataset temporal:** **14.022 registros × 25 variables**;
- **dataset espacial:** **1.068 secciones × 26 variables**.

La lectura satisfactoria de ambos archivos confirma que la capa de aplicación puede acceder tanto a la información **temporal** como a la información **geoespacial** preparada previamente.

> **Resultado:** la configuración básica queda validada y las fuentes principales están listas para ser conectadas con `app.py`.

### 6.2 · Consolidación y construcción de la app base definitiva

Antes de continuar con el desarrollo visual y funcional de *Breathe Barcelona*, se establece un **punto de control maestro** y se genera una **versión base limpia y validada de la aplicación**.

La etapa se divide en dos operaciones complementarias.

En primer lugar, se preserva el último estado completamente funcional de `app.py`:

- se verifica la existencia del archivo;
- se comprueba la **sintaxis Python**;
- se confirma la presencia de las cinco vistas principales: **Overview, Temporal, Spatial, Model Insights y Predictor**;
- se valida la conservación de los componentes esenciales de **Model Insights**, incluyendo KPI, SHAP y LOSO;
- se genera una copia exacta en `app_FINAL_FUNCIONAL.py`;
- se comprueba que el backup es idéntico al archivo original y mantiene una sintaxis válida.

A continuación, se construye una **app base definitiva**, simplificada y preparada para el desarrollo incremental de la interfaz. Esta versión establece:

- la configuración general de **Streamlit** en formato panorámico;
- la lectura optimizada y cacheada de los datasets temporal y espacial;
- la gestión del estado de navegación mediante `st.session_state`;
- funciones reutilizables para navegación y representación de KPI;
- una identidad visual propia mediante **CSS personalizado**;
- una cabecera estructurada con marca, icono y navegación;
- las cinco vistas principales como estructura funcional de la aplicación;
- marcadores internos que facilitan la incorporación posterior de contenido en cada módulo.

Antes de escribir el nuevo `app.py`, el código completo se compila para garantizar que la versión generada sea sintácticamente válida.

> **Objetivo:** preservar una versión funcional de referencia y, al mismo tiempo, establecer una **arquitectura base estable, modular y visualmente coherente** sobre la que desarrollar las distintas vistas de *Breathe Barcelona*.

In [22]:
# ============================================================
# 06.2 PREVIA · BACKUP MAESTRO DEL DASHBOARD FUNCIONAL
# ============================================================

from pathlib import Path

APP_PATH = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard/app.py"
)

FINAL_BACKUP_PATH = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard/"
    "app_FINAL_FUNCIONAL.py"
)


print("=" * 90)
print("🌬️ BREATHE BARCELONA · 06.2 PREVIA")
print("=" * 90)


# ============================================================
# 1. COMPROBAR APP ACTUAL
# ============================================================

if not APP_PATH.exists():

    raise FileNotFoundError(
        f"❌ No existe el app.py actual:\n"
        f"{APP_PATH}"
    )


app_code = APP_PATH.read_text(
    encoding="utf-8"
)

print("✅ app.py localizado")


# ============================================================
# 2. VERIFICAR SINTAXIS
# ============================================================

compile(
    app_code,
    str(APP_PATH),
    "exec"
)

print("✅ Sintaxis Python correcta")


# ============================================================
# 3. COMPROBAR LAS 5 VISTAS
# ============================================================

views = [
    "Overview",
    "Temporal",
    "Spatial",
    "Model Insights",
    "Predictor",
]


for view in views:

    marker = f'selected == "{view}"'

    if marker not in app_code:

        raise ValueError(
            f"❌ No se ha localizado "
            f"la vista: {view}"
        )

    print(
        f"✅ {view} localizada"
    )


# ============================================================
# 4. COMPROBAR MODEL INSIGHTS
# ============================================================

model_checks = {
    "KPI": "mi-kpi",
    "SHAP": "SHAP",
    "LOSO": "LOSO",
}


print()
print("COMPROBACIÓN MODEL INSIGHTS:")


for nombre, texto in model_checks.items():

    if texto not in app_code:

        raise ValueError(
            f"❌ Falta {nombre} "
            f"en Model Insights"
        )

    print(
        f"✅ {nombre}"
    )


# ============================================================
# 5. GUARDAR COPIA MAESTRA
# ============================================================

FINAL_BACKUP_PATH.write_text(
    app_code,
    encoding="utf-8"
)

print()
print(
    "✅ Copia escrita:",
    FINAL_BACKUP_PATH.name
)


# ============================================================
# 6. VERIFICAR COPIA
# ============================================================

backup_code = FINAL_BACKUP_PATH.read_text(
    encoding="utf-8"
)


if backup_code != app_code:

    raise ValueError(
        "❌ La copia no coincide "
        "exactamente con app.py"
    )


compile(
    backup_code,
    str(FINAL_BACKUP_PATH),
    "exec"
)


# ============================================================
# 7. RESULTADO FINAL
# ============================================================

print()
print("=" * 90)
print("✅ 06.2 PREVIA COMPLETADA")
print("=" * 90)

print(
    "📄 Backup maestro:",
    FINAL_BACKUP_PATH.name
)

print(
    "✅ Copia idéntica al app.py actual"
)

print(
    "✅ Las 5 vistas quedan congeladas"
)

print(
    "✅ Model Insights preservado"
)

print(
    "✅ Predictor preservado dentro del app.py"
)

print(
    "✅ Ya podemos ejecutar 06.2 con seguridad"
)

print("=" * 90)

🌬️ BREATHE BARCELONA · 06.2 PREVIA
✅ app.py localizado
✅ Sintaxis Python correcta
✅ Overview localizada
✅ Temporal localizada
✅ Spatial localizada
✅ Model Insights localizada
✅ Predictor localizada

COMPROBACIÓN MODEL INSIGHTS:
✅ KPI
✅ SHAP
✅ LOSO

✅ Copia escrita: app_FINAL_FUNCIONAL.py

✅ 06.2 PREVIA COMPLETADA
📄 Backup maestro: app_FINAL_FUNCIONAL.py
✅ Copia idéntica al app.py actual
✅ Las 5 vistas quedan congeladas
✅ Model Insights preservado
✅ Predictor preservado dentro del app.py
✅ Ya podemos ejecutar 06.2 con seguridad


In [23]:
# ============================================================
# 06.2 · BREATHE BARCELONA · APP BASE DEFINITIVA
# ============================================================

from pathlib import Path

APP_PATH = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard/app.py"
)

APP_CODE = r'''
import streamlit as st
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
from pathlib import Path


# ============================================================
# CONFIGURACIÓN
# ============================================================

st.set_page_config(
    page_title="Breathe Barcelona",
    page_icon="🌬️",
    layout="wide",
    initial_sidebar_state="collapsed"
)


BASE_DIR = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard"
)

TEMPORAL_PATH = (
    BASE_DIR
    / "01_Datos"
    / "dashboard_temporal.parquet"
)

SPATIAL_PATH = (
    BASE_DIR
    / "01_Datos"
    / "dashboard_spatial.gpkg"
)


# ============================================================
# DATOS
# ============================================================

@st.cache_data
def load_temporal():
    return pd.read_parquet(TEMPORAL_PATH)


@st.cache_data
def load_spatial():
    return gpd.read_file(SPATIAL_PATH)


df_temporal = load_temporal()
gdf_spatial = load_spatial()


# ============================================================
# FUNCIONES
# ============================================================

def set_view(view):
    st.session_state["selected_view"] = view


def cycle_option(key, options, step=1):

    current = st.session_state.get(
        key,
        options[0]
    )

    try:
        idx = options.index(current)

    except ValueError:
        idx = 0

    st.session_state[key] = (
        options[
            (idx + step) % len(options)
        ]
    )


def render_kpi(
    title,
    value,
    unit="",
    note="",
    highlight=False
):

    color = (
        "#49D8D5"
        if highlight
        else "#F2F8F8"
    )

    st.markdown(
        f"""
        <div class="bb-kpi-card">
            <div class="bb-kpi-title">
                {title}
            </div>

            <div
                class="bb-kpi-value"
                style="color:{color};"
            >
                {value}

                <span class="bb-kpi-unit">
                    {unit}
                </span>
            </div>

            <div class="bb-kpi-note">
                {note}
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )


# ============================================================
# CSS
# ============================================================

st.markdown(
    """
    <style>

    html,
    body,
    [data-testid="stAppViewContainer"],
    [data-testid="stMain"] {
        background: #061820;
    }


    .stApp {

        background:
            linear-gradient(
                180deg,
                #061820 0%,
                #071C25 100%
            );

        color: #E9F4F5;
    }


    .block-container {

        max-width: 1600px;

        padding-top: 1.2rem;

        padding-bottom: 2rem;

        padding-left: 1rem;

        padding-right: 1rem;
    }


    header[data-testid="stHeader"] {
        background: transparent;
    }


    #MainMenu,
    footer {
        visibility: hidden;
    }


    /* ========================================================
       MARCA
       ======================================================== */

    .bb-brand {

        font-size: 24px;

        font-weight: 800;

        letter-spacing: 0.04em;

        color: #F4FAFA;

        line-height: 1;

        white-space: nowrap;
    }


    .bb-brand span {
        color: #20CED0;
    }


    .bb-tagline {

        margin-top: 5px;

        color: #7897A1;

        font-size: 9px;

        letter-spacing: 0.12em;

        text-transform: uppercase;

        white-space: nowrap;
    }


    /* ========================================================
       ICONO
       ======================================================== */

    .bb-face {

        font-size: 62px;

        line-height: 0.95;

        text-align: center;

        margin-top: 1px;

        white-space: nowrap;
    }


    /* ========================================================
       TÍTULO
       ======================================================== */

    h1 {

        font-size: 28px !important;

        line-height: 1.15 !important;

        margin-top: 0 !important;

        margin-bottom: 0.35rem !important;

        font-weight: 700 !important;
    }


    div[data-testid="stCaptionContainer"] {

        font-size: 11px;

        color: #7897A1;
    }


    /* ========================================================
       KPI
       ======================================================== */

    .bb-kpi-card {

        background: #092631;

        border:
            1px solid
            rgba(63,116,128,0.30);

        border-radius: 8px;

        padding: 14px 15px;

        min-height: 92px;
    }


    .bb-kpi-title {

        color: #7C9BA4;

        text-transform: uppercase;

        letter-spacing: 0.08em;

        font-size: 8px;
    }


    .bb-kpi-value {

        margin-top: 9px;

        font-size: 24px;

        font-weight: 700;

        line-height: 1;
    }


    .bb-kpi-unit {

        font-size: 9px;

        font-weight: 400;

        color: #91AAB1;
    }


    .bb-kpi-note {

        margin-top: 9px;

        font-size: 8px;

        color: #68858E;
    }


    /* ========================================================
       BOTONES
       ======================================================== */

    div[data-testid="stButton"] button {

        min-height: 37px;

        background: #092631;

        color: #A8C4C9;

        border:
            1px solid
            rgba(65,118,130,0.30);

        border-radius: 7px;
    }


    div[data-testid="stButton"] button:hover {

        color: #48D8D5;

        border-color: #48D8D5;
    }

    </style>
    """,
    unsafe_allow_html=True
)


# ============================================================
# CABECERA
# ============================================================

icon_col, main_col, nav_col = st.columns(
    [0.065, 0.255, 0.680],
    gap="small"
)


# ============================================================
# ICONO
# ============================================================

with icon_col:

    st.markdown(
        """
        <div class="bb-face">
            🌬️
        </div>
        """,
        unsafe_allow_html=True
    )


# ============================================================
# BREATHE BARCELONA
# ============================================================

with main_col:

    st.markdown(
        """
        <div class="bb-brand">
            BREATHE <span>BARCELONA</span>
        </div>

        <div class="bb-tagline">
            Urban Air Quality Intelligence
        </div>
        """,
        unsafe_allow_html=True
    )


# ============================================================
# NAVEGACIÓN
# ============================================================

views = [
    "Overview",
    "Temporal",
    "Spatial",
    "Model Insights",
    "Predictor"
]


if "selected_view" not in st.session_state:
    st.session_state["selected_view"] = "Overview"


with nav_col:

    nav_cols = st.columns(
        len(views),
        gap="small"
    )

    for col, view in zip(
        nav_cols,
        views
    ):

        with col:

            st.button(
                view,
                key=f"nav_{view}",
                use_container_width=True,
                on_click=set_view,
                args=(view,)
            )


# ============================================================
# SEGUNDA FILA
# ============================================================

st.markdown(
    "<div style='height:8px'></div>",
    unsafe_allow_html=True
)


title_icon_col, title_main_col, title_right_col = st.columns(
    [0.065, 0.255, 0.680],
    gap="small"
)


# ============================================================
# VISTAS
# ============================================================

selected = st.session_state[
    "selected_view"
]


with title_main_col:

    if selected == "Overview":

        # OVERVIEW_START

        st.title(
            "Barcelona Air Quality"
        )

        st.caption(
            "Overview listo para construir."
        )

        # OVERVIEW_END


    elif selected == "Temporal":

        st.title(
            "Temporal Analysis"
        )

        st.caption(
            "Vista temporal pendiente."
        )


    elif selected == "Spatial":

        st.title(
            "Spatial Analysis"
        )

        st.caption(
            "Vista espacial pendiente."
        )


    elif selected == "Model Insights":

        st.title(
            "Model Insights"
        )

        st.caption(
            "Vista de modelos pendiente."
        )


    elif selected == "Predictor":

        st.title(
            "Predictor"
        )

        st.caption(
            "Predictor pendiente."
        )
'''


# ============================================================
# VERIFICAR Y GUARDAR
# ============================================================

compile(
    APP_CODE,
    str(APP_PATH),
    "exec"
)

APP_PATH.write_text(
    APP_CODE,
    encoding="utf-8"
)


print("=" * 90)
print("✅ 06.2 COMPLETADA")
print("✅ Icono ajustado a 62 px")
print("✅ Títulos alineados")
print("✅ Barcelona Air Quality acercado a la cabecera")
print("✅ Navegación conservada")
print("✅ Sintaxis verificada")
print("=" * 90)

✅ 06.2 COMPLETADA
✅ Icono ajustado a 62 px
✅ Títulos alineados
✅ Barcelona Air Quality acercado a la cabecera
✅ Navegación conservada
✅ Sintaxis verificada


### Resultado

La fase 06.2 deja establecido un **doble mecanismo de seguridad y construcción** para el desarrollo de *Breathe Barcelona*.

Por una parte, el estado funcional previo queda preservado en:

`app_FINAL_FUNCIONAL.py`

La copia se valida como idéntica al `app.py` de origen y conserva correctamente las cinco vistas de la aplicación, así como los componentes esenciales de **Model Insights** y el módulo **Predictor**.

Por otra parte, se genera una nueva **app base definitiva** con una estructura simplificada y preparada para el desarrollo incremental. La versión resultante incorpora:

- configuración panorámica de Streamlit;
- carga cacheada de los datasets temporal y espacial;
- navegación persistente entre las cinco vistas;
- estructura modular basada en `st.session_state`;
- funciones reutilizables para componentes interactivos;
- sistema visual común mediante CSS personalizado;
- cabecera, marca e identidad gráfica de *Breathe Barcelona*;
- alineación y jerarquía tipográfica de los títulos;
- estructura preparada para desarrollar de forma independiente cada módulo.

La compilación previa confirma que el código generado presenta una **sintaxis Python válida** antes de ser escrito en `app.py`.

> **Resultado:** queda disponible una **versión maestra de restauración** y una **base de aplicación limpia, validada y estructurada**, sobre la que pueden incorporarse progresivamente las vistas Overview, Temporal, Spatial, Model Insights y Predictor sin comprometer el estado funcional previamente alcanzado.

### 6.3 · Diagnóstico de datos para Overview

Antes de construir la vista **Overview**, se realiza una inspección estructurada de las dos fuentes principales de información de *Breathe Barcelona* con el fin de identificar qué variables pueden utilizarse directamente en los indicadores, filtros y visualizaciones iniciales.

El diagnóstico analiza:

* dimensiones y estructura del **dataset temporal**;
* dimensiones, sistema de referencia y variables del **dataset espacial**;
* rango temporal disponible;
* variables categóricas susceptibles de emplearse como filtros;
* variables numéricas disponibles para indicadores y gráficos;
* detección automática de variables relacionadas con los principales contaminantes.

Esta revisión permite comprobar que la información preparada en las etapas anteriores contiene los elementos necesarios para construir una vista Overview coherente, sintética y conectada con el resto de módulos de la aplicación.

> **Objetivo:** identificar las variables, categorías, cobertura temporal y fuentes disponibles para definir los KPI, filtros y visualizaciones principales de la vista Overview.


In [24]:
# ============================================================
# 06.3 · DIAGNÓSTICO DE DATOS PARA OVERVIEW
# ============================================================

import pandas as pd
import geopandas as gpd
from pathlib import Path


# ============================================================
# RUTAS
# ============================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard"
)

TEMPORAL_PATH = (
    BASE_DIR
    / "01_Datos"
    / "dashboard_temporal.parquet"
)

SPATIAL_PATH = (
    BASE_DIR
    / "01_Datos"
    / "dashboard_spatial.gpkg"
)


# ============================================================
# CARGA
# ============================================================

df = pd.read_parquet(
    TEMPORAL_PATH
)

gdf = gpd.read_file(
    SPATIAL_PATH
)


# ============================================================
# TEMPORAL
# ============================================================

print("=" * 90)
print("06.3 · DATASET TEMPORAL")
print("=" * 90)

print(
    f"\nDimensiones: "
    f"{df.shape[0]:,} filas × "
    f"{df.shape[1]} columnas"
)

print("\nCOLUMNAS:\n")

for i, col in enumerate(
    df.columns,
    start=1
):

    print(
        f"{i:02d}. "
        f"{col:<35} "
        f"{str(df[col].dtype)}"
    )


# ============================================================
# SPATIAL
# ============================================================

print("\n")
print("=" * 90)
print("06.3 · DATASET SPATIAL")
print("=" * 90)

print(
    f"\nDimensiones: "
    f"{gdf.shape[0]:,} filas × "
    f"{gdf.shape[1]} columnas"
)

print(
    f"\nCRS: {gdf.crs}"
)

print("\nCOLUMNAS:\n")

for i, col in enumerate(
    gdf.columns,
    start=1
):

    print(
        f"{i:02d}. "
        f"{col:<35} "
        f"{str(gdf[col].dtype)}"
    )


# ============================================================
# FECHAS
# ============================================================

print("\n")
print("=" * 90)
print("RANGO TEMPORAL")
print("=" * 90)

fecha_encontrada = False

for col in df.columns:

    if pd.api.types.is_datetime64_any_dtype(
        df[col]
    ):

        fecha_encontrada = True

        print(
            f"\n{col}:"
        )

        print(
            f"  Inicio: {df[col].min()}"
        )

        print(
            f"  Fin:    {df[col].max()}"
        )


if not fecha_encontrada:

    print(
        "\n⚠️ No hay columnas datetime detectadas."
    )


# ============================================================
# VARIABLES CATEGÓRICAS / FILTROS
# ============================================================

print("\n")
print("=" * 90)
print("VARIABLES CATEGÓRICAS / FILTROS")
print("=" * 90)

for col in df.columns:

    try:

        n = df[col].nunique(
            dropna=True
        )

        if 1 < n <= 15:

            valores = (
                df[col]
                .dropna()
                .unique()
                .tolist()
            )

            print(
                f"\n{col} "
                f"({n} valores)"
            )

            print(
                valores
            )

    except Exception:

        pass


# ============================================================
# VARIABLES NUMÉRICAS
# ============================================================

print("\n")
print("=" * 90)
print("VARIABLES NUMÉRICAS")
print("=" * 90)

numericas = (
    df.select_dtypes(
        include="number"
    )
    .columns
    .tolist()
)

for col in numericas:

    print(
        f"• {col}"
    )


# ============================================================
# POSIBLES CONTAMINANTES
# ============================================================

print("\n")
print("=" * 90)
print("POSIBLES CONTAMINANTES")
print("=" * 90)

keywords = [
    "NO2",
    "NO₂",
    "PM10",
    "PM2",
    "PM25",
    "PM2.5",
    "O3",
    "O₃"
]

encontrados = []

for col in df.columns:

    col_upper = col.upper()

    if any(
        key.upper() in col_upper
        for key in keywords
    ):

        encontrados.append(
            col
        )


if encontrados:

    for col in encontrados:

        print(
            f"✅ {col}"
        )

else:

    print(
        "⚠️ No se detectaron automáticamente."
    )


# ============================================================
# FIN
# ============================================================

print("\n")
print("=" * 90)
print("✅ 06.3 COMPLETADA")
print("=" * 90)

06.3 · DATASET TEMPORAL

Dimensiones: 14,022 filas × 25 columnas

COLUMNAS:

01. fecha                               datetime64[ns]
02. estacion_geo                        object
03. NO2_ug_m3                           float64
04. O3_ug_m3                            float64
05. PM10_ug_m3                          float64
06. PM25_ug_m3                          float64
07. lat                                 float64
08. lon                                 float64
09. TM                                  float64
10. HRM                                 float64
11. PPT                                 float64
12. VVM10                               float64
13. trafico_idw_500m                    float64
14. anio                                int64
15. LAeq_idw_1500m                      float64
16. Vuelos_total                        float64
17. Puerto_movimientos                  float64
18. mes                                 int64
19. periodo_covid                       object
20. clase_

### Resultados

El diagnóstico confirma que la vista **Overview** dispone de una base temporal y espacial suficientemente estructurada para construir los principales indicadores y componentes gráficos de *Breathe Barcelona*.

El dataset temporal contiene **14.022 registros y 25 variables**, con cobertura completa entre **2019 y 2024** y datos correspondientes a **8 estaciones de calidad del aire**. Incluye los cuatro contaminantes principales del estudio —**NO₂, O₃, PM10 y PM2.5**— junto con variables meteorológicas, de tráfico, ruido, actividad aérea y portuaria, contexto temporal y variables derivadas de NO₂.

También se identifican filtros directamente utilizables en la aplicación:

* **8 estaciones**;
* **6 años**;
* **12 meses**;
* **3 periodos**: Pre-COVID, COVID y Post-COVID;
* **3 clases de calidad de NO₂**: Buena, Media y Mala;
* **7 días de la semana**.

El dataset espacial contiene **1.068 secciones censales y 26 variables** en **EPSG:4326**, incluyendo concentraciones observadas y predichas de NO₂, errores del modelo XGBoost, variables socioeconómicas y territoriales, información de conjunto de modelización y resultados de autocorrelación espacial **LISA**.

> **Resultado:** quedan identificadas y validadas las variables necesarias para construir una Overview que combine **estado de la calidad del aire, evolución temporal, contexto territorial y acceso a los principales módulos analíticos** del dashboard.


### 6.4 · Construcción de la vista Overview

Una vez validada la disponibilidad y estructura de los datos, se construye la vista **Overview**, concebida como el punto de entrada a *Breathe Barcelona* y como una síntesis visual de los principales resultados del estudio.

La vista combina información **temporal y espacial** mediante una composición panorámica organizada en tres áreas: controles e indicadores, representación cartográfica y análisis gráfico.

Se incorporan los siguientes componentes:

* selector de periodo para explorar el conjunto **2019–2024** o cada año individualmente;
* cuatro **KPI dinámicos**: concentración media de NO₂, percentil 95, porcentaje de observaciones clasificadas como calidad mala y estación con mayor concentración media;
* mapa coroplético de la concentración media de **NO₂ por sección censal**, con información contextual mediante *hover*;
* evolución mensual conjunta de **NO₂, O₃, PM10 y PM2.5**;
* boxplots para comparar la distribución de los cuatro contaminantes;
* distribución porcentual de las categorías **Buena, Media y Mala** de NO₂.

La construcción se realiza mediante sustitución controlada del módulo Overview dentro de `app.py`, preservando las vistas **Temporal, Spatial, Model Insights y Predictor**. Antes de modificar el archivo se comprueba la estructura del código y, posteriormente, se valida nuevamente su sintaxis y la presencia de todos los componentes requeridos.

> **Objetivo:** construir una vista inicial capaz de resumir el estado temporal y territorial de la calidad del aire de Barcelona y facilitar el acceso visual a los principales patrones desarrollados posteriormente en los módulos especializados.


In [25]:
# ============================================================
# 🌬️ BREATHE BARCELONA
# 06.4 · OVERVIEW DEFINITIVO · FINAL
# ============================================================
#
# ✓ Reproducible desde 06.2
# ✓ Reejecutable sobre una 06.4 anterior
# ✓ Limpia residuo aislado "el" si existiera
# ✓ Título + columna izquierda alineados
# ✓ Separación título / Periodo
# ✓ Tipografía secundaria reforzada
# ✓ Leyenda temporal elevada
# ✓ Selector de periodo
# ✓ 4 KPI
# ✓ Mapa NO₂ Barcelona
# ✓ ESRI World Dark Gray
# ✓ Hover: sección censal + NO₂ + calidad
# ✓ Evolución mensual
# ✓ Boxplots
# ✓ Clasificación NO₂
# ✓ Conserva Temporal / Spatial / Model Insights / Predictor
# ✓ Valida sintaxis ANTES de guardar
# ============================================================

from pathlib import Path


APP_PATH = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard/app.py"
)


# ============================================================
# 0 · CARGAR APP
# ============================================================

if not APP_PATH.exists():

    raise FileNotFoundError(
        f"❌ No existe app.py:\n{APP_PATH}"
    )


app_code = APP_PATH.read_text(
    encoding="utf-8"
)


# ============================================================
# 0.1 · LIMPIEZA DEFENSIVA
# ============================================================

original_lines = app_code.splitlines()

clean_lines = [
    line
    for line in original_lines
    if line.strip() != "el"
]

removed_el = (
    len(original_lines)
    - len(clean_lines)
)

if removed_el:

    print(
        f"🧹 Eliminado residuo aislado 'el': "
        f"{removed_el} línea(s)"
    )

app_code = (
    "\n".join(clean_lines)
    + "\n"
)


# ============================================================
# 1 · LOCALIZAR VISTAS
# ============================================================

wrapper_start = app_code.find(
    "with title_main_col:"
)

overview_direct = app_code.find(
    'if selected == "Overview":'
)


# ------------------------------------------------------------
# CASO A · app.py recién generado por 06.2
# ------------------------------------------------------------

if wrapper_start != -1:

    overview_start = app_code.find(
        '    if selected == "Overview":',
        wrapper_start
    )

    temporal_start = app_code.find(
        '    elif selected == "Temporal":',
        overview_start
    )

    if overview_start == -1:
        raise ValueError(
            "❌ No se encontró Overview."
        )

    if temporal_start == -1:
        raise ValueError(
            "❌ No se encontró Temporal."
        )

    prefix = app_code[:wrapper_start]

    tail = app_code[temporal_start:]

    tail_lines = tail.splitlines()

    views_tail = "\n".join(
        line[4:]
        if line.startswith("    ")
        else line
        for line in tail_lines
    )


# ------------------------------------------------------------
# CASO B · reejecución sobre 06.4 existente
# ------------------------------------------------------------

else:

    if overview_direct == -1:
        raise ValueError(
            "❌ No se encontró Overview."
        )

    temporal_start = app_code.find(
        'elif selected == "Temporal":',
        overview_direct
    )

    if temporal_start == -1:
        raise ValueError(
            "❌ No se encontró Temporal."
        )

    prefix = app_code[:overview_direct]

    views_tail = app_code[temporal_start:]


# ============================================================
# 2 · OVERVIEW COMPLETO
# ============================================================

NEW_OVERVIEW = r'''
if selected == "Overview":

    # ========================================================
    # CSS OVERVIEW
    # ========================================================

    st.markdown(
        """
<style>


/* ---------------------------------------------------------
   KPI
   --------------------------------------------------------- */

.bb-kpi-card {

    height: 128px;

    box-sizing: border-box;

    display: flex;

    align-items: center;

    gap: 13px;

    padding: 16px 15px;

    margin-bottom: 14px;

    background:
        linear-gradient(
            145deg,
            rgba(7,34,44,0.98),
            rgba(5,25,33,0.98)
        );

    border:
        1px solid
        rgba(
            40,
            213,
            210,
            0.34
        );

    border-radius: 8px;

    box-shadow:
        0 2px 10px
        rgba(
            0,
            0,
            0,
            0.10
        );
}


.bb-kpi-icon {

    width: 35px;

    min-width: 35px;

    height: 35px;

    display: flex;

    align-items: center;

    justify-content: center;

    font-size: 27px;

    line-height: 1;
}


.bb-kpi-cyan {
    color: #28D5D2;
}


.bb-kpi-blue {
    color: #53A7E8;
}


.bb-kpi-red {
    color: #E87575;
}


.bb-kpi-content {

    min-width: 0;

    flex: 1;

    display: flex;

    flex-direction: column;

    justify-content: center;
}


.bb-kpi-label {

    color: #E6F2F3;

    font-size: 12px;

    font-weight: 650;

    line-height: 1.15;

    margin-bottom: 6px;
}


.bb-kpi-value {

    color: #FFFFFF;

    font-size: 25px;

    font-weight: 650;

    line-height: 1.05;

    white-space: nowrap;

    margin-bottom: 8px;
}


.bb-kpi-station {
    font-size: 23px;
}


.bb-kpi-note {

    display: inline-block;

    align-self: flex-start;

    color: #A7BEC4;

    background:
        rgba(
            116,
            150,
            160,
            0.16
        );

    border-radius: 12px;

    padding: 3px 8px;

    font-size: 10.5px;

    font-weight: 500;

    line-height: 1.2;
}


/* ---------------------------------------------------------
   SELECTOR PERIODO
   --------------------------------------------------------- */

div[data-testid="stSelectbox"] > label {

    color: #F0F7F8 !important;

    font-size: 12.5px !important;

    font-weight: 650 !important;
}


div[data-testid="stSelectbox"]
div[data-baseweb="select"] > div {

    min-height: 43px !important;

    background:
        rgba(
            7,
            34,
            44,
            0.98
        ) !important;

    border:
        1px solid
        rgba(
            40,
            213,
            210,
            0.62
        ) !important;

    border-radius:
        8px !important;

    box-shadow:
        0 0 0 1px
        rgba(
            40,
            213,
            210,
            0.05
        );
}


</style>
        """,
        unsafe_allow_html=True
    )


    # ========================================================
    # TÍTULO
    # ========================================================

    title_left, title_main, title_right = st.columns(
        [
            0.045,
            0.44,
            0.515
        ],
        gap="small"
    )


    with title_main:

        st.markdown(
            """
<div style="
color:#F5FAFA;
font-size:24px;
font-weight:700;
line-height:1.10;
margin-top:-1px;
margin-bottom:4px;
">
Barcelona Air Quality
</div>

<div style="
color:#A5BCC2;
font-size:12px;
font-weight:450;
line-height:1.25;
margin-bottom:13px;
">
Urban air pollution · 2019–2024
</div>
            """,
            unsafe_allow_html=True
        )


    # ========================================================
    # DATOS
    # ========================================================

    df_overview = (
        df_temporal.copy()
    )


    df_overview[
        "fecha"
    ] = pd.to_datetime(
        df_overview[
            "fecha"
        ]
    )


    # ========================================================
    # LAYOUT
    # ========================================================

    (
        left_spacer,
        control_col,
        map_col,
        analysis_col

    ) = st.columns(

        [
            0.040,
            0.170,
            0.395,
            0.395
        ],

        gap="medium"
    )


    # ========================================================
    # IZQUIERDA
    # ========================================================

    with control_col:


        periodo_sel = st.selectbox(

            "Periodo",

            [
                "Todos · 2019–2024",
                "2019",
                "2020",
                "2021",
                "2022",
                "2023",
                "2024"
            ],

            index=0,

            key=(
                "overview_periodo_final"
            )
        )


        # ----------------------------------------------------
        # FILTRO TEMPORAL
        # ----------------------------------------------------

        if (
            periodo_sel
            == "Todos · 2019–2024"
        ):

            df_sel = (
                df_overview.copy()
            )

            periodo_label = (
                "2019–2024"
            )


        else:

            year = int(
                periodo_sel
            )

            df_sel = (
                df_overview[
                    df_overview[
                        "anio"
                    ]
                    == year
                ]
                .copy()
            )

            periodo_label = (
                periodo_sel
            )


        # ====================================================
        # KPI
        # ====================================================

        no2 = (
            df_sel[
                "NO2_ug_m3"
            ]
            .dropna()
        )


        media_no2 = (

            no2.mean()

            if len(no2)

            else float(
                "nan"
            )
        )


        p95_no2 = (

            no2.quantile(
                0.95
            )

            if len(no2)

            else float(
                "nan"
            )
        )


        clases = (
            df_sel[
                "clase_NO2_3"
            ]
            .dropna()
        )


        pct_mala = (

            clases
            .eq(
                "Mala"
            )
            .mean()
            * 100

            if len(clases)

            else float(
                "nan"
            )
        )


        estaciones = (

            df_sel

            .groupby(
                "estacion_geo"
            )[
                "NO2_ug_m3"
            ]

            .mean()

            .dropna()
        )


        if len(
            estaciones
        ):

            estacion_max = (
                estaciones.idxmax()
            )

            estacion_max_value = (
                estaciones.max()
            )


        else:

            estacion_max = "—"

            estacion_max_value = (
                float(
                    "nan"
                )
            )


        # ----------------------------------------------------
        # FORMATOS
        # ----------------------------------------------------

        valor_media = (

            f"{media_no2:.1f} µg/m³"

            if pd.notna(
                media_no2
            )

            else "—"
        )


        valor_p95 = (

            f"{p95_no2:.1f} µg/m³"

            if pd.notna(
                p95_no2
            )

            else "—"
        )


        valor_mala = (

            f"{pct_mala:.1f} %"

            if pd.notna(
                pct_mala
            )

            else "—"
        )


        valor_estacion = (

            f"{estacion_max_value:.1f} µg/m³"

            if pd.notna(
                estacion_max_value
            )

            else "Sin datos"
        )


        # ----------------------------------------------------
        # KPI 1
        # ----------------------------------------------------

        html_kpi_1 = (

            f'<div class="bb-kpi-card">'

            f'<div '
            f'class="bb-kpi-icon '
            f'bb-kpi-cyan">'
            f'☁'
            f'</div>'

            f'<div '
            f'class="bb-kpi-content">'

            f'<div '
            f'class="bb-kpi-label">'
            f'NO₂ medio'
            f'</div>'

            f'<div '
            f'class="bb-kpi-value">'
            f'{valor_media}'
            f'</div>'

            f'<div '
            f'class="bb-kpi-note">'
            f'↑ {periodo_label}'
            f'</div>'

            f'</div>'

            f'</div>'
        )


        st.markdown(
            html_kpi_1,
            unsafe_allow_html=True
        )


        # ----------------------------------------------------
        # KPI 2
        # ----------------------------------------------------

        html_kpi_2 = (

            f'<div class="bb-kpi-card">'

            f'<div '
            f'class="bb-kpi-icon '
            f'bb-kpi-blue">'
            f'◔'
            f'</div>'

            f'<div '
            f'class="bb-kpi-content">'

            f'<div '
            f'class="bb-kpi-label">'
            f'Percentil 95 · NO₂'
            f'</div>'

            f'<div '
            f'class="bb-kpi-value">'
            f'{valor_p95}'
            f'</div>'

            f'<div '
            f'class="bb-kpi-note">'
            f'↑ Concentraciones altas'
            f'</div>'

            f'</div>'

            f'</div>'
        )


        st.markdown(
            html_kpi_2,
            unsafe_allow_html=True
        )


        # ----------------------------------------------------
        # KPI 3
        # ----------------------------------------------------

        html_kpi_3 = (

            f'<div class="bb-kpi-card">'

            f'<div '
            f'class="bb-kpi-icon '
            f'bb-kpi-red">'
            f'☹'
            f'</div>'

            f'<div '
            f'class="bb-kpi-content">'

            f'<div '
            f'class="bb-kpi-label">'
            f'Calidad mala · NO₂'
            f'</div>'

            f'<div '
            f'class="bb-kpi-value">'
            f'{valor_mala}'
            f'</div>'

            f'<div '
            f'class="bb-kpi-note">'
            f'↑ Observaciones'
            f'</div>'

            f'</div>'

            f'</div>'
        )


        st.markdown(
            html_kpi_3,
            unsafe_allow_html=True
        )


        # ----------------------------------------------------
        # KPI 4
        # ----------------------------------------------------

        html_kpi_4 = (

            f'<div class="bb-kpi-card">'

            f'<div '
            f'class="bb-kpi-icon '
            f'bb-kpi-cyan">'
            f'⌖'
            f'</div>'

            f'<div '
            f'class="bb-kpi-content">'

            f'<div '
            f'class="bb-kpi-label">'
            f'Estación más expuesta'
            f'</div>'

            f'<div '
            f'class="bb-kpi-value '
            f'bb-kpi-station">'
            f'{estacion_max}'
            f'</div>'

            f'<div '
            f'class="bb-kpi-note">'
            f'↑ {valor_estacion}'
            f'</div>'

            f'</div>'

            f'</div>'
        )


        st.markdown(
            html_kpi_4,
            unsafe_allow_html=True
        )


    # ========================================================
    # PALETA
    # ========================================================

    pollutants = {

        "NO2_ug_m3": {

            "label": "NO₂",

            "color": "#28D5D2"
        },


        "O3_ug_m3": {

            "label": "O₃",

            "color": "#53A7E8"
        },


        "PM10_ug_m3": {

            "label": "PM10",

            "color": "#E7B969"
        },


        "PM25_ug_m3": {

            "label": "PM2.5",

            "color": "#E87575"
        }
    }


    # ========================================================
    # MAPA
    # ========================================================

    with map_col:


        gdf_map = (

            gdf_spatial[
                [
                    "NO2_MEDIO_SECCION",
                    "CLAVE_SECCION",
                    "NO2_CLASE",
                    "geometry"
                ]
            ]

            .dropna(
                subset=[
                    "NO2_MEDIO_SECCION",
                    "geometry"
                ]
            )

            .copy()
        )


        if (

            gdf_map.crs
            is not None

            and

            gdf_map.crs.to_epsg()
            != 4326
        ):

            gdf_map = (
                gdf_map.to_crs(
                    epsg=4326
                )
            )


        gdf_map = (
            gdf_map
            .reset_index(
                drop=True
            )
        )


        gdf_map[
            "_map_id"
        ] = (

            gdf_map.index
            .astype(
                str
            )
        )


        geojson_map = (
            gdf_map
            .__geo_interface__
        )


        fig_map = (
            go.Figure()
        )


        fig_map.add_trace(

            go.Choroplethmapbox(

                geojson=(
                    geojson_map
                ),

                locations=(
                    gdf_map[
                        "_map_id"
                    ]
                ),

                z=(
                    gdf_map[
                        "NO2_MEDIO_SECCION"
                    ]
                ),

                featureidkey=(
                    "properties._map_id"
                ),

                colorscale=[

                    [
                        0.00,
                        "#10303A"
                    ],

                    [
                        0.22,
                        "#176B78"
                    ],

                    [
                        0.48,
                        "#28D5D2"
                    ],

                    [
                        0.73,
                        "#E7B969"
                    ],

                    [
                        1.00,
                        "#E87575"
                    ]
                ],

                marker_opacity=(
                    0.76
                ),

                marker_line_width=(
                    0.15
                ),

                marker_line_color=(
                    "rgba("
                    "220,"
                    "235,"
                    "238,"
                    "0.20"
                    ")"
                ),

                colorbar=dict(

                    title=dict(

                        text="NO₂",

                        font=dict(

                            size=11,

                            color="#E5F0F2"
                        )
                    ),

                    thickness=11,

                    len=0.70,

                    x=0.965,

                    y=0.50,

                    tickfont=dict(

                        size=10.5,

                        color="#D7E4E6"
                    )
                ),

                customdata=(

                    gdf_map[
                        [
                            "CLAVE_SECCION",
                            "NO2_MEDIO_SECCION",
                            "NO2_CLASE"
                        ]
                    ]
                ),

                hovertemplate=(

                    "<b>Sección censal</b>: "

                    "%{customdata[0]}"

                    "<br>"

                    "<b>NO₂ medio</b>: "

                    "%{customdata[1]:.1f} µg/m³"

                    "<br>"

                    "<b>Calidad</b>: "

                    "%{customdata[2]}"

                    "<extra></extra>"
                )
            )
        )


        fig_map.update_layout(

            mapbox=dict(

                style="white-bg",

                center=dict(

                    lat=41.39,

                    lon=2.16
                ),

                zoom=10.6,

                layers=[


                    dict(

                        below="traces",

                        sourcetype=(
                            "raster"
                        ),

                        source=[

                            (
                                "https://"
                                "server.arcgisonline.com/"
                                "ArcGIS/rest/services/"
                                "Canvas/"
                                "World_Dark_Gray_Base/"
                                "MapServer/tile/"
                                "{z}/{y}/{x}"
                            )
                        ]
                    ),


                    dict(

                        sourcetype=(
                            "raster"
                        ),

                        source=[

                            (
                                "https://"
                                "server.arcgisonline.com/"
                                "ArcGIS/rest/services/"
                                "Canvas/"
                                "World_Dark_Gray_Reference/"
                                "MapServer/tile/"
                                "{z}/{y}/{x}"
                            )
                        ]
                    )
                ]
            ),


            title=dict(

                text=(
                    "NO₂ espacial · "
                    "promedio 2019–2024"
                ),

                x=0.025,

                y=0.975,

                font=dict(

                    size=14,

                    color="#F5FAFA"
                )
            ),


            height=660,


            margin=dict(

                l=3,

                r=3,

                t=42,

                b=3
            ),


            paper_bgcolor=(
                "#071C25"
            ),


            plot_bgcolor=(
                "#071C25"
            ),


            hoverlabel=dict(

                bgcolor=(
                    "#092832"
                ),

                bordercolor=(
                    "#28D5D2"
                ),

                font=dict(

                    size=12,

                    color="#FFFFFF"
                )
            )
        )


        st.plotly_chart(

            fig_map,

            use_container_width=True,

            config={

                "displayModeBar": False,

                "displaylogo": False,

                "scrollZoom": True,

                "responsive": True,

                "modeBarButtonsToRemove": [

                    "select2d",

                    "lasso2d",

                    "toImage"
                ]
            }
        )


    # ========================================================
    # DERECHA
    # ========================================================

    with analysis_col:


        RIGHT_CHART_HEIGHT = 210


        # ====================================================
        # 1 · EVOLUCIÓN MENSUAL
        # ====================================================

        serie = (

            df_sel

            .set_index(
                "fecha"
            )

            [
                [
                    "NO2_ug_m3",
                    "O3_ug_m3",
                    "PM10_ug_m3",
                    "PM25_ug_m3"
                ]
            ]

            .resample(
                "MS"
            )

            .mean()

            .reset_index()
        )


        fig_time = (
            go.Figure()
        )


        for (
            col,
            cfg

        ) in pollutants.items():


            fig_time.add_trace(

                go.Scatter(

                    x=(
                        serie[
                            "fecha"
                        ]
                    ),

                    y=(
                        serie[
                            col
                        ]
                    ),

                    mode="lines",

                    name=(
                        cfg[
                            "label"
                        ]
                    ),

                    line=dict(

                        color=(
                            cfg[
                                "color"
                            ]
                        ),

                        width=2.2
                    ),

                    hovertemplate=(

                        f"<b>{cfg['label']}</b>"

                        "<br>"

                        "%{x|%b %Y}"

                        "<br>"

                        "%{y:.1f} µg/m³"

                        "<extra></extra>"
                    )
                )
            )


        fig_time.update_layout(

            title=dict(

                text=(
                    "Evolución mensual "
                    "de contaminantes"
                ),

                x=0.025,

                font=dict(

                    size=14,

                    color="#F5FAFA"
                )
            ),


            height=(
                RIGHT_CHART_HEIGHT
            ),


            margin=dict(

                l=48,

                r=8,

                t=42,

                b=28
            ),


            paper_bgcolor=(
                "#071C25"
            ),


            plot_bgcolor=(
                "#071C25"
            ),


            hovermode=(
                "x unified"
            ),


            # ================================================
            # LEYENDA FINAL
            # Elevada para no competir con las series.
            # ================================================

            legend=dict(

                orientation="h",

                y=1.16,

                yanchor="top",

                x=0.985,

                xanchor="right",

                font=dict(

                    size=11,

                    color="#DDE9EA"
                )
            ),


            xaxis=dict(

                showgrid=False,

                tickfont=dict(

                    size=11,

                    color="#C7D8DB"
                )
            ),


            yaxis=dict(

                title=dict(

                    text="µg/m³",

                    font=dict(

                        size=11,

                        color="#DDE9EA"
                    )
                ),

                gridcolor=(
                    "rgba("
                    "170,"
                    "195,"
                    "200,"
                    "0.12"
                    ")"
                ),

                zeroline=False,

                tickfont=dict(

                    size=11,

                    color="#C7D8DB"
                )
            )
        )


        st.plotly_chart(

            fig_time,

            use_container_width=True,

            config={

                "displayModeBar": False,

                "responsive": True
            }
        )


        # ====================================================
        # 2 · BOXPLOTS
        # ====================================================

        fig_box = (
            go.Figure()
        )


        for (
            col,
            cfg

        ) in pollutants.items():


            fig_box.add_trace(

                go.Box(

                    y=(
                        df_sel[
                            col
                        ]
                        .dropna()
                    ),

                    name=(
                        cfg[
                            "label"
                        ]
                    ),

                    marker_color=(
                        cfg[
                            "color"
                        ]
                    ),

                    line=dict(

                        color=(
                            cfg[
                                "color"
                            ]
                        ),

                        width=1.4
                    ),

                    boxpoints=False
                )
            )


        fig_box.update_layout(

            title=dict(

                text=(
                    "Distribución de "
                    "contaminantes"
                ),

                x=0.025,

                font=dict(

                    size=14,

                    color="#F5FAFA"
                )
            ),


            height=(
                RIGHT_CHART_HEIGHT
            ),


            margin=dict(

                l=48,

                r=8,

                t=40,

                b=28
            ),


            showlegend=False,


            paper_bgcolor=(
                "#071C25"
            ),


            plot_bgcolor=(
                "#071C25"
            ),


            xaxis=dict(

                showgrid=False,

                tickfont=dict(

                    size=11,

                    color="#DDE9EA"
                )
            ),


            yaxis=dict(

                title=dict(

                    text="µg/m³",

                    font=dict(

                        size=11,

                        color="#DDE9EA"
                    )
                ),

                rangemode="tozero",

                gridcolor=(
                    "rgba("
                    "170,"
                    "195,"
                    "200,"
                    "0.12"
                    ")"
                ),

                zeroline=False,

                tickfont=dict(

                    size=11,

                    color="#C7D8DB"
                )
            )
        )


        st.plotly_chart(

            fig_box,

            use_container_width=True,

            config={

                "displayModeBar": False,

                "responsive": True
            }
        )


        # ====================================================
        # 3 · CLASIFICACIÓN
        # ====================================================

        orden_clases = [

            "Buena",

            "Media",

            "Mala"
        ]


        conteos = (

            df_sel[
                "clase_NO2_3"
            ]

            .value_counts()

            .reindex(

                orden_clases,

                fill_value=0
            )
        )


        total_clases = (
            conteos.sum()
        )


        if total_clases > 0:

            porcentajes = (

                conteos

                / total_clases

                * 100
            )


        else:

            porcentajes = (
                conteos
                .astype(
                    float
                )
            )


        fig_class = (
            go.Figure()
        )


        fig_class.add_trace(

            go.Bar(

                x=(
                    porcentajes.values
                ),

                y=(
                    orden_clases
                ),

                orientation="h",

                marker=dict(

                    color=[

                        "#28D5D2",

                        "#E7B969",

                        "#E87575"
                    ]
                ),

                text=[

                    f"{x:.1f}%"

                    for x
                    in porcentajes.values
                ],

                textposition=(
                    "outside"
                ),

                textfont=dict(

                    size=11,

                    color="#F5FAFA"
                ),

                hovertemplate=(

                    "<b>%{y}</b>"

                    "<br>"

                    "%{x:.1f}%"

                    "<extra></extra>"
                )
            )
        )


        fig_class.update_layout(

            title=dict(

                text=(
                    "Clasificación de "
                    "calidad · NO₂"
                ),

                x=0.025,

                font=dict(

                    size=14,

                    color="#F5FAFA"
                )
            ),


            height=(
                RIGHT_CHART_HEIGHT
            ),


            margin=dict(

                l=58,

                r=28,

                t=40,

                b=28
            ),


            showlegend=False,


            paper_bgcolor=(
                "#071C25"
            ),


            plot_bgcolor=(
                "#071C25"
            ),


            xaxis=dict(

                range=[
                    0,
                    100
                ],

                ticksuffix="%",

                gridcolor=(
                    "rgba("
                    "170,"
                    "195,"
                    "200,"
                    "0.12"
                    ")"
                ),

                zeroline=False,

                tickfont=dict(

                    size=11,

                    color="#C7D8DB"
                )
            ),


            yaxis=dict(

                categoryorder="array",

                categoryarray=[

                    "Mala",

                    "Media",

                    "Buena"
                ],

                tickfont=dict(

                    size=12,

                    color="#E1ECEE"
                )
            )
        )


        st.plotly_chart(

            fig_class,

            use_container_width=True,

            config={

                "displayModeBar": False,

                "responsive": True
            }
        )

'''


# ============================================================
# 3 · RECONSTRUIR APP
# ============================================================

app_new = (
    prefix
    + NEW_OVERVIEW
    + "\n"
    + views_tail
)


# ============================================================
# 4 · COMPROBACIONES DE ESTRUCTURA
# ============================================================

required = [

    'if selected == "Overview":',

    'elif selected == "Temporal":',

    'elif selected == "Spatial":',

    'elif selected == "Model Insights":',

    'elif selected == "Predictor":',

    "bb-kpi-card",

    "html_kpi_1",

    "html_kpi_2",

    "html_kpi_3",

    "html_kpi_4",

    "NO2_ug_m3",

    "O3_ug_m3",

    "PM10_ug_m3",

    "PM25_ug_m3",

    "clase_NO2_3",

    "estacion_geo",

    "CLAVE_SECCION",

    "NO2_CLASE",

    "NO2_MEDIO_SECCION",

    "World_Dark_Gray_Base",

    "World_Dark_Gray_Reference",

    "Evolución mensual",

    "Distribución de",

    "Clasificación de"
]


for item in required:

    if item not in app_new:

        raise ValueError(
            "❌ Falta elemento obligatorio: "
            f"{item}"
        )


# ============================================================
# 5 · COMPROBAR RESIDUO "el"
# ============================================================

isolated_el = [

    i + 1

    for i, line
    in enumerate(
        app_new.splitlines()
    )

    if line.strip() == "el"
]


if isolated_el:

    raise ValueError(
        "❌ Se detectó una línea aislada 'el' "
        f"en app.py: {isolated_el}"
    )


# ============================================================
# 6 · COMPROBACIÓN ESPECÍFICA LEYENDA FINAL
# ============================================================

if "y=1.16" not in app_new:

    raise ValueError(
        "❌ No se encontró y=1.16 "
        "en la leyenda temporal."
    )


if 'yanchor="top"' not in app_new:

    raise ValueError(
        "❌ No se encontró yanchor='top' "
        "en la leyenda temporal."
    )


# ============================================================
# 7 · VALIDACIÓN DE SINTAXIS
# ============================================================
#
# app.py NO se escribe si compile() falla.
# ============================================================

try:

    compile(
        app_new,
        str(APP_PATH),
        "exec"
    )


except SyntaxError as e:

    print()

    print("=" * 80)

    print(
        "❌ ERROR DE SINTAXIS"
    )

    print("=" * 80)

    print()

    print(
        f"Línea   : {e.lineno}"
    )

    print(
        f"Columna : {e.offset}"
    )

    print(
        f"Mensaje : {e.msg}"
    )

    print()

    print(
        "⚠️ app.py NO ha sido modificado."
    )

    print("=" * 80)

    raise


# ============================================================
# 8 · GUARDAR
# ============================================================

APP_PATH.write_text(
    app_new,
    encoding="utf-8"
)


# ============================================================
# 9 · VERIFICACIÓN DEL ARCHIVO ESCRITO
# ============================================================

saved_code = (
    APP_PATH.read_text(
        encoding="utf-8"
    )
)


compile(
    saved_code,
    str(APP_PATH),
    "exec"
)


saved_el = [

    i + 1

    for i, line
    in enumerate(
        saved_code.splitlines()
    )

    if line.strip() == "el"
]


if saved_el:

    raise ValueError(
        "❌ Se detectó 'el' después de guardar: "
        f"{saved_el}"
    )


if "y=1.16" not in saved_code:

    raise ValueError(
        "❌ No se guardó la posición final "
        "de la leyenda."
    )


if 'yanchor="top"' not in saved_code:

    raise ValueError(
        "❌ No se guardó yanchor='top'."
    )


# ============================================================
# RESULTADO
# ============================================================

print("=" * 80)

print(
    "🌬️ BREATHE BARCELONA · 06.4 FINAL"
)

print("=" * 80)

print()

print(
    "✅ Overview reconstruido"
)

print(
    "✅ Residuo aislado 'el': 0"
)

print(
    "✅ Título alineado"
)

print(
    "✅ Separación título / Periodo"
)

print(
    "✅ Periodo + KPI alineados"
)

print(
    "✅ KPI equilibrados"
)

print(
    "✅ Tipografía secundaria reforzada"
)

print(
    "✅ Mapa ESRI conservado"
)

print(
    "✅ Modebar del mapa oculta"
)

print(
    "✅ Escala NO₂ conservada"
)

print(
    "✅ Evolución mensual conservada"
)

print(
    "✅ Leyenda temporal elevada · y=1.16"
)

print(
    "✅ Boxplots conservados"
)

print(
    "✅ Clasificación NO₂ conservada"
)

print(
    "✅ Temporal conservado"
)

print(
    "✅ Spatial conservado"
)

print(
    "✅ Model Insights conservado"
)

print(
    "✅ Predictor conservado"
)

print(
    "✅ Sintaxis verificada antes de guardar"
)

print(
    "✅ Sintaxis verificada después de guardar"
)

print()

print(
    "🎯 06.4 · OVERVIEW FINAL"
)

print("=" * 80)

🌬️ BREATHE BARCELONA · 06.4 FINAL

✅ Overview reconstruido
✅ Residuo aislado 'el': 0
✅ Título alineado
✅ Separación título / Periodo
✅ Periodo + KPI alineados
✅ KPI equilibrados
✅ Tipografía secundaria reforzada
✅ Mapa ESRI conservado
✅ Modebar del mapa oculta
✅ Escala NO₂ conservada
✅ Evolución mensual conservada
✅ Leyenda temporal elevada · y=1.16
✅ Boxplots conservados
✅ Clasificación NO₂ conservada
✅ Temporal conservado
✅ Spatial conservado
✅ Model Insights conservado
✅ Predictor conservado
✅ Sintaxis verificada antes de guardar
✅ Sintaxis verificada después de guardar

🎯 06.4 · OVERVIEW FINAL


### Resultados

La vista **Overview** queda reconstruida y validada como pantalla inicial de *Breathe Barcelona*, integrando en una única composición los principales indicadores temporales y espaciales de calidad del aire.

La versión definitiva incorpora:

* filtrado dinámico por periodo;
* cuatro KPI equilibrados y alineados;
* mapa espacial de NO₂ sobre cartografía **ESRI World Dark Gray**, manteniendo la escala cromática y la información interactiva por sección censal;
* evolución mensual de **NO₂, O₃, PM10 y PM2.5**;
* comparación de sus distribuciones mediante boxplots;
* clasificación porcentual de la calidad del NO₂;
* jerarquía tipográfica y alineación visual coherentes con la identidad gráfica del dashboard.

La implementación conserva íntegramente las vistas **Temporal, Spatial, Model Insights y Predictor**. Asimismo, se aplican controles defensivos sobre la estructura del código y se verifica la sintaxis de `app.py` **antes y después de su escritura**, evitando guardar una versión sintácticamente inválida.

> **Resultado:** la **Overview definitiva** queda integrada como una síntesis interactiva del proyecto, conectando indicadores, dimensión temporal y distribución espacial en una primera lectura global de la calidad del aire de Barcelona.


### 6.5 · Construcción de la vista Temporal

Se construye la vista **Temporal** de *Breathe Barcelona* como módulo específico para explorar la dinámica del **NO₂ entre 2019 y 2024**, su relación con diferentes factores urbanos y ambientales y el comportamiento de los modelos de predicción temporal.

La interfaz incorpora cuatro controles interactivos que permiten seleccionar:

* **periodo:** conjunto completo, Pre-COVID, COVID o Post-COVID;
* **resolución temporal:** mensual, semanal o diaria;
* **driver:** tráfico, temperatura, ruido, vuelos o actividad portuaria;
* **modelo de forecast:** SARIMAX con tráfico, SARIMA o Naive t−1.

A partir de estas selecciones se actualizan dinámicamente cuatro **KPI** —NO₂ medio, percentil 95, concentración máxima estación-día y número de observaciones— y se construyen dos niveles complementarios de análisis.

La parte superior muestra la **evolución temporal del NO₂**, destacando el periodo COVID cuando se visualiza la serie completa, y permite explorar su asociación con distintos drivers mediante gráficos de dispersión y el correspondiente coeficiente de correlación.

La parte inferior integra los resultados de modelización mediante el **rolling forecast de 2024**, sus métricas de evaluación y la importancia global de variables mediante **SHAP para la clase Mala** del modelo XGBoost.

La reconstrucción sustituye de forma controlada el bloque Temporal completo dentro de `app.py`, manteniendo intactas las restantes vistas y aplicando validaciones estructurales y sintácticas antes y después de guardar el archivo.

> **Objetivo:** integrar en una única vista interactiva la **evolución, factores asociados, predicción e interpretabilidad temporal del NO₂**, conectando el análisis exploratorio con los modelos desarrollados durante el proyecto.


In [26]:
# ============================================================
# 🌬️ BREATHE BARCELONA
# 06.5 · TEMPORAL · FINAL V5
# ============================================================
#
# ✓ BLOQUE TEMPORAL COMPLETO
# ✓ NO ES UN PARCHE
# ✓ Mantiene layout V4 validado
# ✓ Mantiene separación cabecera / selectores
# ✓ Mantiene KPI
# ✓ Mantiene alturas de gráficos
# ✓ SHAP: nueva paleta azul → cyan
# ✓ Elimina amarillo del SHAP
# ✓ Misma lógica / datos / modelos
# ✓ Validación antes y después de escribir app.py
# ============================================================

from pathlib import Path


# ============================================================
# 0 · RUTA
# ============================================================

APP_PATH = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard/app.py"
)

if not APP_PATH.exists():
    raise FileNotFoundError(
        f"❌ No existe app.py:\n{APP_PATH}"
    )


# ============================================================
# 1 · LEER APP
# ============================================================

code = APP_PATH.read_text(
    encoding="utf-8"
)


# ============================================================
# 2 · LIMPIEZA DEFENSIVA
# ============================================================

original_lines = code.splitlines()

clean_lines = [
    line
    for line in original_lines
    if line.strip() != "el"
]

removed_el = len(original_lines) - len(clean_lines)

if removed_el:
    print(
        f"🧹 Eliminado residuo aislado 'el': "
        f"{removed_el} línea(s)"
    )

code = "\n".join(clean_lines) + "\n"


# ============================================================
# 3 · LOCALIZAR TEMPORAL
# ============================================================

start = code.find(
    'elif selected == "Temporal":'
)

end = code.find(
    'elif selected == "Spatial":',
    start
)

if start == -1:
    raise RuntimeError(
        "❌ No se encontró el bloque Temporal."
    )

if end == -1:
    raise RuntimeError(
        "❌ No se encontró el bloque Spatial."
    )

if end <= start:
    raise RuntimeError(
        "❌ Orden de vistas incorrecto."
    )


# ============================================================
# 4 · BLOQUE TEMPORAL COMPLETO
# ============================================================

TEMPORAL = r'''
elif selected == "Temporal":

    # ========================================================
    # CSS TEMPORAL
    # ========================================================

    st.markdown(
        """
<style>
div[data-testid="stSelectbox"] > label {
    color: #F3F8FA !important;
    font-size: 12.5px !important;
    font-weight: 650 !important;
    margin-bottom: 1px !important;
}

div[data-testid="stSelectbox"] div[data-baseweb="select"] > div {
    min-height: 39px !important;
    font-size: 12px !important;
    border-radius: 7px !important;
}

div[data-testid="stSelectbox"] div[data-baseweb="select"] > div:hover {
    border-color: rgba(31,232,217,0.65) !important;
}
</style>
        """,
        unsafe_allow_html=True
    )

    # ========================================================
    # PALETA
    # ========================================================

    CYAN = "#1FE8D9"
    CYAN_2 = "#32F3E5"
    BLUE = "#369CFF"
    BLUE_2 = "#58B8FF"
    AMBER = "#FFC44D"
    CORAL = "#FF5E61"
    WHITE = "#F8FBFC"
    MUTED = "#B5C8CE"
    BG = "#071C25"
    GRID = "rgba(255,255,255,0.085)"

    # ========================================================
    # CABECERA COMPACTA
    # ========================================================

    st.markdown(
        """
<div style="margin-top:-12px;margin-bottom:8px;">
<div style="display:flex;align-items:baseline;gap:7px;line-height:1;">
<span style="font-size:24px;font-weight:760;color:#1FE8D9;text-shadow:0 0 12px rgba(31,232,217,0.20);">NO₂</span>
<span style="font-size:24px;font-weight:730;color:#F8FBFC;letter-spacing:-0.2px;">Temporal Dynamics</span>
<span style="font-size:10.5px;font-weight:450;color:#91AAB0;margin-left:5px;">2019–2024</span>
</div>
<div style="font-size:10px;color:#9CB2B8;margin-top:4px;line-height:1.05;">Temporal evolution · drivers · forecasting · classification</div>
</div>
        """,
        unsafe_allow_html=True
    )

    # ========================================================
    # DATOS
    # ========================================================

    df_temp = df_temporal.copy()

    df_temp["fecha"] = pd.to_datetime(
        df_temp["fecha"]
    )

    df_temp = df_temp.sort_values(
        "fecha"
    )

    # ========================================================
    # SELECTORES
    # ========================================================

    c1, c2, c3, c4 = st.columns(
        [1, 1, 1, 1],
        gap="small"
    )

    with c1:
        periodo_sel = st.selectbox(
            "Periodo",
            [
                "Todo · 2019–2024",
                "Pre-COVID",
                "COVID",
                "Post-COVID"
            ],
            key="temp_periodo"
        )

    with c2:
        resolucion_sel = st.selectbox(
            "Resolución",
            [
                "Mensual",
                "Semanal",
                "Diaria"
            ],
            key="temp_resolucion"
        )

    with c3:
        driver_sel = st.selectbox(
            "Driver",
            [
                "Tráfico",
                "Temperatura",
                "Ruido",
                "Vuelos",
                "Puerto"
            ],
            key="temp_driver"
        )

    with c4:
        modelo_sel = st.selectbox(
            "Forecast",
            [
                "SARIMAX + tráfico",
                "SARIMA",
                "Naive t-1"
            ],
            key="temp_modelo"
        )

    # ========================================================
    # FILTRADO
    # ========================================================

    df_filtrado = df_temp.copy()

    if periodo_sel != "Todo · 2019–2024":
        df_filtrado = df_filtrado[
            df_filtrado["periodo_covid"] == periodo_sel
        ].copy()

    freq_map = {
        "Mensual": "MS",
        "Semanal": "W",
        "Diaria": "D"
    }

    freq = freq_map[resolucion_sel]

    df_no2 = (
        df_filtrado
        .set_index("fecha")["NO2_ug_m3"]
        .resample(freq)
        .mean()
        .dropna()
        .reset_index()
    )

    # ========================================================
    # KPI
    # ========================================================

    no2_media = df_filtrado["NO2_ug_m3"].mean()

    no2_p95 = df_filtrado[
        "NO2_ug_m3"
    ].quantile(
        0.95
    )

    no2_max = df_filtrado[
        "NO2_ug_m3"
    ].max()

    n_obs = int(
        df_filtrado[
            "NO2_ug_m3"
        ]
        .notna()
        .sum()
    )

    # ========================================================
    # FUNCIÓN KPI
    # ========================================================

    def temporal_kpi(
        icon,
        label,
        value,
        unit,
        accent
    ):

        html = (
            f'<div style="'
            f'border:1px solid {accent};'
            f'border-radius:9px;'
            f'background:linear-gradient(135deg,rgba(12,38,48,0.99),rgba(6,24,32,0.99));'
            f'box-shadow:0 0 12px {accent}14,inset 0 1px 0 rgba(255,255,255,0.025);'
            f'height:74px;'
            f'box-sizing:border-box;'
            f'padding:8px 13px;'
            f'display:flex;'
            f'align-items:center;'
            f'overflow:hidden;'
            f'">'
            f'<div style="'
            f'width:39px;'
            f'height:39px;'
            f'min-width:39px;'
            f'border-radius:50%;'
            f'border:1px solid {accent};'
            f'color:{accent};'
            f'box-shadow:0 0 9px {accent}20;'
            f'display:flex;'
            f'align-items:center;'
            f'justify-content:center;'
            f'font-size:20px;'
            f'margin-right:11px;'
            f'box-sizing:border-box;'
            f'">{icon}</div>'
            f'<div style="'
            f'flex:1;'
            f'display:flex;'
            f'flex-direction:column;'
            f'justify-content:center;'
            f'">'
            f'<div style="'
            f'font-size:11.5px;'
            f'color:{accent};'
            f'font-weight:650;'
            f'line-height:1.05;'
            f'">{label}</div>'
            f'<div style="'
            f'display:flex;'
            f'align-items:baseline;'
            f'margin-top:4px;'
            f'">'
            f'<span style="'
            f'font-size:24px;'
            f'font-weight:760;'
            f'color:#F8FBFC;'
            f'line-height:1;'
            f'">{value}</span>'
            f'<span style="'
            f'font-size:10.5px;'
            f'color:#B5C8CE;'
            f'margin-left:5px;'
            f'">{unit}</span>'
            f'</div>'
            f'</div>'
            f'</div>'
        )

        st.markdown(
            html,
            unsafe_allow_html=True
        )

    # ========================================================
    # FILA KPI
    # ========================================================

    k1, k2, k3, k4 = st.columns(
        [1, 1, 1, 1],
        gap="small"
    )

    with k1:
        temporal_kpi(
            "⌁",
            "NO₂ medio",
            f"{no2_media:.1f}",
            "µg/m³",
            CYAN
        )

    with k2:
        temporal_kpi(
            "▥",
            "Percentil 95",
            f"{no2_p95:.1f}",
            "µg/m³",
            BLUE
        )

    with k3:
        temporal_kpi(
            "↗",
            "Pico estación-día",
            f"{no2_max:.1f}",
            "µg/m³",
            CORAL
        )

    with k4:
        temporal_kpi(
            "▦",
            "Observaciones",
            f"{n_obs:,}",
            "",
            CYAN
        )

    # ========================================================
    # FILA SUPERIOR
    # ========================================================

    left, right = st.columns(
        [0.54, 0.46],
        gap="large"
    )

    # ========================================================
    # EVOLUCIÓN NO₂
    # ========================================================

    with left:

        fig_evol = go.Figure()

        fig_evol.add_trace(
            go.Scatter(
                x=df_no2["fecha"],
                y=df_no2["NO2_ug_m3"],
                mode="lines",
                name="NO₂",
                line=dict(
                    color=CYAN_2,
                    width=2.4
                ),
                hovertemplate=(
                    "%{x|%d %b %Y}"
                    "<br>"
                    "<b>NO₂ %{y:.1f} µg/m³</b>"
                    "<extra></extra>"
                )
            )
        )

        if periodo_sel == "Todo · 2019–2024":

            covid_dates = df_temp.loc[
                df_temp[
                    "periodo_covid"
                ] == "COVID",
                "fecha"
            ]

            if not covid_dates.empty:

                fig_evol.add_vrect(
                    x0=covid_dates.min(),
                    x1=covid_dates.max(),
                    fillcolor=AMBER,
                    opacity=0.095,
                    line_width=0,
                    layer="below"
                )

                fig_evol.add_annotation(
                    x=(
                        covid_dates.min()
                        + (
                            covid_dates.max()
                            - covid_dates.min()
                        ) / 2
                    ),
                    y=1.025,
                    yref="paper",
                    text="COVID 2020–2021",
                    showarrow=False,
                    font=dict(
                        size=11,
                        color=AMBER
                    )
                )

        periodo_titulo = (
            "2019–2024"
            if periodo_sel == "Todo · 2019–2024"
            else periodo_sel
        )

        fig_evol.update_layout(
            title=dict(
                text=(
                    "<b>NO₂ evolution</b>"
                    f" · {periodo_titulo}"
                ),
                font=dict(
                    size=14,
                    color=WHITE
                ),
                x=0.01
            ),
            height=250,
            margin=dict(
                l=50,
                r=14,
                t=38,
                b=30
            ),
            paper_bgcolor=BG,
            plot_bgcolor=BG,
            font=dict(
                color=MUTED,
                size=10
            ),
            showlegend=False,
            hovermode="x unified",
            xaxis=dict(
                showgrid=False,
                title=None,
                tickfont=dict(
                    size=10.5,
                    color=MUTED
                ),
                range=(
                    [
                        df_no2["fecha"].min(),
                        df_no2["fecha"].max()
                    ]
                    if not df_no2.empty
                    else None
                )
            ),
            yaxis=dict(
                title=dict(
                    text="µg/m³",
                    font=dict(
                        size=11,
                        color=WHITE
                    )
                ),
                gridcolor=GRID,
                zeroline=False,
                tickfont=dict(
                    size=10.5,
                    color=MUTED
                )
            )
        )

        st.plotly_chart(
            fig_evol,
            use_container_width=True,
            config={
                "displayModeBar": False,
                "responsive": True
            }
        )

    # ========================================================
    # DRIVER
    # ========================================================

    driver_map = {
        "Tráfico": (
            "trafico_idw_500m",
            "Traffic intensity"
        ),
        "Temperatura": (
            "TM",
            "Temperature"
        ),
        "Ruido": (
            "LAeq_idw_1500m",
            "Noise LAeq"
        ),
        "Vuelos": (
            "Vuelos_total",
            "Flights"
        ),
        "Puerto": (
            "Puerto_movimientos",
            "Port movements"
        )
    }

    driver_col, driver_label = driver_map[
        driver_sel
    ]

    driver_df = (
        df_filtrado[
            [
                "NO2_ug_m3",
                driver_col
            ]
        ]
        .dropna()
        .copy()
    )

    corr = driver_df[
        "NO2_ug_m3"
    ].corr(
        driver_df[
            driver_col
        ]
    )

    # ========================================================
    # SCATTER DRIVER
    # ========================================================

    with right:

        fig_driver = go.Figure()

        fig_driver.add_trace(
            go.Scattergl(
                x=driver_df[
                    driver_col
                ],
                y=driver_df[
                    "NO2_ug_m3"
                ],
                mode="markers",
                marker=dict(
                    size=4.5,
                    color=driver_df[
                        "NO2_ug_m3"
                    ],
                    colorscale=[
                        [0.00, "#155A82"],
                        [0.25, "#258BD5"],
                        [0.50, "#32C5ED"],
                        [0.75, "#1FE8D9"],
                        [1.00, "#FFC44D"]
                    ],
                    opacity=0.55,
                    showscale=False
                ),
                hovertemplate=(
                    f"{driver_label}: "
                    "%{x:.1f}"
                    "<br>"
                    "<b>NO₂ %{y:.1f} µg/m³</b>"
                    "<extra></extra>"
                )
            )
        )

        fig_driver.update_layout(
            title=dict(
                text=(
                    f"<b>NO₂ vs {driver_sel}</b>"
                    f" · r = {corr:.2f}"
                ),
                font=dict(
                    size=14,
                    color=WHITE
                ),
                x=0.01
            ),
            height=250,
            margin=dict(
                l=50,
                r=14,
                t=38,
                b=35
            ),
            paper_bgcolor=BG,
            plot_bgcolor=BG,
            font=dict(
                color=MUTED,
                size=10
            ),
            xaxis=dict(
                title=dict(
                    text=driver_label,
                    font=dict(
                        size=11,
                        color=WHITE
                    )
                ),
                showgrid=False,
                zeroline=False,
                tickfont=dict(
                    size=10.5,
                    color=MUTED
                )
            ),
            yaxis=dict(
                title=dict(
                    text="NO₂ µg/m³",
                    font=dict(
                        size=11,
                        color=WHITE
                    )
                ),
                gridcolor=GRID,
                zeroline=False,
                tickfont=dict(
                    size=10.5,
                    color=MUTED
                )
            )
        )

        st.plotly_chart(
            fig_driver,
            use_container_width=True,
            config={
                "displayModeBar": False,
                "responsive": True
            }
        )

    # ========================================================
    # DATOS FORECAST
    # ========================================================

    TS_DIR = Path(
        "/content/drive/MyDrive/TFM/"
        "13_Modelos_Series_Temporales"
    )

    metricas_ts = pd.read_csv(
        TS_DIR
        / "LF04_metricas_modelos_rolling.csv"
    )

    pred_ts = pd.read_csv(
        TS_DIR
        / "LF04_predicciones_rolling_2024.csv"
    )

    pred_ts[
        "fecha"
    ] = pd.to_datetime(
        pred_ts[
            "fecha"
        ]
    )

    metricas_no2 = metricas_ts[
        metricas_ts[
            "contaminante"
        ] == "NO2"
    ].copy()

    pred_no2 = pred_ts[
        pred_ts[
            "contaminante"
        ] == "NO2"
    ].copy()

    pred_col_map = {
        "SARIMAX + tráfico": "sarimax",
        "SARIMA": "sarima",
        "Naive t-1": "naive_t1"
    }

    metric_model_map = {
        "SARIMAX + tráfico": "SARIMAX_Trafico",
        "SARIMA": "SARIMA_AR1_SAR1",
        "Naive t-1": "Naive t-1"
    }

    pred_col = pred_col_map[
        modelo_sel
    ]

    metric_name = metric_model_map[
        modelo_sel
    ]

    metric_rows = metricas_no2[
        metricas_no2[
            "modelo"
        ] == metric_name
    ]

    if metric_rows.empty:
        raise ValueError(
            "❌ No se encontró la fila de métricas "
            f"para {metric_name}"
        )

    metric_row = metric_rows.iloc[0]

    # ========================================================
    # FILA INFERIOR
    # ========================================================

    b1, b2, b3 = st.columns(
        [0.43, 0.25, 0.32],
        gap="large"
    )

    # ========================================================
    # FORECAST
    # ========================================================

    with b1:

        fig_pred = go.Figure()

        fig_pred.add_trace(
            go.Scatter(
                x=pred_no2[
                    "fecha"
                ],
                y=pred_no2[
                    "observado"
                ],
                mode="lines",
                name="Observed",
                line=dict(
                    color="rgba(248,251,252,0.82)",
                    width=1.35
                ),
                hovertemplate=(
                    "%{x|%d %b %Y}"
                    "<br>"
                    "Observed: <b>%{y:.1f}</b>"
                    "<extra></extra>"
                )
            )
        )

        fig_pred.add_trace(
            go.Scatter(
                x=pred_no2[
                    "fecha"
                ],
                y=pred_no2[
                    pred_col
                ],
                mode="lines",
                name=modelo_sel,
                line=dict(
                    color=CYAN_2,
                    width=1.9
                ),
                hovertemplate=(
                    "%{x|%d %b %Y}"
                    "<br>"
                    f"{modelo_sel}: "
                    "<b>%{y:.1f}</b>"
                    "<extra></extra>"
                )
            )
        )

        fig_pred.update_layout(
            title=dict(
                text=(
                    "<b>Rolling forecast 2024</b>"
                    f" · {modelo_sel}"
                ),
                font=dict(
                    size=13,
                    color=WHITE
                ),
                x=0.01
            ),
            height=235,
            margin=dict(
                l=50,
                r=10,
                t=39,
                b=30
            ),
            paper_bgcolor=BG,
            plot_bgcolor=BG,
            font=dict(
                color=MUTED,
                size=10
            ),
            hovermode="x unified",
            legend=dict(
                orientation="h",
                y=1.12,
                x=0.50,
                xanchor="center",
                font=dict(
                    size=10.5,
                    color=WHITE
                )
            ),
            xaxis=dict(
                showgrid=False,
                tickfont=dict(
                    size=10.5,
                    color=MUTED
                )
            ),
            yaxis=dict(
                title=dict(
                    text="µg/m³",
                    font=dict(
                        size=11,
                        color=WHITE
                    )
                ),
                gridcolor=GRID,
                zeroline=False,
                tickfont=dict(
                    size=10.5,
                    color=MUTED
                )
            )
        )

        st.plotly_chart(
            fig_pred,
            use_container_width=True,
            config={
                "displayModeBar": False,
                "responsive": True
            }
        )

    # ========================================================
    # MÉTRICAS
    # ========================================================

    with b2:

        mejora = float(
            metric_row[
                "mejora_RMSE_vs_naive_pct"
            ]
        )

        mejora_color = (
            CYAN
            if mejora >= 0
            else CORAL
        )

        signo = (
            "+"
            if mejora >= 0
            else ""
        )

        html_metrics = (
            f'<div style="'
            f'border:1px solid rgba(31,232,217,0.42);'
            f'border-radius:9px;'
            f'background:linear-gradient(145deg,rgba(11,38,48,0.99),rgba(6,24,32,0.99));'
            f'box-shadow:0 0 15px rgba(31,232,217,0.055),inset 0 1px 0 rgba(255,255,255,0.025);'
            f'height:214px;'
            f'box-sizing:border-box;'
            f'padding:13px;'
            f'">'
            f'<div style="font-size:12.5px;font-weight:680;color:#F8FBFC;margin-bottom:12px;">'
            f'<span style="color:#1FE8D9;">◎</span> Rolling test · 2024'
            f'</div>'
            f'<div style="display:flex;gap:6px;">'
            f'<div style="flex:1;background:rgba(9,33,42,0.72);border:1px solid rgba(31,232,217,0.20);border-radius:6px;padding:7px;">'
            f'<div style="font-size:10px;color:#B5C8CE;">RMSE</div>'
            f'<div style="font-size:21px;font-weight:760;color:#1FE8D9;">{metric_row["RMSE"]:.2f}</div>'
            f'</div>'
            f'<div style="flex:1;background:rgba(9,33,42,0.72);border:1px solid rgba(54,156,255,0.24);border-radius:6px;padding:7px;">'
            f'<div style="font-size:10px;color:#B5C8CE;">MAE</div>'
            f'<div style="font-size:21px;font-weight:760;color:#58B8FF;">{metric_row["MAE"]:.2f}</div>'
            f'</div>'
            f'<div style="flex:1;background:rgba(9,33,42,0.72);border:1px solid rgba(31,232,217,0.20);border-radius:6px;padding:7px;">'
            f'<div style="font-size:10px;color:#B5C8CE;">R²</div>'
            f'<div style="font-size:21px;font-weight:760;color:#1FE8D9;">{metric_row["R2"]:.2f}</div>'
            f'</div>'
            f'</div>'
            f'<div style="margin-top:10px;background:rgba(9,33,42,0.72);border:1px solid {mejora_color}35;border-radius:6px;padding:8px 9px;">'
            f'<div style="font-size:10px;color:#B5C8CE;">RMSE vs Naive</div>'
            f'<div style="font-size:23px;font-weight:760;color:{mejora_color};text-shadow:0 0 8px {mejora_color}25;line-height:1.05;margin-top:2px;">'
            f'{signo}{mejora:.1f}%'
            f'</div>'
            f'</div>'
            f'</div>'
        )

        st.markdown(
            html_metrics,
            unsafe_allow_html=True
        )

    # ========================================================
    # SHAP
    # ========================================================

    with b3:

        shap_path = Path(
            "/content/drive/MyDrive/TFM/"
            "11_Machine_Learning_Temporal/"
            "03_Datos_Modelado/"
            "shap_importance_clase_mala.csv"
        )

        shap_df = (
            pd.read_csv(
                shap_path
            )
            .head(5)
            .copy()
        )

        shap_df[
            "feature"
        ] = (
            shap_df[
                "feature"
            ]
            .str.replace(
                "num__",
                "",
                regex=False
            )
            .str.replace(
                "_ug_m3",
                "",
                regex=False
            )
            .str.replace(
                "_",
                " ",
                regex=False
            )
        )

        shap_df = shap_df.sort_values(
            "mean_abs_shap"
        )

        # ----------------------------------------------------
        # PALETA SHAP FINAL
        # Azul → cyan
        # El driver principal destaca por LONGITUD,
        # no por un amarillo excesivamente dominante.
        # ----------------------------------------------------

        shap_colors = [
            "#369CFF",
            "#3FAAF5",
            "#42BFEA",
            "#36D5DD",
            "#1FE8D9"
        ]

        fig_shap = go.Figure()

        fig_shap.add_trace(
            go.Bar(
                x=shap_df[
                    "mean_abs_shap"
                ],
                y=shap_df[
                    "feature"
                ],
                orientation="h",
                marker=dict(
                    color=shap_colors,
                    line=dict(
                        color=(
                            "rgba("
                            "255,255,255,0.10"
                            ")"
                        ),
                        width=0.5
                    )
                ),
                text=[
                    f"{x:.2f}"
                    for x
                    in shap_df[
                        "mean_abs_shap"
                    ]
                ],
                textposition="outside",
                textfont=dict(
                    size=10.5,
                    color=WHITE
                ),
                hovertemplate=(
                    "<b>%{y}</b>"
                    "<br>"
                    "mean |SHAP|: %{x:.3f}"
                    "<extra></extra>"
                )
            )
        )

        fig_shap.update_layout(
            title=dict(
                text=(
                    "<b>XGBoost</b>"
                    " · SHAP class Mala"
                ),
                font=dict(
                    size=13,
                    color=WHITE
                ),
                x=0.01
            ),
            height=235,
            margin=dict(
                l=103,
                r=35,
                t=39,
                b=31
            ),
            paper_bgcolor=BG,
            plot_bgcolor=BG,
            font=dict(
                color=MUTED,
                size=10
            ),
            bargap=0.27,
            xaxis=dict(
                title=dict(
                    text="mean |SHAP|",
                    font=dict(
                        size=11,
                        color=WHITE
                    )
                ),
                showgrid=False,
                zeroline=False,
                tickfont=dict(
                    size=10.5,
                    color=MUTED
                )
            ),
            yaxis=dict(
                tickfont=dict(
                    size=10.5,
                    color=WHITE
                )
            )
        )

        st.plotly_chart(
            fig_shap,
            use_container_width=True,
            config={
                "displayModeBar": False,
                "responsive": True
            }
        )

'''


# ============================================================
# 5 · RECONSTRUIR APP
# ============================================================

new_code = (
    code[:start]
    + TEMPORAL
    + "\n"
    + code[end:]
)


# ============================================================
# 6 · VALIDACIÓN ESTRUCTURAL
# ============================================================

required = [
    'if selected == "Overview":',
    'elif selected == "Temporal":',
    'elif selected == "Spatial":',
    'elif selected == "Model Insights":',
    'elif selected == "Predictor":',
    "Temporal Dynamics",
    "temp_periodo",
    "temp_resolucion",
    "temp_driver",
    "temp_modelo",
    "NO2_ug_m3",
    "periodo_covid",
    "trafico_idw_500m",
    "Rolling forecast 2024",
    "LF04_metricas_modelos_rolling.csv",
    "LF04_predicciones_rolling_2024.csv",
    "shap_importance_clase_mala.csv",
    "mean_abs_shap"
]

for item in required:

    if item not in new_code:

        raise ValueError(
            "❌ Falta elemento obligatorio "
            f"en 06.5: {item}"
        )


# ============================================================
# 7 · UNA ÚNICA VISTA TEMPORAL
# ============================================================

n_temporal = new_code.count(
    'elif selected == "Temporal":'
)

if n_temporal != 1:

    raise ValueError(
        "❌ Se esperaba exactamente "
        "1 bloque Temporal y se encontraron "
        f"{n_temporal}."
    )


# ============================================================
# 8 · COMPROBAR RESIDUO "el"
# ============================================================

isolated_el = [
    i + 1
    for i, line
    in enumerate(
        new_code.splitlines()
    )
    if line.strip() == "el"
]

if isolated_el:

    raise ValueError(
        "❌ Línea aislada 'el' detectada "
        f"en {isolated_el}. "
        "app.py NO se modifica."
    )


# ============================================================
# 9 · VALIDAR SINTAXIS ANTES DE GUARDAR
# ============================================================

try:

    compile(
        new_code,
        str(APP_PATH),
        "exec"
    )

except SyntaxError as e:

    print()
    print("=" * 80)
    print("❌ ERROR DE SINTAXIS EN 06.5")
    print("=" * 80)
    print(f"Línea   : {e.lineno}")
    print(f"Columna : {e.offset}")
    print(f"Mensaje : {e.msg}")
    print()
    print("⚠️ app.py NO ha sido modificado.")
    print("=" * 80)

    raise


# ============================================================
# 10 · GUARDAR
# ============================================================

APP_PATH.write_text(
    new_code,
    encoding="utf-8"
)


# ============================================================
# 11 · VERIFICACIÓN POST-GUARDADO
# ============================================================

saved_code = APP_PATH.read_text(
    encoding="utf-8"
)

compile(
    saved_code,
    str(APP_PATH),
    "exec"
)

if saved_code.count(
    'elif selected == "Temporal":'
) != 1:

    raise ValueError(
        "❌ Número incorrecto de bloques Temporal "
        "después de guardar."
    )

saved_el = [
    i + 1
    for i, line
    in enumerate(
        saved_code.splitlines()
    )
    if line.strip() == "el"
]

if saved_el:

    raise ValueError(
        "❌ Se detectó una línea aislada 'el' "
        f"después de guardar: {saved_el}"
    )


# ============================================================
# 12 · RESULTADO
# ============================================================

print("=" * 80)
print("🌬️ BREATHE BARCELONA · 06.5 TEMPORAL")
print("=" * 80)
print()

print("✅ Temporal V5 reconstruido COMPLETO")
print("✅ Sin parches")
print("✅ Layout V4 conservado")
print("✅ Separación cabecera / selectores conservada")
print("✅ KPI conservados")
print("✅ Gráficos superiores conservados")
print("✅ Gráficos inferiores conservados")
print()

print("🎨 SHAP")
print("   ✓ Amarillo eliminado")
print("   ✓ Paleta azul → cyan")
print("   ✓ O3 destaca por magnitud, no por color")
print("   ✓ Ámbar reservado para COVID / eventos")
print()

print("📊 LÓGICA")
print("   ✓ Datos sin modificar")
print("   ✓ Cálculos sin modificar")
print("   ✓ Forecast sin modificar")
print("   ✓ Métricas sin modificar")
print("   ✓ Valores SHAP sin modificar")
print()

print("🧩 RESTO DEL DASHBOARD")
print("   ✓ Overview conservado")
print("   ✓ Spatial conservado")
print("   ✓ Model Insights conservado")
print("   ✓ Predictor conservado")
print()

print("🛡️ VALIDACIÓN")
print("   ✓ Un único bloque Temporal")
print("   ✓ Residuo aislado 'el': 0")
print("   ✓ Sintaxis correcta antes de guardar")
print("   ✓ Sintaxis correcta después de guardar")
print()

print("🎯 06.5 · TEMPORAL · V5 FINAL APLICADA")
print("=" * 80)

🌬️ BREATHE BARCELONA · 06.5 TEMPORAL

✅ Temporal V5 reconstruido COMPLETO
✅ Sin parches
✅ Layout V4 conservado
✅ Separación cabecera / selectores conservada
✅ KPI conservados
✅ Gráficos superiores conservados
✅ Gráficos inferiores conservados

🎨 SHAP
   ✓ Amarillo eliminado
   ✓ Paleta azul → cyan
   ✓ O3 destaca por magnitud, no por color
   ✓ Ámbar reservado para COVID / eventos

📊 LÓGICA
   ✓ Datos sin modificar
   ✓ Cálculos sin modificar
   ✓ Forecast sin modificar
   ✓ Métricas sin modificar
   ✓ Valores SHAP sin modificar

🧩 RESTO DEL DASHBOARD
   ✓ Overview conservado
   ✓ Spatial conservado
   ✓ Model Insights conservado
   ✓ Predictor conservado

🛡️ VALIDACIÓN
   ✓ Un único bloque Temporal
   ✓ Residuo aislado 'el': 0
   ✓ Sintaxis correcta antes de guardar
   ✓ Sintaxis correcta después de guardar

🎯 06.5 · TEMPORAL · V5 FINAL APLICADA


### Resultados

La vista **Temporal** queda reconstruida como un módulo interactivo completo que combina análisis descriptivo, exploración de factores asociados, forecasting e interpretabilidad del modelo.

La versión definitiva mantiene la lógica y los resultados analíticos originales e incorpora:

* selección dinámica de **periodo, resolución temporal, driver y modelo de forecast**;
* cuatro KPI de síntesis del comportamiento del NO₂;
* evolución temporal configurable entre 2019 y 2024, con identificación visual del **periodo COVID 2020–2021**;
* análisis de asociación entre NO₂ y **tráfico, temperatura, ruido, vuelos y actividad portuaria**;
* comparación entre valores observados y predicciones del **rolling forecast 2024**;
* evaluación mediante **RMSE, MAE, R² y mejora frente al modelo Naive**;
* representación de las cinco variables con mayor importancia global mediante **mean |SHAP|** para la clase Mala del modelo XGBoost.

La identidad visual se homogeneiza mediante una paleta **azul–cyan** para SHAP, de forma que la importancia de cada variable se comunique principalmente mediante la longitud de las barras y no mediante un color de alerta. El **ámbar queda reservado para acontecimientos o periodos contextuales**, como COVID.

La reconstrucción conserva las vistas **Overview, Spatial, Model Insights y Predictor**, verifica la existencia de un único bloque Temporal y valida la sintaxis de `app.py` antes y después de su escritura.

> **Resultado:** la vista **Temporal definitiva** integra en una única interfaz la evolución histórica del NO₂, sus principales drivers, la capacidad predictiva de los modelos temporales y la interpretación de las variables relevantes, manteniendo intactos los datos, cálculos, métricas y resultados previamente obtenidos.


### 6.6 · Construcción de la vista Spatial

La vista **Spatial** integra la dimensión territorial del análisis de calidad del aire sobre las **1.068 secciones censales de Barcelona**, combinando la distribución espacial de NO₂ con variables urbanas y los resultados de modelización geoespacial.

El módulo incorpora la exploración cartográfica, el análisis **LISA** de autocorrelación espacial local, la asociación espacial bivariada entre NO₂ y factores urbanos, y los resultados de los modelos **SAR** y **XGBoost**. Se mantiene únicamente información espacial reproducible y trazable desde el pipeline analítico, evitando incorporar indicadores globales no consolidados en el flujo final.

La construcción se realiza preservando el resto de vistas de *Breathe Barcelona* y verificando la integridad de los componentes analíticos y la sintaxis de la aplicación antes y después del guardado.

> **Objetivo:** integrar en una única vista interactiva la distribución territorial del NO₂, los patrones locales de autocorrelación espacial y los principales resultados de la modelización geoespacial.


In [32]:
# ============================================================
# 🌬️ BREATHE BARCELONA
# 06.6 · SPATIAL · LIMPIEZA SEGURA MORAN GLOBAL
# ============================================================
#
# OBJETIVO:
# - NO reconstruir Spatial
# - NO tocar LISA
# - NO tocar Moran bivariado
# - NO tocar SAR
# - NO tocar XGBoost
# - NO tocar mapas ni selectores
# - Eliminar únicamente:
#       MORAN_GLOBAL_NO2
#       MORAN_GLOBAL_P
#       referencia visual "Global Moran I ..."
#
# Si alguna condición de seguridad falla:
#       app.py NO se modifica
#
# ============================================================

from pathlib import Path
from datetime import datetime
import re


# ============================================================
# 0 · RUTAS
# ============================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard"
)

APP_PATH = BASE_DIR / "app.py"


if not APP_PATH.exists():
    raise FileNotFoundError(
        f"❌ No existe app.py:\n{APP_PATH}"
    )


# ============================================================
# 1 · LEER APP
# ============================================================

code = APP_PATH.read_text(
    encoding="utf-8"
)


# ============================================================
# 2 · LOCALIZAR SPATIAL
# ============================================================

start_marker = 'elif selected == "Spatial":'
end_marker = 'elif selected == "Model Insights":'

start = code.find(
    start_marker
)

end = code.find(
    end_marker,
    start
)


if start == -1:
    raise RuntimeError(
        "❌ No se encontró Spatial."
    )

if end == -1:
    raise RuntimeError(
        "❌ No se encontró Model Insights "
        "después de Spatial."
    )

if end <= start:
    raise RuntimeError(
        "❌ Orden incorrecto de vistas."
    )


spatial = code[start:end]


# ============================================================
# 3 · VALIDACIONES DE SEGURIDAD
# ============================================================
#
# No usamos títulos ni textos decorativos.
# Solo comprobamos componentes analíticos reales.
# ============================================================

security_groups = {

    "LISA":
        [
            "LISA_CLUSTER",
            "lisa_counts",
            "n_hh",
            "n_ll"
        ],

    "Moran bivariado":
        [
            "MORAN_BIV",
            "Moran I bivariado"
        ],

    "SAR":
        [
            "SAR_R2",
            "SAR_RMSE"
        ],

    "XGBoost":
        [
            "XGBoost",
            "NO2_PRED_XGB"
        ]
}


print("=" * 88)
print("🌬️ 06.6 · COMPROBACIÓN PREVIA SPATIAL")
print("=" * 88)
print()

print(
    f"Spatial localizado: "
    f"{len(spatial):,} caracteres"
)

print()


for name, markers in security_groups.items():

    found = any(
        marker in spatial
        for marker in markers
    )

    print(
        f"{'✅' if found else '❌'} {name}"
    )

    if not found:

        raise RuntimeError(
            f"\n❌ No se detecta {name}.\n"
            "⚠️ Por seguridad app.py NO se modifica."
        )


# ============================================================
# 4 · GUARDAR COPIAS DE LOS COMPONENTES CRÍTICOS
# ============================================================
#
# Después de la modificación volveremos a comprobar
# que siguen presentes.
# ============================================================

critical_markers = [

    "LISA_CLUSTER",

    "MORAN_BIV",

    "SAR_R2",

    "SAR_RMSE",

    "XGBoost"
]


before_presence = {
    marker: marker in spatial
    for marker in critical_markers
}


# ============================================================
# 5 · DIAGNÓSTICO MORAN GLOBAL
# ============================================================

has_global_no2 = (
    "MORAN_GLOBAL_NO2"
    in spatial
)

has_global_p = (
    "MORAN_GLOBAL_P"
    in spatial
)

has_global_text = (
    "Global Moran I"
    in spatial
)


print()
print("MORAN GLOBAL NO TRAZADO")
print("-" * 88)

print(
    "MORAN_GLOBAL_NO2:",
    "✅ detectado"
    if has_global_no2
    else "— no presente"
)

print(
    "MORAN_GLOBAL_P:",
    "✅ detectado"
    if has_global_p
    else "— no presente"
)

print(
    "Texto Global Moran I:",
    "✅ detectado"
    if has_global_text
    else "— no presente"
)


# ============================================================
# 6 · CREAR BACKUP ANTES DE CUALQUIER CAMBIO
# ============================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

backup_path = (
    BASE_DIR
    / (
        "app_backup_antes_066_moran_"
        f"{timestamp}.py"
    )
)

backup_path.write_text(
    code,
    encoding="utf-8"
)


# ============================================================
# 7 · MODIFICAR SOLO SPATIAL
# ============================================================

spatial_new = spatial


# ------------------------------------------------------------
# 7.1 · ELIMINAR DEFINICIONES MORAN GLOBAL
# ------------------------------------------------------------

spatial_new = re.sub(
    r'(?m)^[ \t]*MORAN_GLOBAL_NO2\s*=.*\n?',
    '',
    spatial_new
)

spatial_new = re.sub(
    r'(?m)^[ \t]*MORAN_GLOBAL_P\s*=.*\n?',
    '',
    spatial_new
)


# ============================================================
# 7.2 · SUSTITUIR SOLO LA CABECERA QUE USA ESAS VARIABLES
# ============================================================
#
# Buscamos específicamente el bloque moran_title
# que contiene Global Moran I.
#
# Si existe, se sustituye por una versión bivariada.
# Si no existe, no se toca ningún otro bloque.
# ============================================================

pattern = re.compile(
    r'''
    (?P<indent>^[ \t]*)
    moran_title\s*=\s*\(
    (?P<body>.*?)
    \n(?P=indent)\)
    ''',
    flags=(
        re.MULTILINE
        | re.DOTALL
        | re.VERBOSE
    )
)


matches = list(
    pattern.finditer(
        spatial_new
    )
)


global_title_matches = [

    m
    for m in matches

    if (
        "Global Moran I"
        in m.group(0)

        or

        "MORAN_GLOBAL_NO2"
        in m.group(0)

        or

        "MORAN_GLOBAL_P"
        in m.group(0)
    )
]


if len(global_title_matches) > 1:

    raise RuntimeError(
        "❌ Se encontraron varios bloques "
        "moran_title con Moran global.\n"
        "⚠️ Por seguridad app.py NO se modifica."
    )


if len(global_title_matches) == 1:

    m = global_title_matches[0]

    indent = m.group(
        "indent"
    )

    replacement = (
        f'{indent}moran_title = (\n'
        f'{indent}    "<b>'
        f'Bivariate spatial association · NO₂'
        f'</b>"\n'
        f'{indent}    "<br>"\n'
        f'{indent}    "<span style=\'"\n'
        f'{indent}    "font-size:11px;"\n'
        f'{indent}    "color:#B5C7CD;"\n'
        f'{indent}    "\'>"\n'
        f'{indent}    '
        f'f"{{factor_note.lstrip(\' · \')}}"\n'
        f'{indent}    "</span>"\n'
        f'{indent})'
    )

    spatial_new = (
        spatial_new[:m.start()]
        + replacement
        + spatial_new[m.end():]
    )


# ============================================================
# 8 · VALIDAR QUE NO QUEDA MORAN GLOBAL
# ============================================================

for forbidden in [

    "MORAN_GLOBAL_NO2",

    "MORAN_GLOBAL_P",

    "Global Moran I",

    "0.7821"

]:

    if forbidden in spatial_new:

        raise RuntimeError(
            "❌ Sigue presente contenido "
            f"no trazado: {forbidden}\n"
            "⚠️ app.py NO se modifica."
        )


# ============================================================
# 9 · VALIDAR QUE NO HEMOS TOCADO LISA / MORAN BIV / SAR / XGB
# ============================================================

for marker, was_present in before_presence.items():

    if (
        was_present
        and marker not in spatial_new
    ):

        raise RuntimeError(
            "❌ Se perdería un componente "
            f"crítico: {marker}\n"
            "⚠️ app.py NO se modifica."
        )


# ============================================================
# 10 · VALIDAR DE NUEVO LOS GRUPOS ANALÍTICOS
# ============================================================

for name, markers in security_groups.items():

    if not any(
        marker in spatial_new
        for marker in markers
    ):

        raise RuntimeError(
            f"❌ Tras la limpieza falta {name}.\n"
            "⚠️ app.py NO se modifica."
        )


# ============================================================
# 11 · RECONSTRUIR APP
# ============================================================

new_code = (
    code[:start]
    + spatial_new
    + code[end:]
)


# ============================================================
# 12 · VALIDAR LAS CINCO VISTAS
# ============================================================

required_views = [

    'selected == "Overview"',

    'selected == "Temporal"',

    'selected == "Spatial"',

    'selected == "Model Insights"',

    'selected == "Predictor"'
]


for view in required_views:

    if view not in new_code:

        raise RuntimeError(
            f"❌ Se perdería la vista "
            f"{view}.\n"
            "⚠️ app.py NO se modifica."
        )


# ============================================================
# 13 · VALIDAR UN ÚNICO SPATIAL
# ============================================================

if new_code.count(
    'elif selected == "Spatial":'
) != 1:

    raise RuntimeError(
        "❌ Número incorrecto "
        "de bloques Spatial.\n"
        "⚠️ app.py NO se modifica."
    )


# ============================================================
# 14 · VALIDAR SINTAXIS ANTES DE GUARDAR
# ============================================================

try:

    compile(
        new_code,
        str(APP_PATH),
        "exec"
    )

except SyntaxError as e:

    print()
    print("=" * 88)
    print("❌ ERROR DE SINTAXIS")
    print("=" * 88)

    print(
        "Línea:",
        e.lineno
    )

    print(
        "Columna:",
        e.offset
    )

    print(
        "Mensaje:",
        e.msg
    )

    print()
    print(
        "⚠️ app.py NO se modifica."
    )

    raise


# ============================================================
# 15 · GUARDAR
# ============================================================

APP_PATH.write_text(
    new_code,
    encoding="utf-8"
)


# ============================================================
# 16 · RELEER Y VALIDAR ARCHIVO GUARDADO
# ============================================================

saved_code = APP_PATH.read_text(
    encoding="utf-8"
)


compile(
    saved_code,
    str(APP_PATH),
    "exec"
)


saved_start = saved_code.find(
    start_marker
)

saved_end = saved_code.find(
    end_marker,
    saved_start
)

saved_spatial = saved_code[
    saved_start:saved_end
]


# ============================================================
# 17 · VALIDACIÓN FINAL MUY ESTRICTA
# ============================================================

for forbidden in [

    "MORAN_GLOBAL_NO2",

    "MORAN_GLOBAL_P",

    "Global Moran I",

    "0.7821"

]:

    if forbidden in saved_spatial:

        raise RuntimeError(
            "❌ Después de guardar "
            f"sigue presente: {forbidden}"
        )


for name, markers in security_groups.items():

    if not any(
        marker in saved_spatial
        for marker in markers
    ):

        raise RuntimeError(
            "❌ Después de guardar "
            f"se perdió: {name}"
        )


# ============================================================
# 18 · RESULTADO
# ============================================================

print()
print("=" * 88)

print(
    "🌬️ BREATHE BARCELONA · "
    "06.6 SPATIAL VALIDADO"
)

print("=" * 88)
print()

print(
    f"✅ Spatial conservado · "
    f"{len(saved_spatial):,} caracteres"
)

print()

print("🧭 MORAN")
print(
    "   ✅ Moran global no trazado eliminado"
)
print(
    "   ✅ MORAN_GLOBAL_NO2 eliminado"
)
print(
    "   ✅ MORAN_GLOBAL_P eliminado"
)
print(
    "   ✅ Global Moran I eliminado de la interfaz"
)
print(
    "   ✅ Moran bivariado conservado"
)

print()

print("🗺️ ANÁLISIS ESPACIAL")
print(
    "   ✅ LISA conservado"
)
print(
    "   ✅ SAR conservado"
)
print(
    "   ✅ XGBoost conservado"
)

print()

print("🛡️ SEGURIDAD")
print(
    "   ✅ Overview conservado"
)
print(
    "   ✅ Temporal conservado"
)
print(
    "   ✅ Spatial conservado"
)
print(
    "   ✅ Model Insights conservado"
)
print(
    "   ✅ Predictor conservado"
)
print(
    "   ✅ Sintaxis correcta antes de guardar"
)
print(
    "   ✅ Sintaxis correcta después de guardar"
)

print()

print("📦 Backup previo:")
print(
    backup_path
)

print()

print("=" * 88)
print(
    "🎯 06.6 · LIMPIEZA MORAN COMPLETADA"
)
print("=" * 88)

🌬️ 06.6 · COMPROBACIÓN PREVIA SPATIAL

Spatial localizado: 48,046 caracteres

✅ LISA
✅ Moran bivariado
✅ SAR
✅ XGBoost

MORAN GLOBAL NO TRAZADO
----------------------------------------------------------------------------------------
MORAN_GLOBAL_NO2: — no presente
MORAN_GLOBAL_P: — no presente
Texto Global Moran I: — no presente

🌬️ BREATHE BARCELONA · 06.6 SPATIAL VALIDADO

✅ Spatial conservado · 48,046 caracteres

🧭 MORAN
   ✅ Moran global no trazado eliminado
   ✅ MORAN_GLOBAL_NO2 eliminado
   ✅ MORAN_GLOBAL_P eliminado
   ✅ Global Moran I eliminado de la interfaz
   ✅ Moran bivariado conservado

🗺️ ANÁLISIS ESPACIAL
   ✅ LISA conservado
   ✅ SAR conservado
   ✅ XGBoost conservado

🛡️ SEGURIDAD
   ✅ Overview conservado
   ✅ Temporal conservado
   ✅ Spatial conservado
   ✅ Model Insights conservado
   ✅ Predictor conservado
   ✅ Sintaxis correcta antes de guardar
   ✅ Sintaxis correcta después de guardar

📦 Backup previo:
/content/drive/MyDrive/TFM/12_Dashboard/app_backup_antes_066_m

### Resultados

La vista **Spatial** queda consolidada como módulo de análisis territorial de *Breathe Barcelona*, manteniendo de forma integrada:

* el análisis **LISA**, con identificación de clústeres locales **HH, LL, HL y LH** y áreas no significativas;
* el **Moran bivariado**, destinado a explorar la asociación espacial entre NO₂ y los factores urbanos seleccionados;
* el modelo **SAR**, con **R² = 0,8112** y **RMSE = 2,3464**;
* el modelo **XGBoost geoespacial**, incluyendo la comparación entre valores observados y predichos y el análisis de residuos;
* la representación cartográfica y los controles interactivos de exploración espacial.

Se excluye de la interfaz el **Moran global de NO₂ no trazado en el pipeline final**, manteniendo únicamente los resultados espaciales reproducibles utilizados en el análisis definitivo. La validación final confirma la conservación de **LISA, Moran bivariado, SAR y XGBoost**, así como la integridad de las cinco vistas del dashboard y la correcta sintaxis de la aplicación.

> **Resultado:** se obtiene una vista espacial estable y metodológicamente trazable que combina autocorrelación local, relaciones espaciales con factores urbanos y modelización predictiva del NO₂.


### 6.7 · Construcción de la vista Model Insights

La vista **Model Insights** concentra la evaluación e interpretación de los modelos de clasificación temporal, complementando las vistas descriptivas y espaciales con información sobre **rendimiento, generalización e importancia de variables**.

El módulo integra los principales indicadores de evaluación —**Accuracy, F1 Macro y Balanced Accuracy**—, el análisis de errores de clasificación y del rendimiento por clase, la validación espacial **Leave-One-Station-Out (LOSO)** y la interpretación mediante **SHAP**. Esta combinación permite valorar el comportamiento global del modelo sin perder de vista las diferencias entre clases ni su capacidad de generalización entre estaciones.

La vista se incorpora de forma independiente, preservando las restantes secciones de *Breathe Barcelona* y verificando la integridad y sintaxis de la aplicación.

> **Objetivo:** proporcionar una lectura integrada del rendimiento, los errores, la generalización espacial y la interpretabilidad del modelo temporal.


In [33]:
# ============================================================
# 🌬️ BREATHE BARCELONA
# 06.7 · MODEL INSIGHTS — RECUPERACIÓN FINAL DESDE MASTER
# ============================================================
#
# OBJETIVO:
#   Recuperar EXCLUSIVAMENTE la vista Model Insights completa
#   desde:
#
#   app_FINAL_FUNCIONAL.py
#
#   y sustituir el placeholder actual de app.py.
#
# CONSERVA:
#   - Overview
#   - Temporal
#   - Spatial
#   - Predictor actual
#   - router / navegación
#   - resto de app.py
#
# RECUPERA:
#   - KPI finales V6
#   - Confusion matrix
#   - SHAP
#   - LOSO
#   - Performance by class
#   - Key insight
#
# NO RECONSTRUYE LA SHEET
# NO PARCHEA CSS
# COPIA EL BLOQUE FINAL YA VALIDADO DEL MASTER
# ============================================================

from pathlib import Path
from datetime import datetime


# ============================================================
# 1 · RUTAS
# ============================================================

APP_PATH = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard/app.py"
)

MASTER_PATH = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard/app_FINAL_FUNCIONAL.py"
)


print("=" * 88)
print("🌬️ BREATHE BARCELONA · 06.7 · MODEL INSIGHTS FINAL")
print("=" * 88)


# ============================================================
# 2 · COMPROBAR ARCHIVOS
# ============================================================

if not APP_PATH.exists():
    raise FileNotFoundError(
        f"❌ No existe app.py:\n{APP_PATH}"
    )

if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f"❌ No existe master:\n{MASTER_PATH}"
    )


app_code = APP_PATH.read_text(
    encoding="utf-8"
)

master_code = MASTER_PATH.read_text(
    encoding="utf-8"
)


compile(
    app_code,
    str(APP_PATH),
    "exec"
)

compile(
    master_code,
    str(MASTER_PATH),
    "exec"
)


print("✅ app.py localizado")
print("✅ master localizado")
print("✅ Sintaxis app.py correcta")
print("✅ Sintaxis master correcta")
print()

print(
    f"📄 app.py actual: "
    f"{len(app_code):,} caracteres"
)

print(
    f"📦 master final: "
    f"{len(master_code):,} caracteres"
)

print()


# ============================================================
# 3 · MARCADORES DE VISTA
# ============================================================

MODEL_MARK = 'elif selected == "Model Insights":'
PRED_MARK = 'elif selected == "Predictor":'


# ============================================================
# 4 · LOCALIZAR MODEL INSIGHTS ACTUAL
# ============================================================

app_model_start = app_code.find(
    MODEL_MARK
)

if app_model_start == -1:
    raise ValueError(
        "❌ No se localiza Model Insights en app.py"
    )


app_pred_start = app_code.find(
    PRED_MARK,
    app_model_start
)

if app_pred_start == -1:
    raise ValueError(
        "❌ No se localiza Predictor después de "
        "Model Insights en app.py"
    )


if app_pred_start <= app_model_start:
    raise ValueError(
        "❌ Orden incorrecto de vistas en app.py"
    )


current_model_block = app_code[
    app_model_start:app_pred_start
]


print(
    "✅ Model Insights actual localizada"
)

print(
    f"   Tamaño actual: "
    f"{len(current_model_block):,} caracteres"
)


if "Vista de modelos pendiente." in current_model_block:
    print(
        "✅ Placeholder actual detectado correctamente"
    )
else:
    print(
        "ℹ️ Model Insights actual no es placeholder; "
        "se sustituirá igualmente por la versión master"
    )

print()


# ============================================================
# 5 · LOCALIZAR MODEL INSIGHTS FINAL EN MASTER
# ============================================================

master_model_start = master_code.find(
    MODEL_MARK
)

if master_model_start == -1:
    raise ValueError(
        "❌ El master no contiene Model Insights"
    )


master_pred_start = master_code.find(
    PRED_MARK,
    master_model_start
)

if master_pred_start == -1:
    raise ValueError(
        "❌ El master no contiene Predictor después "
        "de Model Insights"
    )


if master_pred_start <= master_model_start:
    raise ValueError(
        "❌ Orden incorrecto de vistas en master"
    )


master_model_block = master_code[
    master_model_start:master_pred_start
]


print(
    "✅ Model Insights final localizada en master"
)

print(
    f"   Tamaño master: "
    f"{len(master_model_block):,} caracteres"
)

print()


# ============================================================
# 6 · VALIDAR QUE EL BLOQUE MASTER ES EL FINAL
# ============================================================

required_master = [
    "mi6-card",
    "mi6-main",
    "mi6-label",
    "mi6-value",
    "mi6-sub",
    "ACCURACY · TEST 2024",
    "F1 MACRO · TEST",
    "BALANCED ACCURACY",
    "LOSO · F1 MACRO",
    "Classification errors",
    "SHAP",
    "LOSO",
    "Performance by class",
    "Key insight",
    "st.plotly_chart",
    "st.columns",
]


print("🔍 VALIDACIÓN MODEL INSIGHTS MASTER")
print("-" * 88)

for marker in required_master:

    if marker not in master_model_block:
        raise ValueError(
            f"❌ Falta en el bloque master: {marker}"
        )

    print(
        f"✅ {marker}"
    )

print()


# ============================================================
# 7 · PROTEGER CONTRA BLOQUE DEMASIADO PEQUEÑO
# ============================================================

if len(master_model_block) < 10000:
    raise ValueError(
        "❌ El bloque Model Insights del master es "
        "anormalmente pequeño. No se escribe nada."
    )


print(
    "✅ Tamaño del bloque compatible con la sheet completa"
)

print()


# ============================================================
# 8 · CONSTRUIR NUEVO app.py
# ============================================================

new_code = (
    app_code[:app_model_start]
    + master_model_block
    + "\n\n"
    + app_code[app_pred_start:]
)


# ============================================================
# 9 · VALIDAR LAS CINCO VISTAS
# ============================================================

views = [
    'selected == "Overview"',
    'selected == "Temporal"',
    'selected == "Spatial"',
    'selected == "Model Insights"',
    'selected == "Predictor"',
]


for marker in views:

    if marker not in new_code:
        raise ValueError(
            f"❌ Se ha perdido una vista: {marker}"
        )


print("✅ Overview conservado")
print("✅ Temporal conservado")
print("✅ Spatial conservado")
print("✅ Model Insights presente")
print("✅ Predictor conservado")

print()


# ============================================================
# 10 · VALIDAR MODEL INSIGHTS RESULTANTE
# ============================================================

new_model_start = new_code.find(
    MODEL_MARK
)

new_pred_start = new_code.find(
    PRED_MARK,
    new_model_start
)

new_model_block = new_code[
    new_model_start:new_pred_start
]


for marker in required_master:

    if marker not in new_model_block:
        raise ValueError(
            f"❌ Tras reconstruir falta: {marker}"
        )


if "Vista de modelos pendiente." in new_model_block:
    raise ValueError(
        "❌ Sigue presente el placeholder "
        "'Vista de modelos pendiente.'"
    )


print(
    "✅ Placeholder eliminado"
)

print(
    f"✅ Model Insights resultante: "
    f"{len(new_model_block):,} caracteres"
)

print()


# ============================================================
# 11 · VALIDAR SINTAXIS ANTES DE ESCRIBIR
# ============================================================

compile(
    new_code,
    str(APP_PATH),
    "exec"
)


print(
    "✅ Sintaxis final validada ANTES de escribir"
)

print()


# ============================================================
# 12 · CREAR BACKUP DEL app.py ACTUAL
# ============================================================

marca = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

BACKUP_PATH = (
    APP_PATH.parent
    / f"app_backup_antes_067_model_insights_{marca}.py"
)


BACKUP_PATH.write_text(
    app_code,
    encoding="utf-8"
)


if (
    BACKUP_PATH.read_text(
        encoding="utf-8"
    )
    != app_code
):
    raise RuntimeError(
        "❌ El backup no coincide con el app.py original"
    )


print(
    f"✅ Backup creado: "
    f"{BACKUP_PATH.name}"
)

print()


# ============================================================
# 13 · ESCRIBIR app.py
# ============================================================

APP_PATH.write_text(
    new_code,
    encoding="utf-8"
)


# ============================================================
# 14 · VERIFICACIÓN POST-ESCRITURA
# ============================================================

final_code = APP_PATH.read_text(
    encoding="utf-8"
)


compile(
    final_code,
    str(APP_PATH),
    "exec"
)


final_model_start = final_code.find(
    MODEL_MARK
)

final_pred_start = final_code.find(
    PRED_MARK,
    final_model_start
)

if final_model_start == -1:
    raise RuntimeError(
        "❌ Model Insights desapareció tras escribir"
    )

if final_pred_start == -1:
    raise RuntimeError(
        "❌ Predictor desapareció tras escribir"
    )


final_model_block = final_code[
    final_model_start:final_pred_start
]


for marker in required_master:

    if marker not in final_model_block:
        raise RuntimeError(
            f"❌ Verificación final: falta {marker}"
        )


if "Vista de modelos pendiente." in final_model_block:
    raise RuntimeError(
        "❌ Verificación final: "
        "placeholder todavía presente"
    )


for marker in views:

    if marker not in final_code:
        raise RuntimeError(
            f"❌ Verificación final: "
            f"falta vista {marker}"
        )


# ============================================================
# 15 · RESULTADO
# ============================================================

print("=" * 88)
print("🌬️ 06.7 · MODEL INSIGHTS RECUPERADA CORRECTAMENTE")
print("=" * 88)
print()

print("📊 MODEL INSIGHTS")
print(
    f"✅ Bloque final: "
    f"{len(final_model_block):,} caracteres"
)
print("✅ KPI V6 finales")
print("✅ Accuracy")
print("✅ F1 Macro")
print("✅ Balanced Accuracy")
print("✅ LOSO F1 Macro")
print("✅ Classification errors")
print("✅ SHAP")
print("✅ LOSO")
print("✅ Performance by class")
print("✅ Key insight")
print()

print("🔒 INTEGRIDAD")
print("✅ Overview intacto")
print("✅ Temporal intacto")
print("✅ Spatial intacto")
print("✅ Predictor intacto")
print("✅ Sintaxis Python correcta")
print()

print(
    "📦 Backup:",
    BACKUP_PATH.name
)

print()
print(
    "👉 06.7 CERRADA."
)
print(
    "👉 Ahora pásame esta salida antes de ejecutar 06.8."
)
print("=" * 88)

🌬️ BREATHE BARCELONA · 06.7 · MODEL INSIGHTS FINAL
✅ app.py localizado
✅ master localizado
✅ Sintaxis app.py correcta
✅ Sintaxis master correcta

📄 app.py actual: 113,430 caracteres
📦 master final: 308,719 caracteres

✅ Model Insights actual localizada
   Tamaño actual: 144 caracteres
✅ Placeholder actual detectado correctamente

✅ Model Insights final localizada en master
   Tamaño master: 39,785 caracteres

🔍 VALIDACIÓN MODEL INSIGHTS MASTER
----------------------------------------------------------------------------------------
✅ mi6-card
✅ mi6-main
✅ mi6-label
✅ mi6-value
✅ mi6-sub
✅ ACCURACY · TEST 2024
✅ F1 MACRO · TEST
✅ BALANCED ACCURACY
✅ LOSO · F1 MACRO
✅ Classification errors
✅ SHAP
✅ LOSO
✅ Performance by class
✅ Key insight
✅ st.plotly_chart
✅ st.columns

✅ Tamaño del bloque compatible con la sheet completa

✅ Overview conservado
✅ Temporal conservado
✅ Spatial conservado
✅ Model Insights presente
✅ Predictor conservado

✅ Placeholder eliminado
✅ Model Insights resultante:

### Resultados

La vista **Model Insights** queda integrada como módulo específico de evaluación e interpretabilidad del modelo, incorporando:

* indicadores globales mediante **Accuracy, F1 Macro y Balanced Accuracy**;
* análisis de los **errores de clasificación** y del rendimiento diferenciado por clase;
* validación **LOSO**, orientada a evaluar la capacidad de generalización entre estaciones;
* análisis **SHAP**, basado en la importancia media absoluta de las variables;
* una síntesis final de los principales resultados mediante **Key insight**.

La combinación de métricas globales y balanceadas permite interpretar el rendimiento considerando la distribución desigual de las clases, mientras que LOSO y SHAP amplían la evaluación hacia la **generalización espacial** y la **interpretabilidad del modelo**. La validación final confirma además que la incorporación de esta vista conserva intactos **Overview, Temporal, Spatial y Predictor** y mantiene una sintaxis válida de la aplicación.

> **Resultado:** se obtiene una vista de diagnóstico del modelo que reúne rendimiento predictivo, comportamiento por clase, generalización entre estaciones e importancia de variables en una única capa interpretativa.


### 6.8 · Construcción del Dual Predictor

La vista **Predictor** constituye la capa de simulación interactiva de *Breathe Barcelona* y combina dos ámbitos complementarios de predicción: un módulo **espacial**, orientado a la estimación de NO₂ a partir de características urbanas, y un módulo **temporal**, destinado a evaluar escenarios de calidad del aire.

El **Spatial Predictor** permite modificar las variables de entrada mediante controles interactivos y obtener una estimación de NO₂ dentro del dominio empírico definido durante la preparación de los datos. El **Temporal Predictor** incorpora la clasificación de escenarios en **tres categorías de calidad del aire**, mostrando tanto la clase estimada como las probabilidades asociadas.

La interfaz mantiene una identidad visual común mediante una escala fría **cobalto–cian–turquesa**, aplicada a controles y elementos interactivos, y conserva de forma independiente las restantes vistas del dashboard.

> **Objetivo:** transformar los modelos desarrollados en herramientas interactivas que permitan explorar estimaciones espaciales de NO₂ y escenarios temporales de calidad del aire.


In [34]:
# ============================================================
# 🌬️ BREATHE BARCELONA
# 06.8 · DUAL PREDICTOR — FINAL AUTOSUFICIENTE
# PREDICTOR + SCOPE V14 + THEME STREAMLIT CYAN
# ============================================================
#
# OBJETIVO:
#   1. Recuperar Predictor completo desde master
#   2. Mantener Scope V14 final
#   3. Mantener sliders / gráficos / predicciones / modelos
#   4. Fijar el primaryColor nativo de Streamlit en cyan
#      para evitar el coral por defecto en los sliders
#
# FLUJO FINAL:
#   06.7 → 06.8 → 06.9 → 06.10 → 06.11
#
# NO TOCA:
#   - Overview
#   - Temporal
#   - Spatial
#   - Model Insights
#   - navegación
#   - router
#   - modelos
#   - inputs
#   - predicciones
#   - probabilidades
#   - textos
#   - gráficos
#
# ============================================================

from pathlib import Path
from datetime import datetime


# ============================================================
# 1 · RUTAS
# ============================================================

APP_PATH = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard/app.py"
)

MASTER_PATH = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard/app_FINAL_FUNCIONAL.py"
)

STREAMLIT_CONFIG_DIR = Path(
    "/content/.streamlit"
)

STREAMLIT_CONFIG_PATH = (
    STREAMLIT_CONFIG_DIR
    / "config.toml"
)


print("=" * 96)
print("🌬️ BREATHE BARCELONA · 06.8 · DUAL PREDICTOR FINAL")
print("=" * 96)


# ============================================================
# 2 · COMPROBAR ARCHIVOS
# ============================================================

if not APP_PATH.exists():
    raise FileNotFoundError(
        f"❌ No existe app.py:\n{APP_PATH}"
    )

if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f"❌ No existe master:\n{MASTER_PATH}"
    )


app_code = APP_PATH.read_text(
    encoding="utf-8"
)

master_code = MASTER_PATH.read_text(
    encoding="utf-8"
)


compile(
    app_code,
    str(APP_PATH),
    "exec"
)

compile(
    master_code,
    str(MASTER_PATH),
    "exec"
)


print("✅ app.py localizado")
print("✅ master localizado")
print("✅ Sintaxis app.py correcta")
print("✅ Sintaxis master correcta")
print()

print(
    f"📄 app.py actual: "
    f"{len(app_code):,} caracteres"
)

print(
    f"📦 master final: "
    f"{len(master_code):,} caracteres"
)

print()


# ============================================================
# 3 · MARCADOR PREDICTOR
# ============================================================

PRED_MARK = 'elif selected == "Predictor":'


# ============================================================
# 4 · LOCALIZAR PREDICTOR ACTUAL
# ============================================================

app_pred_start = app_code.find(
    PRED_MARK
)

if app_pred_start == -1:
    raise ValueError(
        "❌ No se localiza Predictor en app.py"
    )


current_predictor = app_code[
    app_pred_start:
]


print("✅ Predictor actual localizado")
print(
    f"   Tamaño actual: "
    f"{len(current_predictor):,} caracteres"
)

if (
    "Predictor pendiente." in current_predictor
    or len(current_predictor) < 1000
):
    print(
        "✅ Placeholder actual detectado"
    )
else:
    print(
        "ℹ️ Predictor ya existe; "
        "se reinstalará desde master"
    )

print()


# ============================================================
# 5 · RECUPERAR PREDICTOR FINAL DESDE MASTER
# ============================================================

master_pred_start = master_code.find(
    PRED_MARK
)

if master_pred_start == -1:
    raise ValueError(
        "❌ El master no contiene Predictor"
    )


master_predictor = master_code[
    master_pred_start:
]


print(
    "✅ Predictor final localizado en master"
)

print(
    f"   Tamaño master: "
    f"{len(master_predictor):,} caracteres"
)

print()


# ============================================================
# 6 · VALIDAR DUAL PREDICTOR
# ============================================================

required_master = [
    "Model scope",
    "Temporal scope",
    "3-class classification",
    "Scenario classification",
    "Spatial",
    "Temporal",
    "NO₂",
    "st.slider",
    "st.plotly_chart",
    "predict",
]


print("🔍 VALIDACIÓN DUAL PREDICTOR")
print("-" * 96)

for marker in required_master:

    if marker not in master_predictor:
        raise ValueError(
            f"❌ Falta contenido: {marker}"
        )

    print(
        f"✅ {marker}"
    )

print()


# ============================================================
# 7 · VALIDAR SCOPE V14
# ============================================================

v14_markers = [
    "BB_SCOPE_V14_START",
    ".pr-note",
    "scope-accent",
    "pr-badge",
]


print("🎨 VALIDACIÓN SCOPE V14")
print("-" * 96)

for marker in v14_markers:

    if marker not in master_predictor:
        raise ValueError(
            f"❌ Falta V14: {marker}"
        )

    print(
        f"✅ {marker}"
    )

print()


# ============================================================
# 8 · PROTECCIÓN DE TAMAÑO
# ============================================================

if len(master_predictor) < 50000:
    raise ValueError(
        "❌ Predictor master anormalmente pequeño. "
        "No se escribe nada."
    )


print(
    "✅ Tamaño compatible con Dual Predictor completo"
)

print()


# ============================================================
# 9 · VALIDAR PALETA FRÍA DEL PREDICTOR
# ============================================================

slider_palette_markers = [
    "#315CCE",
    "#3978DE",
    "#3299D4",
    "#24BCD2",
    "#43D2C1",
    "#67DECB",
]


print("🎚️ VALIDACIÓN PALETA SLIDERS")
print("-" * 96)

for marker in slider_palette_markers:

    if marker not in master_predictor:
        raise ValueError(
            f"❌ Falta color de identidad: {marker}"
        )

    print(
        f"✅ {marker}"
    )

print()

print(
    "✅ Gradiente cobalto → cian → turquesa presente"
)

print()


# ============================================================
# 10 · RECONSTRUIR app.py
# ============================================================

prefix = app_code[
    :app_pred_start
]


new_code = (
    prefix
    + master_predictor
)


# ============================================================
# 11 · VALIDAR CINCO VISTAS
# ============================================================

views = [
    'selected == "Overview"',
    'selected == "Temporal"',
    'selected == "Spatial"',
    'selected == "Model Insights"',
    'selected == "Predictor"',
]


for marker in views:

    if marker not in new_code:
        raise ValueError(
            f"❌ Se ha perdido una vista: {marker}"
        )


print("✅ Overview conservado")
print("✅ Temporal conservado")
print("✅ Spatial conservado")
print("✅ Model Insights conservado")
print("✅ Predictor presente")

print()


# ============================================================
# 12 · VALIDAR MODEL INSIGHTS
# ============================================================

model_markers = [
    "ACCURACY · TEST 2024",
    "F1 MACRO · TEST",
    "BALANCED ACCURACY",
    "LOSO · F1 MACRO",
    "Classification errors",
    "SHAP",
    "Performance by class",
    "Key insight",
]


for marker in model_markers:

    if marker not in new_code:
        raise ValueError(
            f"❌ Model Insights se ha alterado: {marker}"
        )


print(
    "✅ Model Insights final sigue intacto"
)

print()


# ============================================================
# 13 · VALIDAR PREDICTOR RESULTANTE
# ============================================================

new_pred_start = new_code.find(
    PRED_MARK
)

new_predictor = new_code[
    new_pred_start:
]


for marker in required_master:

    if marker not in new_predictor:
        raise ValueError(
            f"❌ Predictor resultante sin: {marker}"
        )


for marker in v14_markers:

    if marker not in new_predictor:
        raise ValueError(
            f"❌ Predictor resultante sin V14: {marker}"
        )


for marker in slider_palette_markers:

    if marker not in new_predictor:
        raise ValueError(
            f"❌ Predictor resultante sin color: {marker}"
        )


if "Predictor pendiente." in new_predictor:
    raise ValueError(
        "❌ Sigue presente Predictor pendiente"
    )


print(
    f"✅ Predictor resultante: "
    f"{len(new_predictor):,} caracteres"
)

print("✅ Scope V14 conservado")
print("✅ Paleta fría conservada")
print()


# ============================================================
# 14 · VALIDAR SINTAXIS ANTES DE ESCRIBIR
# ============================================================

compile(
    new_code,
    str(APP_PATH),
    "exec"
)


print(
    "✅ Sintaxis final validada ANTES de escribir"
)

print()


# ============================================================
# 15 · BACKUP DE app.py
# ============================================================

stamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

BACKUP_PATH = (
    APP_PATH.parent
    / f"app_backup_antes_068_predictor_final_{stamp}.py"
)


BACKUP_PATH.write_text(
    app_code,
    encoding="utf-8"
)


if (
    BACKUP_PATH.read_text(
        encoding="utf-8"
    )
    != app_code
):
    raise RuntimeError(
        "❌ El backup no coincide con app.py original"
    )


print(
    "✅ Backup app.py:",
    BACKUP_PATH.name
)

print()


# ============================================================
# 16 · ESCRIBIR app.py
# ============================================================

APP_PATH.write_text(
    new_code,
    encoding="utf-8"
)


# ============================================================
# 17 · CREAR CONFIG STREAMLIT
# ============================================================
#
# IMPORTANTE:
# El Predictor ya contiene su gradiente frío.
# Esto fija además el primaryColor NATIVO de Streamlit
# para impedir que BaseWeb vuelva al coral por defecto.
# ============================================================

STREAMLIT_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CONFIG_CONTENT = """[theme]
primaryColor = "#24BCD2"
"""


STREAMLIT_CONFIG_PATH.write_text(
    CONFIG_CONTENT,
    encoding="utf-8"
)


if not STREAMLIT_CONFIG_PATH.exists():
    raise RuntimeError(
        "❌ No se ha creado config.toml"
    )


saved_config = STREAMLIT_CONFIG_PATH.read_text(
    encoding="utf-8"
)


if saved_config != CONFIG_CONTENT:
    raise RuntimeError(
        "❌ El config.toml escrito no coincide"
    )


if 'primaryColor = "#24BCD2"' not in saved_config:
    raise RuntimeError(
        "❌ primaryColor cyan no localizado"
    )


print(
    "✅ Streamlit theme creado"
)

print(
    "✅ primaryColor = #24BCD2"
)

print()


# ============================================================
# 18 · VERIFICACIÓN POST-ESCRITURA
# ============================================================

final_code = APP_PATH.read_text(
    encoding="utf-8"
)


compile(
    final_code,
    str(APP_PATH),
    "exec"
)


final_pred_start = final_code.find(
    PRED_MARK
)

if final_pred_start == -1:
    raise RuntimeError(
        "❌ Predictor desapareció tras escribir"
    )


final_predictor = final_code[
    final_pred_start:
]


for marker in required_master:

    if marker not in final_predictor:
        raise RuntimeError(
            f"❌ Verificación final: falta {marker}"
        )


for marker in v14_markers:

    if marker not in final_predictor:
        raise RuntimeError(
            f"❌ Verificación final V14: falta {marker}"
        )


for marker in slider_palette_markers:

    if marker not in final_predictor:
        raise RuntimeError(
            f"❌ Verificación final slider: falta {marker}"
        )


for marker in views:

    if marker not in final_code:
        raise RuntimeError(
            f"❌ Verificación final: falta {marker}"
        )


for marker in model_markers:

    if marker not in final_code:
        raise RuntimeError(
            f"❌ Model Insights final: falta {marker}"
        )


if "Predictor pendiente." in final_predictor:
    raise RuntimeError(
        "❌ Placeholder Predictor todavía presente"
    )


if not STREAMLIT_CONFIG_PATH.exists():
    raise RuntimeError(
        "❌ config.toml no existe tras escritura"
    )


final_config = STREAMLIT_CONFIG_PATH.read_text(
    encoding="utf-8"
)


if 'primaryColor = "#24BCD2"' not in final_config:
    raise RuntimeError(
        "❌ primaryColor final incorrecto"
    )


# ============================================================
# 19 · RESULTADO
# ============================================================

print("=" * 96)
print("🌬️ 06.8 · DUAL PREDICTOR FINAL INSTALADO")
print("=" * 96)
print()

print("🤖 PREDICTOR")
print(
    f"✅ Bloque final: "
    f"{len(final_predictor):,} caracteres"
)
print("✅ Spatial · NO₂")
print("✅ Temporal · Air Quality")
print("✅ Model scope")
print("✅ Temporal scope")
print("✅ Scenario classification")
print("✅ 3-class classification")
print("✅ Sliders")
print("✅ Gráficos")
print("✅ Predicciones")
print("✅ Probabilidades")
print()

print("🎨 IDENTIDAD VISUAL")
print("✅ Scope V14 conservado")
print("✅ Gradiente cobalto → cian → turquesa")
print("✅ #315CCE")
print("✅ #3978DE")
print("✅ #3299D4")
print("✅ #24BCD2")
print("✅ #43D2C1")
print("✅ #67DECB")
print()

print("🎚️ STREAMLIT THEME")
print("✅ config.toml creado desde 06.8")
print("✅ primaryColor = #24BCD2")
print("✅ Coral nativo de Streamlit desactivado")
print()

print("🔒 INTEGRIDAD")
print("✅ Overview intacto")
print("✅ Temporal intacto")
print("✅ Spatial intacto")
print("✅ Model Insights intacto")
print("✅ Predictor completo")
print("✅ Sintaxis Python correcta")
print()

print(
    "📦 Backup:",
    BACKUP_PATH.name
)

print()

print(
    "⚠️ Para que Streamlit lea el nuevo tema,"
)
print(
    "   el servidor debe arrancarse DESPUÉS de esta 06.8."
)

print()

print(
    "👉 Sigue con 06.9."
)
print(
    "👉 Después ejecuta 06.10 y 06.11."
)

print("=" * 96)

🌬️ BREATHE BARCELONA · 06.8 · DUAL PREDICTOR FINAL
✅ app.py localizado
✅ master localizado
✅ Sintaxis app.py correcta
✅ Sintaxis master correcta

📄 app.py actual: 153,073 caracteres
📦 master final: 308,719 caracteres

✅ Predictor actual localizado
   Tamaño actual: 125 caracteres
✅ Placeholder actual detectado

✅ Predictor final localizado en master
   Tamaño master: 140,932 caracteres

🔍 VALIDACIÓN DUAL PREDICTOR
------------------------------------------------------------------------------------------------
✅ Model scope
✅ Temporal scope
✅ 3-class classification
✅ Scenario classification
✅ Spatial
✅ Temporal
✅ NO₂
✅ st.slider
✅ st.plotly_chart
✅ predict

🎨 VALIDACIÓN SCOPE V14
------------------------------------------------------------------------------------------------
✅ BB_SCOPE_V14_START
✅ .pr-note
✅ scope-accent
✅ pr-badge

✅ Tamaño compatible con Dual Predictor completo

🎚️ VALIDACIÓN PALETA SLIDERS
------------------------------------------------------------------------------

### Resultados

La vista **Predictor** queda integrada como un módulo dual de simulación y explotación de los modelos desarrollados:

* **Spatial · NO₂:** estimación interactiva de la concentración de NO₂ a partir de las variables territoriales empleadas por el modelo geoespacial.
* **Temporal · Air Quality:** clasificación de escenarios temporales en **tres categorías**, incorporando la predicción obtenida y sus probabilidades asociadas.
* **Interactividad:** utilización de *sliders*, gráficos y controles de escenario para modificar las condiciones de entrada y observar dinámicamente la respuesta de los modelos.
* **Coherencia visual:** aplicación de una escala cobalto–cian–turquesa y configuración del color primario de Streamlit en `#24BCD2`, manteniendo una identidad gráfica homogénea con el resto de *Breathe Barcelona*.

La validación final confirma que el Predictor completo se incorpora sin alterar **Overview, Temporal, Spatial ni Model Insights**, manteniendo además una sintaxis válida de la aplicación.

> **Resultado:** se obtiene una capa predictiva interactiva que permite trasladar los modelos espacial y temporal desde el análisis técnico hacia la exploración de escenarios, completando el dashboard con una funcionalidad orientada a la simulación.


### 6.9 · Construcción de la Home / Landing

Como capa final de acceso a *Breathe Barcelona*, se incorpora una **Home / Landing** que funciona como portada del producto y punto de entrada a las cinco vistas analíticas del dashboard.

La portada sintetiza visualmente el flujo completo del proyecto —**Open Data → Space + Time → Models → Product**— y presenta el periodo de estudio **2019–2024**, las principales técnicas empleadas y accesos directos a **Overview, Temporal, Spatial, Model Insights y Predictor**. De este modo, el dashboard adquiere una estructura narrativa que conecta las fuentes de información, el análisis, la modelización y su explotación interactiva.

La navegación se amplía con un acceso específico a la portada, manteniendo independientes e intactos los contenidos y funcionalidades de las cinco vistas analíticas.

> **Objetivo:** proporcionar una entrada clara y coherente al dashboard que comunique el flujo metodológico del proyecto y facilite el acceso directo a sus diferentes niveles de análisis.


In [35]:
# ============================================================
# 🌬️ BREATHE BARCELONA
# 06.9 · HOME / LANDING — FINAL AUTOSUFICIENTE E IDEMPOTENTE
# ============================================================
#
# OBJETIVO:
#   - HOME como entrada inicial
#   - Sin cabecera analítica en HOME
#   - Sin navegación duplicada
#   - 🌬️ como marca original
#   - BREATHE / BARCELONA
#   - OPEN DATA → SPACE + TIME → MODELS → PRODUCT
#   - Cinco accesos inferiores
#   - ⊙ = HOME dentro de las sheets
#
# COMPORTAMIENTO:
#   A) Si HOME final NO existe → la instala
#   B) Si HOME final YA existe → la valida y NO la duplica
#   C) Si detecta instalación parcial → se detiene
#
# NO TOCA:
#   - contenido de Overview
#   - Temporal
#   - Spatial
#   - Model Insights
#   - Predictor
#   - modelos
#   - datos
#   - Streamlit theme de 06.8
#
# ============================================================

from pathlib import Path
from datetime import datetime
import re


# ============================================================
# 1 · RUTA
# ============================================================

APP_PATH = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard/app.py"
)


print("=" * 96)
print("🌬️ BREATHE BARCELONA · 06.9 HOME / LANDING — FINAL")
print("=" * 96)


# ============================================================
# 2 · LEER APP
# ============================================================

if not APP_PATH.exists():

    raise FileNotFoundError(
        f"❌ No existe:\n{APP_PATH}"
    )


pre_069_code = APP_PATH.read_text(
    encoding="utf-8"
)


compile(
    pre_069_code,
    str(APP_PATH),
    "exec"
)


print("✅ app.py localizado")
print("✅ Sintaxis Python correcta")
print(
    f"✅ Tamaño actual: "
    f"{len(pre_069_code):,} caracteres"
)

print()


# ============================================================
# 3 · VALIDAR LAS 5 VISTAS ANALÍTICAS
# ============================================================

VIEWS = {
    "Overview":
        'selected == "Overview"',

    "Temporal":
        'selected == "Temporal"',

    "Spatial":
        'selected == "Spatial"',

    "Model Insights":
        'selected == "Model Insights"',

    "Predictor":
        'selected == "Predictor"',
}


for name, marker in VIEWS.items():

    if marker not in pre_069_code:

        raise RuntimeError(
            f"❌ Falta {name}."
        )

    print(
        f"✅ {name} localizada"
    )


for marker in [
    "mi-kpi",
    "SHAP",
    "LOSO",
]:

    if marker not in pre_069_code:

        raise RuntimeError(
            f"❌ Falta contenido crítico: {marker}"
        )


print(
    "✅ Estado POST-06.8 confirmado"
)

print()


# ============================================================
# 4 · DETECTAR ESTADO DE HOME
# ============================================================

HOME_SIGNATURES = [
    'selected == "PORTADA"',
    "BREATHE BARCELONA · HOME ROUTER",
    "_bb_home_initialized",
    "_bb_route_override",
    "bb-home-marker",
    "bb-home-face",
    "?bb_view=Overview",
    "?bb_view=Temporal",
    "?bb_view=Spatial",
    "?bb_view=Model%20Insights",
    "?bb_view=Predictor",
]


home_presence = {
    marker:
        marker in pre_069_code
    for marker in HOME_SIGNATURES
}


home_count = sum(
    home_presence.values()
)


HOME_ALREADY_COMPLETE = (
    home_count
    == len(HOME_SIGNATURES)
)


HOME_NOT_PRESENT = (
    home_count
    == 0
)


print("🏠 ESTADO HOME")
print("-" * 96)

print(
    f"Marcadores encontrados: "
    f"{home_count}/{len(HOME_SIGNATURES)}"
)


if HOME_ALREADY_COMPLETE:

    print(
        "✅ HOME final ya está instalada"
    )

elif HOME_NOT_PRESENT:

    print(
        "✅ HOME todavía no está instalada"
    )

else:

    print()

    for marker, present in home_presence.items():

        print(
            ("✅ " if present else "❌ ")
            + marker
        )

    raise RuntimeError(
        "❌ Se detectó una instalación PARCIAL de HOME. "
        "No se modifica app.py."
    )


print()


# ============================================================
# 5 · SI HOME YA ESTÁ → VALIDAR Y TERMINAR
# ============================================================

if HOME_ALREADY_COMPLETE:

    code_existing = pre_069_code


    # --------------------------------------------------------
    # 5.1 · Debe existir exactamente una HOME
    # --------------------------------------------------------

    if code_existing.count(
        'selected == "PORTADA"'
    ) != 1:

        raise RuntimeError(
            "❌ Existe más de una rama PORTADA."
        )


    if code_existing.count(
        "BREATHE BARCELONA · HOME ROUTER"
    ) != 1:

        raise RuntimeError(
            "❌ Existe más de un HOME ROUTER."
        )


    if code_existing.count(
        "bb-home-marker"
    ) < 2:

        raise RuntimeError(
            "❌ HOME parece incompleta."
        )


    # --------------------------------------------------------
    # 5.2 · Navegación HOME
    # --------------------------------------------------------

    if '"⊙"' not in code_existing:

        raise RuntimeError(
            "❌ Falta ⊙ en navegación."
        )


    # --------------------------------------------------------
    # 5.3 · Contenido editorial
    # --------------------------------------------------------

    REQUIRED_EXISTING = [
        "URBAN AIR QUALITY · BARCELONA",
        "BREATHE",
        "BARCELONA",
        "Urban Air Quality Intelligence",
        "2019—2024",
        "OPEN DATA",
        "SPACE + TIME",
        "MODELS",
        "PRODUCT",
        "TIME SERIES",
        "SPATIAL ANALYTICS",
        "MACHINE LEARNING",
        "INTERPRETABILITY",
        "SARIMAX",
        "XGBoost",
        "SHAP",
        "About Breathe Barcelona",
    ]


    for marker in REQUIRED_EXISTING:

        if marker not in code_existing:

            raise RuntimeError(
                f"❌ HOME instalada pero falta: {marker}"
            )


    # --------------------------------------------------------
    # 5.4 · Mantener vistas
    # --------------------------------------------------------

    for name, marker in VIEWS.items():

        if marker not in code_existing:

            raise RuntimeError(
                f"❌ Se perdió {name}."
            )


    # --------------------------------------------------------
    # 5.5 · Sintaxis
    # --------------------------------------------------------

    compile(
        code_existing,
        str(APP_PATH),
        "exec"
    )


    print("=" * 96)
    print("🌬️ 06.9 · HOME YA INSTALADA Y VALIDADA")
    print("=" * 96)
    print()

    print("✅ No se ha modificado app.py")
    print("✅ No se ha duplicado PORTADA")
    print("✅ No se ha duplicado router")
    print("✅ ⊙ = HOME")
    print("✅ Cinco accesos presentes")
    print("✅ Overview intacto")
    print("✅ Temporal intacto")
    print("✅ Spatial intacto")
    print("✅ Model Insights intacto")
    print("✅ Predictor intacto")
    print("✅ SHAP intacto")
    print("✅ LOSO intacto")
    print("✅ Sintaxis correcta")
    print()

    print(
        "🎯 06.9 COMPLETADA · NO ERA NECESARIO REINSTALAR"
    )

    print("=" * 96)


# ============================================================
# 6 · SI HOME NO EXISTE → INSTALAR
# ============================================================

else:

    code = pre_069_code


    # ========================================================
    # 6.1 · LOCALIZAR NAVEGACIÓN ORIGINAL
    # ========================================================

    first_overview = code.find(
        'selected == "Overview"'
    )


    if first_overview == -1:

        raise RuntimeError(
            "❌ No se localizó Overview."
        )


    header_zone = code[
        :first_overview
    ]


    NAV_PATTERN = re.compile(
        r'\[\s*'
        r'["\']Overview["\']\s*,\s*'
        r'["\']Temporal["\']\s*,\s*'
        r'["\']Spatial["\']\s*,\s*'
        r'["\']Model Insights["\']\s*,\s*'
        r'["\']Predictor["\']\s*'
        r'\]',
        flags=re.S
    )


    nav_match = NAV_PATTERN.search(
        header_zone
    )


    if nav_match is None:

        raise RuntimeError(
            "❌ No se localizó la navegación original."
        )


    NEW_NAV = (
        '["⊙", "Overview", "Temporal", '
        '"Spatial", "Model Insights", "Predictor"]'
    )


    code = (
        code[:nav_match.start()]
        + NEW_NAV
        + code[nav_match.end():]
    )


    print(
        "✅ ⊙ incorporado como HOME"
    )


    # ========================================================
    # 6.2 · OVERVIEW → ELIF
    # ========================================================

    overview_if = (
        'if selected == "Overview":'
    )

    overview_elif = (
        'elif selected == "Overview":'
    )


    overview_start = code.find(
        overview_if
    )


    if overview_start != -1:

        code = (
            code[:overview_start]
            + code[overview_start:].replace(
                overview_if,
                overview_elif,
                1
            )
        )


    overview_start = code.find(
        overview_elif
    )


    if overview_start == -1:

        raise RuntimeError(
            "❌ No se pudo preparar Overview."
        )


    print(
        "✅ Punto de inserción localizado"
    )


    # ========================================================
    # 6.3 · ROUTER
    # ========================================================

    ROUTER_BLOCK = r'''
# ============================================================
# BREATHE BARCELONA · HOME ROUTER
# ============================================================

_bb_nav_selected = selected


if _bb_nav_selected == "⊙":

    _bb_nav_page = "PORTADA"

else:

    _bb_nav_page = _bb_nav_selected


try:

    _bb_requested_view = st.query_params.get(
        "bb_view",
        None
    )

except Exception:

    _bb_requested_view = None


_bb_valid_views = [
    "PORTADA",
    "Overview",
    "Temporal",
    "Spatial",
    "Model Insights",
    "Predictor",
]


if _bb_requested_view in _bb_valid_views:

    st.session_state[
        "_bb_route_override"
    ] = _bb_requested_view

    try:

        del st.query_params[
            "bb_view"
        ]

    except Exception:

        pass


elif _bb_nav_page != "PORTADA":

    st.session_state.pop(
        "_bb_route_override",
        None
    )


_bb_override = st.session_state.get(
    "_bb_route_override",
    None
)


if "_bb_home_initialized" not in st.session_state:

    st.session_state[
        "_bb_home_initialized"
    ] = True

    if _bb_requested_view in _bb_valid_views:

        selected = _bb_requested_view

    else:

        selected = "PORTADA"


elif _bb_override in _bb_valid_views:

    selected = _bb_override


else:

    selected = _bb_nav_page


'''


    # ========================================================
    # 6.4 · HOME
    # ========================================================

    HOME_BLOCK = r'''
if selected == "PORTADA":

    st.markdown(
        '<div class="bb-home-marker"></div>',
        unsafe_allow_html=True
    )


    _bb_css = """
<style>

/* =========================================================
   HOME · BREATHE BARCELONA
========================================================= */


/* OCULTAR CABECERA ANALÍTICA EN HOME */

body:has(.bb-home-marker)
div[data-testid="stHorizontalBlock"]:has(button) {
    display: none !important;
}

body:has(.bb-home-marker)
div[data-testid="stHorizontalBlock"]:has(.bb-face) {
    display: none !important;
}


/* CONTENEDOR */

body:has(.bb-home-marker)
div[data-testid="stMainBlockContainer"] {
    padding-top: 3.0rem !important;
    padding-bottom: 1.4rem !important;
}


/* FONDO */

body:has(.bb-home-marker)
[data-testid="stAppViewContainer"] {

    background:

        radial-gradient(
            circle at 88% 15%,
            rgba(47,225,210,.040),
            transparent 25%
        ),

        linear-gradient(
            180deg,
            #061A21 0%,
            #071C23 100%
        );
}


/* BRANDLINE */

.bb-brandline {

    display: flex;

    align-items: center;

    justify-content: flex-start;

    gap: .80rem;

    margin:
        0 0 1.35rem 0;

    padding: 0;

    width: 100%;

    text-align: left;
}


.bb-home-face {

    display: flex;

    align-items: center;

    justify-content: center;

    width: 46px;

    height: 46px;

    flex:
        0 0 46px;

    font-size: 2.15rem;

    line-height: 1;

    filter:
        drop-shadow(
            0 0 10px
            rgba(47,225,210,.12)
        );
}


.bb-kicker {

    margin: 0;

    padding: 0;

    color:
        #46E4D6;

    font-size:
        .78rem;

    line-height: 1;

    font-weight: 800;

    letter-spacing:
        .17em;

    text-align: left;
}


/* TITULAR */

.bb-breathe {

    margin: 0;

    color:
        #F3F8F8;

    font-size:
        clamp(
            3.8rem,
            5vw,
            5.35rem
        );

    line-height:
        .90;

    font-weight:
        850;

    letter-spacing:
        -.052em;

    text-align:
        left;
}


.bb-barcelona {

    margin:
        1.20rem 0 0 0;

    color:
        #32E1D3;

    font-size:
        clamp(
            3.8rem,
            5vw,
            5.35rem
        );

    line-height:
        .90;

    font-weight:
        850;

    letter-spacing:
        -.052em;

    text-align:
        left;
}


/* SUBTÍTULO */

.bb-subtitle {

    margin-top:
        1.35rem;

    color:
        #DDE8E9;

    font-size:
        1.18rem;

    line-height:
        1.2;

    font-weight:
        550;

    text-align:
        left;
}


/* META */

.bb-meta {

    margin-top:
        .60rem;

    color:
        #78989E;

    font-size:
        .72rem;

    line-height:
        1.45;

    font-weight:
        700;

    letter-spacing:
        .075em;

    text-align:
        left;
}


.bb-period {

    color:
        #43E2D5;

    font-weight:
        820;
}


/* REGLA */

.bb-rule {

    width:
        100%;

    height:
        1px;

    margin:
        1.30rem 0
        1.05rem 0;

    background:
        rgba(
            50,
            225,
            210,
            .14
        );
}


/* PIPELINE */

.bb-pipeline {

    display:
        grid;

    grid-template-columns:
        minmax(0,1fr) 34px
        minmax(0,1fr) 34px
        minmax(0,1fr) 34px
        minmax(0,1fr);

    width:
        100%;

    align-items:
        start;
}


.bb-stage {

    min-width:
        0;

    text-align:
        left;
}


.bb-stage-num {

    margin-bottom:
        .38rem;

    color:
        #52757B;

    font-size:
        .68rem;

    font-weight:
        800;

    letter-spacing:
        .10em;
}


.bb-stage-title {

    margin-bottom:
        .48rem;

    color:
        #43E3D6;

    font-size:
        .82rem;

    font-weight:
        820;

    letter-spacing:
        .07em;
}


.bb-stage-copy {

    color:
        #829DA2;

    font-size:
        .72rem;

    line-height:
        1.55;

    font-weight:
        500;
}


.bb-arrow {

    padding-top:
        1.55rem;

    color:
        #315C63;

    font-size:
        1.15rem;

    text-align:
        center;
}


/* EXPLORE */

.bb-explore {

    margin-bottom:
        .65rem;

    color:
        #78989E;

    font-size:
        .72rem;

    font-weight:
        800;

    letter-spacing:
        .13em;

    text-align:
        left;
}


/* LINKS */

.bb-links {

    display:
        grid;

    grid-template-columns:
        repeat(
            5,
            minmax(0,1fr)
        );

    gap:
        .72rem;

    width:
        100%;
}


.bb-link {

    display:
        block;

    min-width:
        0;

    padding:
        .82rem
        .78rem
        .75rem
        .78rem;

    border:
        1px solid
        rgba(
            51,
            225,
            210,
            .19
        );

    border-radius:
        2px;

    background:
        rgba(
            5,
            31,
            37,
            .46
        );

    text-decoration:
        none !important;

    transition:
        all .16s ease;

    text-align:
        left;
}


.bb-link:hover {

    border-color:
        rgba(
            51,
            225,
            210,
            .62
        );

    background:
        rgba(
            8,
            53,
            58,
            .80
        );

    transform:
        translateY(-2px);

    text-decoration:
        none !important;
}


.bb-link-num {

    display:
        block;

    margin-bottom:
        .30rem;

    color:
        #5D7F85;

    font-size:
        .64rem;

    font-weight:
        800;
}


.bb-link-title {

    display:
        block;

    color:
        #F1F7F7;

    font-size:
        .80rem;

    font-weight:
        790;
}


.bb-link-sub {

    display:
        block;

    margin-top:
        .40rem;

    color:
        #708F95;

    font-size:
        .66rem;

    font-weight:
        650;

    letter-spacing:
        .04em;
}


.bb-link-arrow {

    float:
        right;

    color:
        #3CE0D2;
}


/* ABOUT */

.bb-about {

    margin-top:
        .90rem;

    padding-top:
        .78rem;

    border-top:
        1px solid
        rgba(
            50,
            225,
            210,
            .09
        );

    color:
        #739197;

    font-size:
        .71rem;

    text-align:
        left;
}


.bb-about summary {

    cursor:
        pointer;

    color:
        #A1B8BC;

    font-weight:
        700;
}


.bb-about-copy {

    max-width:
        960px;

    margin-top:
        .72rem;

    color:
        #829EA3;

    font-size:
        .73rem;

    line-height:
        1.55;
}

</style>
"""


    st.markdown(
        _bb_css,
        unsafe_allow_html=True
    )


    # ========================================================
    # LOGO / CARITA + KICKER
    # ========================================================

    st.markdown(
        '<div class="bb-brandline">'

        '<div class="bb-home-face">'
        '🌬️'
        '</div>'

        '<div class="bb-kicker">'
        'URBAN AIR QUALITY · BARCELONA'
        '</div>'

        '</div>',
        unsafe_allow_html=True
    )


    # ========================================================
    # HERO
    # ========================================================

    st.markdown(
        '<div class="bb-breathe">'
        'BREATHE'
        '</div>'

        '<div class="bb-barcelona">'
        'BARCELONA'
        '</div>'

        '<div class="bb-subtitle">'
        'Urban Air Quality Intelligence'
        '</div>'

        '<div class="bb-meta">'

        '<span class="bb-period">'
        '2019—2024'
        '</span>'

        ' &nbsp;·&nbsp; OPEN DATA'

        ' &nbsp;·&nbsp; TIME SERIES'

        ' &nbsp;·&nbsp; SPATIAL ANALYTICS'

        ' &nbsp;·&nbsp; MACHINE LEARNING'

        ' &nbsp;·&nbsp; INTERPRETABILITY'

        '</div>',
        unsafe_allow_html=True
    )


    st.markdown(
        '<div class="bb-rule"></div>',
        unsafe_allow_html=True
    )


    # ========================================================
    # PIPELINE
    # ========================================================

    st.markdown(
        '<div class="bb-pipeline">'

        '<div class="bb-stage">'
        '<div class="bb-stage-num">'
        '01'
        '</div>'
        '<div class="bb-stage-title">'
        'OPEN DATA'
        '</div>'
        '<div class="bb-stage-copy">'
        'Air quality · Meteorology<br>'
        'Mobility · Urban context'
        '</div>'
        '</div>'

        '<div class="bb-arrow">'
        '→'
        '</div>'

        '<div class="bb-stage">'
        '<div class="bb-stage-num">'
        '02'
        '</div>'
        '<div class="bb-stage-title">'
        'SPACE + TIME'
        '</div>'
        '<div class="bb-stage-copy">'
        'Time series · Spatial analytics<br>'
        'LISA · Urban patterns'
        '</div>'
        '</div>'

        '<div class="bb-arrow">'
        '→'
        '</div>'

        '<div class="bb-stage">'
        '<div class="bb-stage-num">'
        '03'
        '</div>'
        '<div class="bb-stage-title">'
        'MODELS'
        '</div>'
        '<div class="bb-stage-copy">'
        'SARIMAX · XGBoost<br>'
        'Spatial regression · SHAP'
        '</div>'
        '</div>'

        '<div class="bb-arrow">'
        '→'
        '</div>'

        '<div class="bb-stage">'
        '<div class="bb-stage-num">'
        '04'
        '</div>'
        '<div class="bb-stage-title">'
        'PRODUCT'
        '</div>'
        '<div class="bb-stage-copy">'
        'Interactive dashboard<br>'
        'Dual predictor · Scenarios'
        '</div>'
        '</div>'

        '</div>',
        unsafe_allow_html=True
    )


    st.markdown(
        '<div class="bb-rule"></div>',
        unsafe_allow_html=True
    )


    # ========================================================
    # EXPLORE
    # ========================================================

    st.markdown(
        '<div class="bb-explore">'
        'EXPLORE BREATHE BARCELONA'
        '</div>',
        unsafe_allow_html=True
    )


    # ========================================================
    # ACCESOS
    # ========================================================

    st.markdown(
        '<div class="bb-links">'

        '<a class="bb-link" '
        'href="?bb_view=Overview" '
        'target="_self">'

        '<span class="bb-link-num">'
        '01'
        '</span>'

        '<span class="bb-link-title">'
        'OVERVIEW'
        '<span class="bb-link-arrow">'
        '→'
        '</span>'
        '</span>'

        '<span class="bb-link-sub">'
        'CITY AT A GLANCE'
        '</span>'

        '</a>'


        '<a class="bb-link" '
        'href="?bb_view=Temporal" '
        'target="_self">'

        '<span class="bb-link-num">'
        '02'
        '</span>'

        '<span class="bb-link-title">'
        'TEMPORAL'
        '<span class="bb-link-arrow">'
        '→'
        '</span>'
        '</span>'

        '<span class="bb-link-sub">'
        'TIME SERIES'
        '</span>'

        '</a>'


        '<a class="bb-link" '
        'href="?bb_view=Spatial" '
        'target="_self">'

        '<span class="bb-link-num">'
        '03'
        '</span>'

        '<span class="bb-link-title">'
        'SPATIAL'
        '<span class="bb-link-arrow">'
        '→'
        '</span>'
        '</span>'

        '<span class="bb-link-sub">'
        'SPATIAL ANALYTICS'
        '</span>'

        '</a>'


        '<a class="bb-link" '
        'href="?bb_view=Model%20Insights" '
        'target="_self">'

        '<span class="bb-link-num">'
        '04'
        '</span>'

        '<span class="bb-link-title">'
        'MODEL INSIGHTS'
        '<span class="bb-link-arrow">'
        '→'
        '</span>'
        '</span>'

        '<span class="bb-link-sub">'
        'INTERPRETABILITY'
        '</span>'

        '</a>'


        '<a class="bb-link" '
        'href="?bb_view=Predictor" '
        'target="_self">'

        '<span class="bb-link-num">'
        '05'
        '</span>'

        '<span class="bb-link-title">'
        'PREDICTOR'
        '<span class="bb-link-arrow">'
        '→'
        '</span>'
        '</span>'

        '<span class="bb-link-sub">'
        'MODEL → PRODUCT'
        '</span>'

        '</a>'

        '</div>',
        unsafe_allow_html=True
    )


    # ========================================================
    # ABOUT
    # ========================================================

    st.markdown(
        '<details class="bb-about">'

        '<summary>'
        'ⓘ &nbsp; About Breathe Barcelona'
        '</summary>'

        '<div class="bb-about-copy">'

        'Breathe Barcelona integrates open environmental, '

        'meteorological, mobility and urban-context data '

        'to investigate air-quality dynamics across '

        'Barcelona between 2019 and 2024.'

        '<br><br>'

        'The workflow connects time-series analysis, '

        'spatial statistics, predictive modelling and '

        'interpretability with an interactive analytical '

        'product.'

        '<br><br>'

        'Its final layer moves from analysis to '

        'productization through an interactive dashboard '

        'and a dual temporal / geospatial predictor.'

        '</div>'

        '</details>',
        unsafe_allow_html=True
    )


'''


    # ========================================================
    # 6.5 · INSERTAR
    # ========================================================

    INSERT_BLOCK = (
        ROUTER_BLOCK
        + "\n"
        + HOME_BLOCK
        + "\n"
    )


    code_new = (
        code[:overview_start]
        + INSERT_BLOCK
        + code[overview_start:]
    )


    # ========================================================
    # 6.6 · VALIDACIÓN COMPLETA ANTES DE ESCRIBIR
    # ========================================================

    compile(
        code_new,
        str(APP_PATH),
        "exec"
    )


    FINAL_VIEWS = {
        "PORTADA":
            'selected == "PORTADA"',
        **VIEWS,
    }


    for name, marker in FINAL_VIEWS.items():

        if marker not in code_new:

            raise RuntimeError(
                f"❌ Falta {name}."
            )


    REQUIRED_HOME = [
        "bb-home-marker",
        "bb-home-face",
        "🌬️",
        "URBAN AIR QUALITY · BARCELONA",
        "BREATHE",
        "BARCELONA",
        "Urban Air Quality Intelligence",
        "2019—2024",
        "OPEN DATA",
        "SPACE + TIME",
        "MODELS",
        "PRODUCT",
        "TIME SERIES",
        "SPATIAL ANALYTICS",
        "MACHINE LEARNING",
        "INTERPRETABILITY",
        "SARIMAX",
        "XGBoost",
        "SHAP",
        "?bb_view=Overview",
        "?bb_view=Temporal",
        "?bb_view=Spatial",
        "?bb_view=Model%20Insights",
        "?bb_view=Predictor",
    ]


    for marker in REQUIRED_HOME:

        if marker not in code_new:

            raise RuntimeError(
                f"❌ Falta contenido HOME: {marker}"
            )


    for marker in [
        "mi-kpi",
        "SHAP",
        "LOSO",
    ]:

        if marker not in code_new:

            raise RuntimeError(
                f"❌ Se perdió contenido crítico: {marker}"
            )


    if code_new.count(
        'selected == "PORTADA"'
    ) != 1:

        raise RuntimeError(
            "❌ PORTADA no quedó como bloque único."
        )


    if code_new.count(
        "BREATHE BARCELONA · HOME ROUTER"
    ) != 1:

        raise RuntimeError(
            "❌ Router HOME no quedó como bloque único."
        )


    print(
        "✅ Nueva composición sintácticamente correcta"
    )

    print(
        "✅ 🌬️ integrado como marca original"
    )

    print(
        "✅ HOME preparada"
    )

    print(
        "✅ Cinco accesos preparados"
    )

    print()


    # ========================================================
    # 6.7 · BACKUP
    # ========================================================

    stamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )


    BACKUP_PATH = (
        APP_PATH.parent
        /
        f"app_backup_antes_069_home_{stamp}.py"
    )


    BACKUP_PATH.write_text(
        pre_069_code,
        encoding="utf-8"
    )


    if (
        BACKUP_PATH.read_text(
            encoding="utf-8"
        )
        != pre_069_code
    ):

        raise RuntimeError(
            "❌ Falló el backup."
        )


    print(
        f"✅ Backup creado: "
        f"{BACKUP_PATH.name}"
    )


    # ========================================================
    # 6.8 · ESCRIBIR
    # ========================================================

    APP_PATH.write_text(
        code_new,
        encoding="utf-8"
    )


    final_code = APP_PATH.read_text(
        encoding="utf-8"
    )


    if final_code != code_new:

        raise RuntimeError(
            "❌ app.py no coincide "
            "con la versión validada."
        )


    compile(
        final_code,
        str(APP_PATH),
        "exec"
    )


    # ========================================================
    # 6.9 · VERIFICACIÓN POST-ESCRITURA
    # ========================================================

    for name, marker in FINAL_VIEWS.items():

        if marker not in final_code:

            raise RuntimeError(
                f"❌ Verificación final fallida: {name}"
            )


    for marker in [
        "mi-kpi",
        "SHAP",
        "LOSO",
    ]:

        if marker not in final_code:

            raise RuntimeError(
                f"❌ Se perdió contenido crítico: {marker}"
            )


    if final_code.count(
        'selected == "PORTADA"'
    ) != 1:

        raise RuntimeError(
            "❌ Hay más de una PORTADA."
        )


    if final_code.count(
        "BREATHE BARCELONA · HOME ROUTER"
    ) != 1:

        raise RuntimeError(
            "❌ Hay más de un router HOME."
        )


    # ========================================================
    # 6.10 · RESULTADO
    # ========================================================

    print()
    print("=" * 96)
    print("🌬️ BREATHE BARCELONA · 06.9 HOME / LANDING — FINAL")
    print("=" * 96)
    print()

    print("✅ HOME instalada")
    print("✅ 🌬️ recuperado como marca original")
    print("✅ Logo + kicker alineados")
    print("✅ Sin cabecera analítica en HOME")
    print("✅ Sin navegación duplicada")
    print("✅ ⊙ = HOME en las sheets")
    print("✅ BREATHE / BARCELONA conservados")
    print("✅ 2019—2024 integrado")
    print(
        "✅ OPEN DATA → SPACE + TIME → MODELS → PRODUCT"
    )
    print("✅ Cinco accesos inferiores")
    print("✅ Overview conservado")
    print("✅ Temporal conservado")
    print("✅ Spatial conservado")
    print("✅ Model Insights conservado")
    print("✅ Predictor conservado")
    print("✅ SHAP conservado")
    print("✅ LOSO conservado")
    print("✅ Sintaxis Python correcta")
    print()

    print(
        f"📦 Backup: {BACKUP_PATH.name}"
    )

    print()

    print(
        "🎯 06.9 COMPLETADA"
    )

    print("=" * 96)

🌬️ BREATHE BARCELONA · 06.9 HOME / LANDING — FINAL
✅ app.py localizado
✅ Sintaxis Python correcta
✅ Tamaño actual: 293,880 caracteres

✅ Overview localizada
✅ Temporal localizada
✅ Spatial localizada
✅ Model Insights localizada
✅ Predictor localizada
✅ Estado POST-06.8 confirmado

🏠 ESTADO HOME
------------------------------------------------------------------------------------------------
Marcadores encontrados: 0/11
✅ HOME todavía no está instalada

✅ ⊙ incorporado como HOME
✅ Punto de inserción localizado
✅ Nueva composición sintácticamente correcta
✅ 🌬️ integrado como marca original
✅ HOME preparada
✅ Cinco accesos preparados

✅ Backup creado: app_backup_antes_069_home_20260911_115037.py

🌬️ BREATHE BARCELONA · 06.9 HOME / LANDING — FINAL

✅ HOME instalada
✅ 🌬️ recuperado como marca original
✅ Logo + kicker alineados
✅ Sin cabecera analítica en HOME
✅ Sin navegación duplicada
✅ ⊙ = HOME en las sheets
✅ BREATHE / BARCELONA conservados
✅ 2019—2024 integrado
✅ OPEN DATA → SPACE + TIME

### Resultados

La **Home** queda integrada como página inicial de *Breathe Barcelona*, reforzando su carácter de producto analítico y proporcionando una lectura sintética de la arquitectura completa del proyecto.

La portada incorpora la identidad **Breathe Barcelona · Urban Air Quality Intelligence**, el periodo **2019–2024** y una representación secuencial del flujo **Open Data → Space + Time → Models → Product**, conectando las fuentes ambientales y urbanas con el análisis temporal y espacial, la modelización y la capa interactiva final. Asimismo, se habilitan cinco accesos directos a **Overview, Temporal, Spatial, Model Insights y Predictor**.
La navegación incorpora además un acceso específico a **Home**, sin duplicar la cabecera analítica. La validación final confirma que las cinco vistas permanecen intactas, incluyendo los componentes **SHAP y LOSO**, y que la aplicación conserva una sintaxis válida.

> **Resultado:** *Breathe Barcelona* dispone de una portada funcional que estructura el dashboard como un producto analítico completo, conectando de forma explícita datos, análisis, modelos e interacción.


### 6.10 · Configuración global del tema Streamlit

Como ajuste final de la identidad visual de *Breathe Barcelona*, se define una configuración global de **Streamlit en modo oscuro**, alineada con la estética desarrollada en las distintas vistas del dashboard.

La configuración centraliza los colores de fondo, texto, bordes y elementos interactivos mediante una paleta basada en **azul petróleo y turquesa**. El tema se almacena en `.streamlit/config.toml` dentro del directorio del proyecto, permitiendo aplicar una apariencia homogénea también a los componentes nativos de Streamlit.

> **Objetivo:** unificar la identidad visual de la aplicación mediante un tema global coherente con la estética oscura y la paleta cromática de *Breathe Barcelona*.


In [39]:
# ============================================================
# 6.10 CORRECCIÓN GLOBAL DEL TEMA STREAMLIT
# BREATHE BARCELONA
# ============================================================

from pathlib import Path

config_dir = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard/.streamlit"
)

config_dir.mkdir(
    parents=True,
    exist_ok=True
)

config_path = (
    config_dir / "config.toml"
)

config_text = """
[theme]
base = "dark"

primaryColor = "#1FE8D9"

backgroundColor = "#071820"

secondaryBackgroundColor = "#0B202A"

textColor = "#F3F8FA"

borderColor = "#24505F"

showWidgetBorder = true
"""

config_path.write_text(
    config_text.strip() + "\n",
    encoding="utf-8"
)

print("=" * 72)
print("✅ TEMA STREAMLIT DARK CREADO")
print("=" * 72)
print()
print("Ruta:")
print(config_path)
print()
print("base                      = dark")
print("backgroundColor           = #071820")
print("secondaryBackgroundColor  = #0B202A")
print("textColor                 = #F3F8FA")
print("primaryColor              = #1FE8D9")
print()
print("👉 AHORA reinicia Streamlit con 6.10 y 6.11")
print("=" * 72)

✅ TEMA STREAMLIT DARK CREADO

Ruta:
/content/drive/MyDrive/TFM/12_Dashboard/.streamlit/config.toml

base                      = dark
backgroundColor           = #071820
secondaryBackgroundColor  = #0B202A
textColor                 = #F3F8FA
primaryColor              = #1FE8D9

👉 AHORA reinicia Streamlit con 6.10 y 6.11


### Resultados

Se genera correctamente la configuración global de **Streamlit en modo oscuro**, estableciendo como fondo principal `#071820`, fondo secundario `#0B202A`, texto `#F3F8FA`, bordes `#24505F` y turquesa `#1FE8D9` como color principal de interacción.

El archivo `.streamlit/config.toml` queda integrado en el directorio del proyecto, proporcionando una configuración visual común para los componentes nativos de la interfaz.

La aplicación queda preparada para reiniciar el servidor Streamlit y cargar el tema definitivo antes de habilitar nuevamente el acceso público.

> **Resultado:** *Breathe Barcelona* dispone de una configuración visual global consistente con su identidad oscura, azul petróleo y turquesa.


### 6.11 · Puesta en marcha del servidor Streamlit

Una vez completadas las vistas y la navegación de *Breathe Barcelona*, se inicia la aplicación mediante un **servidor Streamlit local**.

Antes del arranque se comprueba la existencia y sintaxis de `app.py`, se cierra cualquier instancia previa de Streamlit y se inicia un nuevo servidor en el **puerto 8501**. Finalmente, se realiza una petición HTTP local para verificar que la aplicación está operativa antes de habilitar su acceso externo.

> **Objetivo:** comprobar que la versión final de *Breathe Barcelona* puede ejecutarse correctamente como aplicación Streamlit antes de su publicación mediante un túnel externo.


In [40]:
# ============================================================
# 🌬️ BREATHE BARCELONA
# 06.11 · SERVIDOR STREAMLIT
# ============================================================
#
# REQUIERE:
#   ✓ app.py construido
#
# HACE:
#   ✓ Comprueba sintaxis de app.py
#   ✓ Cierra un Streamlit anterior si existe
#   ✓ Arranca Streamlit en puerto 8501
#   ✓ Comprueba que responde por HTTP
#
# NO TOCA:
#   ✓ app.py
#
# IMPORTANTE:
#   Ejecutar una sola vez antes del túnel Cloudflare.
# ============================================================

import subprocess
import time
import urllib.request
from pathlib import Path


# ============================================================
# 1 · RUTAS
# ============================================================

APP_PATH = Path(
    "/content/drive/MyDrive/TFM/12_Dashboard/app.py"
)

STREAMLIT_LOG = Path(
    "/content/breathe_streamlit.log"
)


# ============================================================
# 2 · COMPROBAR APP.PY
# ============================================================

if not APP_PATH.exists():

    raise FileNotFoundError(
        f"❌ No existe app.py:\n{APP_PATH}"
    )


code = APP_PATH.read_text(
    encoding="utf-8"
)

compile(
    code,
    str(APP_PATH),
    "exec"
)

print("✅ app.py encontrado")
print("✅ Sintaxis Python correcta")


# ============================================================
# 3 · CERRAR STREAMLIT ANTERIOR
# ============================================================

subprocess.run(
    [
        "pkill",
        "-f",
        "streamlit run"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)


# ============================================================
# 4 · LIMPIAR LOG
# ============================================================

STREAMLIT_LOG.write_text(
    "",
    encoding="utf-8"
)


# ============================================================
# 5 · ARRANCAR STREAMLIT
# ============================================================

log_file = open(
    STREAMLIT_LOG,
    "w"
)

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        str(APP_PATH),

        "--server.port",
        "8501",

        "--server.address",
        "0.0.0.0",

        "--server.headless",
        "true",

        "--browser.gatherUsageStats",
        "false"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)


# ============================================================
# 6 · ESPERAR ARRANQUE
# ============================================================

streamlit_ok = False
http_status = None

for _ in range(25):

    time.sleep(1)

    try:

        response = urllib.request.urlopen(
            "http://127.0.0.1:8501",
            timeout=2
        )

        http_status = response.status

        if http_status == 200:
            streamlit_ok = True
            break

    except Exception:
        pass


# ============================================================
# 7 · RESULTADO
# ============================================================

print()
print("=" * 80)
print("🌬️ BREATHE BARCELONA · SERVIDOR STREAMLIT")
print("=" * 80)
print()

if streamlit_ok:

    print("✅ Streamlit activo")
    print(f"✅ HTTP {http_status}")
    print("✅ Puerto 8501")
    print("✅ app.py cargado correctamente")
    print()
    print("➡️ Ahora puedes ejecutar la celda")
    print("   06.6 · TÚNEL PÚBLICO CLOUDFLARE")

else:

    print("❌ Streamlit no ha arrancado correctamente.")
    print()
    print("Últimas líneas del log:")
    print("-" * 80)

    log_text = STREAMLIT_LOG.read_text(
        encoding="utf-8",
        errors="replace"
    )

    print(
        "\n".join(
            log_text.splitlines()[-50:]
        )
    )

print()
print("=" * 80)

✅ app.py encontrado
✅ Sintaxis Python correcta

🌬️ BREATHE BARCELONA · SERVIDOR STREAMLIT

✅ Streamlit activo
✅ HTTP 200
✅ Puerto 8501
✅ app.py cargado correctamente

➡️ Ahora puedes ejecutar la celda
   06.6 · TÚNEL PÚBLICO CLOUDFLARE



### Resultados

La aplicación se inicia correctamente mediante **Streamlit**, tras validar previamente la sintaxis de `app.py`.

El servidor queda activo en el **puerto 8501** y la comprobación mediante petición local devuelve un estado **HTTP 200**, confirmando que *Breathe Barcelona* se encuentra cargado y respondiendo correctamente.

Esta comprobación establece el punto de partida para la siguiente fase de despliegue, en la que el servidor local se expone temporalmente mediante un túnel público.

> **Resultado:** *Breathe Barcelona* queda ejecutándose correctamente como aplicación Streamlit y preparada para habilitar su acceso externo.


### 6.12 · Publicación temporal mediante Cloudflare Tunnel

Una vez verificado el funcionamiento local de *Breathe Barcelona*, se habilita un acceso externo mediante **Cloudflare Tunnel**, conectando el servidor Streamlit activo en el puerto 8501 con una URL pública temporal.

Antes de abrir el túnel se comprueba que Streamlit responde correctamente. Si es necesario, se descarga el ejecutable `cloudflared`, se cierra cualquier túnel previo y se inicia una nueva conexión hacia el servidor local, extrayendo automáticamente la URL pública generada.

El procedimiento no modifica `app.py` y actúa únicamente como capa de exposición externa de la aplicación.

> **Objetivo:** permitir el acceso remoto a *Breathe Barcelona* mediante una URL pública temporal, manteniendo intacta la aplicación y su lógica interna.


In [41]:
# ============================================================
# 🌬️ BREATHE BARCELONA
# 06.12 · TÚNEL PÚBLICO CLOUDFLARE
# ============================================================
#
# REQUIERE:
#   ✓ 06.10 ejecutada
#   ✓ Streamlit activo en puerto 8501
#
# HACE:
#   ✓ Comprueba Streamlit
#   ✓ Descarga cloudflared si no existe
#   ✓ Abre túnel público
#   ✓ Extrae URL https://....trycloudflare.com
#
# NO TOCA:
#   ✓ app.py
#
# IMPORTANTE:
#   Ejecutar una sola vez.
# ============================================================

import subprocess
import time
import re
import urllib.request
from pathlib import Path


# ============================================================
# 1 · RUTAS
# ============================================================

CLOUDFLARED_PATH = Path(
    "/content/cloudflared"
)

CLOUDFLARE_LOG = Path(
    "/content/breathe_cloudflare.log"
)


# ============================================================
# 2 · COMPROBAR STREAMLIT
# ============================================================

try:

    response = urllib.request.urlopen(
        "http://127.0.0.1:8501",
        timeout=5
    )

    print(
        f"✅ Streamlit responde · HTTP {response.status}"
    )

except Exception as e:

    raise RuntimeError(
        """
❌ Streamlit no responde en el puerto 8501.

Ejecuta primero 06.10 · SERVIDOR STREAMLIT
y después vuelve a ejecutar 06.11.
"""
    ) from e


# ============================================================
# 3 · DESCARGAR CLOUDFLARED SI NO EXISTE
# ============================================================

if not CLOUDFLARED_PATH.exists():

    print("⬇️ Descargando cloudflared...")

    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            str(CLOUDFLARED_PATH),
            (
                "https://github.com/cloudflare/cloudflared/"
                "releases/latest/download/"
                "cloudflared-linux-amd64"
            )
        ],
        check=True
    )

    subprocess.run(
        [
            "chmod",
            "+x",
            str(CLOUDFLARED_PATH)
        ],
        check=True
    )

    print("✅ cloudflared descargado")

else:

    print("✅ cloudflared disponible")


# ============================================================
# 4 · CERRAR TÚNELES ANTERIORES
# ============================================================

subprocess.run(
    [
        "pkill",
        "-f",
        "cloudflared"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)


# ============================================================
# 5 · LIMPIAR LOG
# ============================================================

CLOUDFLARE_LOG.write_text(
    "",
    encoding="utf-8"
)


# ============================================================
# 6 · ABRIR TÚNEL
# ============================================================

log_file = open(
    CLOUDFLARE_LOG,
    "w"
)

tunnel_process = subprocess.Popen(
    [
        str(CLOUDFLARED_PATH),
        "tunnel",
        "--url",
        "http://127.0.0.1:8501",
        "--no-autoupdate"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)


# ============================================================
# 7 · BUSCAR URL PÚBLICA
# ============================================================

public_url = None

for _ in range(30):

    time.sleep(1)

    if CLOUDFLARE_LOG.exists():

        log_text = CLOUDFLARE_LOG.read_text(
            encoding="utf-8",
            errors="replace"
        )

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            log_text
        )

        if match:

            public_url = match.group(0)
            break


# ============================================================
# 8 · RESULTADO
# ============================================================

print()
print("=" * 80)
print("🌬️ BREATHE BARCELONA · 06.11")
print("=" * 80)
print()

if public_url:

    print("✅ Túnel Cloudflare activo")
    print("✅ Streamlit conectado")
    print("✅ app.py NO ha sido modificado")

    print()
    print("🌐 ABRE EL DASHBOARD AQUÍ:")
    print()
    print(public_url)
    print()

    print("⚠️ No vuelvas a ejecutar 06.11 mientras")
    print("   este enlace siga funcionando.")

else:

    print("❌ No se pudo obtener la URL pública.")

    print()
    print("Últimas líneas del log:")
    print("-" * 80)

    log_text = CLOUDFLARE_LOG.read_text(
        encoding="utf-8",
        errors="replace"
    )

    print(
        "\n".join(
            log_text.splitlines()[-40:]
        )
    )


print()
print("=" * 80)

✅ Streamlit responde · HTTP 200
✅ cloudflared disponible

🌬️ BREATHE BARCELONA · 06.11

✅ Túnel Cloudflare activo
✅ Streamlit conectado
✅ app.py NO ha sido modificado

🌐 ABRE EL DASHBOARD AQUÍ:

https://cooked-naturals-emma-mediterranean.trycloudflare.com

⚠️ No vuelvas a ejecutar 06.11 mientras
   este enlace siga funcionando.



### Resultados

La validación previa confirma que **Streamlit responde correctamente mediante HTTP 200**, tras lo cual se establece con éxito un **túnel público de Cloudflare** conectado al servidor local.

Se obtiene una URL pública temporal bajo el dominio `trycloudflare.com`, permitiendo acceder externamente a *Breathe Barcelona* mientras permanecen activos tanto el servidor Streamlit como el túnel. Durante este proceso, `app.py` permanece sin modificaciones.

> **Resultado:** *Breathe Barcelona* queda accesible externamente mediante una URL pública temporal, completando el flujo desde el análisis y la modelización hasta su explotación como aplicación web interactiva.


---

# LF_01 · Resultado final · Breathe Barcelona

El desarrollo culmina con la construcción de **Breathe Barcelona**, una aplicación interactiva que integra en una única interfaz los principales resultados obtenidos a lo largo del TFM.

La arquitectura mantiene separadas la **fase analítica** y la **fase de explotación**, de forma que el dashboard consume datasets optimizados, resultados previamente validados y artefactos de modelización ya entrenados. Esta separación conserva la trazabilidad entre los análisis realizados y su representación interactiva, evitando reproducir procesos de tratamiento o entrenamiento durante la ejecución de la aplicación.

La versión final se estructura mediante cinco vistas —**Overview, Temporal, Spatial, Model Insights y Predictor**— que permiten recorrer el problema desde la caracterización general de la calidad del aire hasta el análisis temporal y espacial, la evaluación e interpretación de los modelos y la exploración interactiva de escenarios.

La componente espacial integra **LISA**, asociaciones espaciales bivariadas y los resultados de los modelos **SAR y XGBoost**. La componente temporal incorpora la evolución de los contaminantes, sus factores asociados y la modelización de series temporales. **Model Insights** reúne la evaluación del modelo de clasificación, la validación **LOSO** y la interpretación mediante **SHAP**, mientras que el **Dual Predictor** traslada los modelos espacial y temporal a una capa interactiva de simulación.

Finalmente, la incorporación de una **Home / Landing**, una identidad visual homogénea y un sistema de navegación común consolida el conjunto como un producto analítico integrado. Su ejecución mediante **Streamlit** y la habilitación de acceso externo permiten comprobar el funcionamiento de la aplicación como herramienta web interactiva.

> **Resultado final:** *Breathe Barcelona* materializa el recorrido desde los datos abiertos y el análisis espacio-temporal hasta la modelización, la interpretabilidad y la explotación interactiva de los resultados, constituyendo la capa final de comunicación y exploración del TFM.